# Code 2. In-house FT-IR `spec_class` library for field particles, compared with the Raman Code1 Reference (v1.0)

A catalogue of naming disagreement, not a pollution survey: how differently three naming schemes describe the same field particles.

| Part | Data | Content |
|---|---|---|
| 0 | all | preprocessing, literature band ledger, decision engine, feature cache, OMNIC record, threshold checks (0.5-0.10) |
| 1 | <= 06.01, S1+S2+S3 | S1 spans the functional-group diversity of the site and period; basis for continuing with S1 alone |
| 2 | <= 06.28, S1 | builds the in-house library; vocabulary and parameter snapshot |
| 3 | 07.11-07.25 | applies the frozen rules once; stability checks |
| 4 | all | reads the Code 1 reference (4.1); spec_class vs reference (4.2); OMNIC names vs reference (4.3); weathering test (4.4) |
| 5 | | particle x stream catalogue for Code 3 |

**Streams never correct one another.** `spec_class` is the raw output of the rule engine `classify()`; OMNIC names and HQI are recorded verbatim; the Raman Code1 Reference is taken from Code 1 as given, a rule output filtered by an evidence floor and not a ground truth. Part 4 measures agreement between two automated frameworks, not accuracy. Reconciling names is Code 3's job.

**Parameters.** Band windows are verified against pure standards (0.5); thresholds are declared in advance and their influence is measured by sensitivity sweeps (0.6-0.10), never adjusted toward better agreement. A change to any band window, threshold or rule is a new `PARAM_VERSION` and requires Parts 2-5 to be re-run end to end.

In [1]:
from google.colab import drive; drive.mount('/content/drive')
import os, shutil, glob

PROJECT = '/content/drive/MyDrive/mp_project'      # the only path to adapt
FOLDERS = ['spectra', 'standards_ftir', 'raman_manual_out']

for name in FOLDERS:
    dst = f'/content/{name}'
    os.makedirs(dst, exist_ok=True)
    n = sum(bool(shutil.copy(p, dst))
            for p in glob.glob(os.path.join(PROJECT, name, '*')) if os.path.isfile(p))
    print(f'{dst:26s} {n:>4} files')

print('-' * 46)
_must = '/content/raman_manual_out/ground_truth_for_code2.csv'
print('handoff CSV      :', 'OK' if os.path.exists(_must) else '*** MISSING ***')
for f in ('PE_428043.CSV', 'PP_427888.CSV'):
    print(f'{f:17s}:', 'OK' if os.path.exists(f'/content/standards_ftir/{f}')
          else '*** MISSING - check the file-name case ***')

Mounted at /content/drive
/content/spectra            230 files
/content/standards_ftir       2 files
/content/raman_manual_out    71 files
----------------------------------------------
handoff CSV      : OK
PE_428043.CSV    : OK
PP_427888.CSV    : OK


## Part 0. Configuration, literature ledger, decision engine

In [2]:
# --- setup ---
import os, re, glob, io
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.sparse.linalg import spsolve
from scipy.signal import savgol_filter
_TRAPZ = np.trapezoid if hasattr(np,'trapezoid') else np.trapz   # numpy 1.x/2.x shim
pd.set_option('display.width',160); pd.set_option('display.max_rows',200)
print('ready')

ready


In [3]:
# === CONFIG =====================================================
import os
import re
import numpy as np

SPECTRA_DIR = "/content/spectra"           # folder of *.CSV spectra
OUT_DIR     = "/content/out"
os.makedirs(OUT_DIR, exist_ok=True)

GRID      = np.arange(400.0, 4000.0+1e-6, 1.0)   # common 1 cm-1 axis
ALS_LAM   = 1e5      # ALS baseline smoothness (within the range of Eilers & Boelens 2005)
ALS_P     = 0.01     # ALS asymmetry
SG_WIN, SG_ORD = 15, 3    # Savitzky & Golay 1964. Window and order are a design choice,
                          # not a literature value.

# --- band PRESENCE --------------------------------------------------------
REL_PRESENT      = 0.30   # relative height, against the strongest band of that spectrum
REL_PRESENT_WEAK = 0.20   # floor: never accepted below this, even by the S/N route
SNR_PRESENT      = 30.0   # signal-to-noise

# A band counts as present when
#   rel >= REL_PRESENT
#   or (band in SNR_RELAXABLE  and  rel >= REL_PRESENT_WEAK  and  snr >= SNR_PRESENT)
#
# The S/N route is restricted to a whitelist. rel is a ratio against the strongest band of
# the spectrum, so a diagnostic band from a different vibrational family is penalised for
# reasons of physics rather than absence: the PP helix bands (998/973/840) belong to a
# different family from the CH stretch and sit at 20-30% even in pure iPP, which measures
# 998 = 0.278 (below rel) at S/N 74. The SAN nitrile and the PVDF C-F band, by contrast,
# ARE the entire evidence for their class, so the same relaxation would let weak noise
# satisfy a multi-band AND. Measured in section 0.6 (n=230): relaxing every band raises PP
# from 23 to 37 but raises SAN/PVDF/nylon from 13 to 21 alongside it, close to one for one
# as the relaxation widens. Only bands shown on a standard to be "below rel yet unambiguous
# by S/N" are whitelisted.
SNR_RELAXABLE = {'b998', 'b973', 'b840'}   # PP helix; section 0.5 re-checks the candidates
USE_SNR_PRESENCE = True                    # False falls back to rel alone

# --- band STRENGTH: one value, two roles, two names ------------------------
# Both are 0.60 today but they act in different places. Keeping them separate lets the
# sensitivity sweep distinguish "moves the confidence tier" from "moves the class".
REL_STRONG   = 0.60   # CONFIDENCE tier - confident vs tentative
REL_DOMINANT = 0.60   # CLASS decision - mineral dominance, silicate call, Si-O veto

QC_MIN_TOTAL_A = 0.5

CI_POLYOLEFIN_ONLY = True
MASK_REGIONS  = [(2280, 2400)]        # atmospheric CO2
SIO_DOMINATES = 1.15                  # ratio at which a mineral band overwhelms the CH skeleton
CI_WEATHERED  = 0.30                  # weathering cut (modified CI, narrower than Almond 2020)

# --- how far the carbonyl index may decide a class ------------------------
# 'always'          : a high CI withholds the PE call even when pe_doublet is complete
# 'incomplete_only' : in force. A complete diagnostic fingerprint is not overturned by CI.
#
# pe_doublet is True only when 1472 + 730 + 717 are ALL present, and the 730/717 rocking
# doublet is specific to PE and absent in PP (Jung2018). pe_doublet = True therefore already
# means the PE fingerprint is complete - which was precisely the case in which 'always'
# withdrew the answer on the strength of an oxidation index. Anshari2025 shows that
# weathering makes PE and PP harder to separate; a surviving doublet is itself evidence that
# this particle does not have that problem. Narrowing CI to "withhold only when the evidence
# is incomplete" keeps the ungrounded CI_WEATHERED cut out of class decisions (section 0.9).
CI_GATE = 'incomplete_only'

# --- class accumulation curve (design rationale, section 1.2) -------------
# Not a richness estimate: one figure showing that S1 already spans the functional groups
# seen in this period.
RAREF_REPS = 200    # permutations, for a smooth curve
RAREF_SEED = 0
TAIL_FRAC  = 0.25   # classes first appearing in the last 25%, reported descriptively only

# --- time gating ----------------------------------------------------------
PHASE1_END      = "06.01"   # end of Section 1, inclusive. S1+S2+S3 mixed
PHASE2_END      = "06.28"   # end of Section 2, inclusive. End of the training window
HELDOUT_FROM    = "07.01"   # Section 3 starts here
PHASE2_S1_ONLY  = True      # drop non-S1 stations from phase 2 (exclusions are printed)
EXCLUDE_HISTORICAL = True   # 2025 sessions are historical, not used for training

# --- vocabulary: chemical class vs measurement state ----------------------
# QC states and 'unresolved' are abstentions, not classes. They are excluded from the
# vocabulary snapshot, the accumulation curve and class shares, and reported separately.
# polyolefin_unresolved IS a chemical class: the polyolefin family is settled and only
# PE vs PP is left open, which is the honest output of the weathering rule (Anshari2025).
NON_CHEMICAL_STATES = {'unresolved', 'mineral_or_unresolved',
                       'QC_low_contrast', 'QC_low_absorbance', 'QC_water_dominated',
                       'ERROR'}

FRAMEWORK_VERSION   = "v1.0"
# PARAM_VERSION names the parameter set written to the snapshot (section 2.2) and read back
# by section 3. Any change to a band window, threshold or rule is a new parameter set: raise
# it and re-run sections 2 to 5 end to end.
PARAM_VERSION       = "p1"
FROZEN_PARAMS_PATH  = os.path.join(OUT_DIR, f"params_snapshot_{PARAM_VERSION}.json")
SCORING_LOG_PATH    = os.path.join(OUT_DIR, f"july_scoring_log_{PARAM_VERSION}.json")
print("config loaded |", FRAMEWORK_VERSION,
      f"| P={REL_PRESENT} weak={REL_PRESENT_WEAK} snr={SNR_PRESENT:.0f} S={REL_STRONG} D={REL_DOMINANT}")

# === Threshold provenance ledger ==========================================
# Two axes are recorded for every threshold:
#   grade  - where the value CAME FROM.      A literature or physics
#                                            B derived from a standard
#                                            C design choice
#   impact - whether it MOVES THE CONCLUSION. Sections 0.6-0.10 overwrite this at run time.
#
# Combining them gives what can be defended in the manuscript:
#   A or B                        grounded
#   C + demonstrated to be inert  defensible: ungrounded, but shown not to propagate
#   C + not yet measured          open
#   C + shown to matter           needs grounding
#
# The vocabulary below is load-bearing: threshold_report() classifies by matching these
# markers against the impact text, and sections 0.6-0.10 must write impact strings in the
# same vocabulary. Change them here, not in the sections.
IMPACT_UNMEASURED = 'not measured'                      # substring test
IMPACT_INERT_RE   = r'changed 0\b|\b0/\d+|no change|structurally 0'

THRESHOLD_PROVENANCE = {
 'MASK_REGIONS':    dict(v=str(MASK_REGIONS), grade='A', ref='-',
        src='Atmospheric CO2 absorption (2280-2400). Independent of the sample: physics.',
        impact='not applicable - not a sample signal'),
 'ALS_LAM/ALS_P':   dict(v=f'{ALS_LAM:.0e}/{ALS_P}', grade='A', ref='-',
        src='Within the range recommended by Eilers & Boelens 2005',
        impact=f'{IMPACT_UNMEASURED} - preprocessing stage'),
 'SNR_PRESENT':     dict(v=SNR_PRESENT, grade='B', ref='0.5, 0.6',
        src='Lower bound from the IUPAC LOQ (S/N >= 10), upper bound from a standard: the '
            'weakest diagnostic band of pure iPP, 998, measures S/N 74. 30 chosen in '
            '14 < x <= 74.',
        impact='0.6 sweep: across 20-74 the whitelist route adds 0 implausible classes'),
 'REL_PRESENT_WEAK':dict(v=REL_PRESENT_WEAK, grade='B', ref='0.5, 0.6',
        src='Bracketed by a standard: a band that should be absent (pure iPP 1715, rel 0.053) '
            '< x <= the weakest band that should be present (998, rel 0.278). '
            'False-positive margin x3.8.',
        impact='0.6 sweep: 2-4 assignments change across 0.15-0.25'),
 'SNR_RELAXABLE':   dict(v=str(sorted(SNR_RELAXABLE)), grade='B', ref='0.5, 0.6',
        src='Only bands confirmed on a standard to fall below rel yet be unambiguous by S/N '
            '(pure iPP 998 and 840). Section 0.6 shows that applying the relaxation to every '
            'band raises SAN and PVDF almost one for one with PP.',
        impact='0.6: restricted to the whitelist, 0 implausible classes added '
               '(applied to every band: +8 to +28)'),
 'REL_DOMINANT':    dict(v=REL_DOMINANT, grade='B', ref='0.5, 0.7',
        src='Bracketed by standards: the largest SiO in a polymer standard (pure iPP 0.225) '
            '< x <= SiO of a glass-fibre filter (0.877). Margin x2.7 below, x1.46 above.',
        impact=f'{IMPACT_UNMEASURED} (updated when 0.7 runs)'),
 'CI_WEATHERED':    dict(v=CI_WEATHERED, grade='C', ref='0.7, 0.8, 0.9',
        src='A modified form of Almond2020 (windows narrowed to 1690-1760 / 1420-1490), so the '
            'published values do not carry over. Only the ZERO POINT is measured on standards '
            '(virgin PE 0.043, PP 0.076). Section 0.8: the distribution is unimodal with a long '
            'tail and has no valley, so no cut can be derived from it. The cut of 0.30 is '
            'therefore ungrounded, which is why CI_GATE narrows the range over which it can '
            'decide a class.',
        impact=f'{IMPACT_UNMEASURED} (updated when 0.7 runs)'),
 'CI_GATE':         dict(v=CI_GATE, grade='A', ref='0.9',
        src='The 730/717 rocking doublet is specific to PE and absent in PP (Jung2018), so a '
            'complete doublet means a complete PE fingerprint and is not withdrawn by an '
            'oxidation index. Grounded in band assignment.',
        impact='0.9 compares the two modes'),
 'REL_PRESENT':     dict(v=REL_PRESENT, grade='C', ref='0.6',
        src='Design choice. Data-derived cuts (P75/P90) saturate on every subset and cannot be '
            'used; the S/N route compensates, but the value itself is ungrounded.',
        impact=f'{IMPACT_UNMEASURED} - the S/N route compensates at the lower bound'),
 'REL_STRONG':      dict(v=REL_STRONG, grade='C', ref='0.7',
        src='Design choice (twice REL_PRESENT). Confidence tier only; takes no part in class '
            'decisions.',
        impact=f'{IMPACT_UNMEASURED} (updated when 0.7 runs)'),
 'QC_MIN_TOTAL_A':  dict(v=QC_MIN_TOTAL_A, grade='C', ref='0.7',
        src='Design choice (threshold at which assignment begins)',
        impact=f'{IMPACT_UNMEASURED} (updated when 0.7 runs)'),
 'SIO_DOMINATES':   dict(v=SIO_DOMINATES, grade='C', ref='0.7',
        src='Design choice (mineral dominance ratio)',
        impact=f'{IMPACT_UNMEASURED} (updated when 0.7 runs)'),
 'SG_WIN/SG_ORD':   dict(v=f'{SG_WIN}/{SG_ORD}', grade='A', ref='0.10',
        src='Savitzky & Golay 1964. Window and order are a design choice but are never consumed '
            'by the assignment: preprocess() produces pp[d2], while features() reads only '
            'pp[corr] and pp[ref].',
        impact='structurally 0 (proved in code in 0.10)'),
}

VERDICTS = [('grounded',                       'ok'),
            ('defensible (shown to be inert)', 'ok'),
            ('open (impact not measured)',     '!'),
            ('needs grounding (impact shown)', 'X')]


def threshold_report():
    """Classify each threshold by source grade x effect on the conclusion."""
    def _verdict(d):
        if d['grade'] in ('A', 'B'):
            return VERDICTS[0][0]
        im = str(d.get('impact', ''))
        if IMPACT_UNMEASURED in im:
            return VERDICTS[2][0]
        if re.search(IMPACT_INERT_RE, im):
            return VERDICTS[1][0]
        return VERDICTS[3][0]

    groups = {}
    for k, d in THRESHOLD_PROVENANCE.items():
        groups.setdefault(_verdict(d), []).append(k)
    print("\n=== threshold provenance ===")
    for lab, mark in VERDICTS:
        ks = groups.get(lab, [])
        print(f"  {mark} {lab}: {len(ks)}" + (f" - {', '.join(ks)}" if ks else ""))
    print("  detail:  pd.DataFrame(THRESHOLD_PROVENANCE).T[['v','grade','impact','ref']]")


threshold_report()

config loaded | v1.0 | P=0.3 weak=0.2 snr=30 S=0.6 D=0.6

=== threshold provenance ===
  ok grounded: 8 - MASK_REGIONS, ALS_LAM/ALS_P, SNR_PRESENT, REL_PRESENT_WEAK, SNR_RELAXABLE, REL_DOMINANT, CI_GATE, SG_WIN/SG_ORD
  ok defensible (shown to be inert): 0
  ! open (impact not measured): 5 - CI_WEATHERED, REL_PRESENT, REL_STRONG, QC_MIN_TOTAL_A, SIO_DOMINATES
  X needs grounding (impact shown): 0
  detail:  pd.DataFrame(THRESHOLD_PROVENANCE).T[['v','grade','impact','ref']]


In [4]:
# === Reporting labels: three tiers ========================================
# A reporting view over classify(); the engine is untouched and detailed classes still reach
# the catalogue. Non-polymers are grouped because the question is misidentification in the
# NAMING of microplastics, and the Si-O window (1000-1090) contains the cellulose C-O band,
# so silicate cannot be separated from organic matter with this band set. Pairs that cannot
# be separated are named together in parentheses.
POLYMER_CLASSES = {
    'polyolefin_PE', 'polyolefin_PP', 'polyolefin_unresolved',
    'polystyrene', 'polyester_PET', 'PVC', 'nylon_PA', 'SAN', 'PVDF',
    'natural_isoprene',          # tyre wear particles: of interest in microplastic work,
                                 # so treated as a polymer here
    'acrylonitrile_copolymer',   # the Raman-side term. FT-IR calls this 'SAN' and
                                 # VOCAB_BRIDGE links the two. Without it here, to_tier()
                                 # would place a Raman acrylonitrile particle in non_polymer
                                 # and the particle would leak out as a tier mismatch.
}

# Non-polymer hints: the text placed inside the parentheses. Pairs that cannot be
# separated are written together with a slash.
NON_POLYMER_HINT = {
    'silicate':              'silicate/organic',   # Si-O window overlaps cellulose C-O
    'natural_organic':       'silicate/organic',   # same overlap, same hint
    'carbonate_calcite':     'carbonate',          # carbonate is certain, the polymorph is not
    'carbonate_aragonite':   'carbonate',
    'mineral_or_unresolved': 'silicate/organic',
}


def to_reporting(cls):
    """Detailed class -> reporting label. Polymers pass through; non-polymers are grouped
    and carry a hint."""
    if cls in POLYMER_CLASSES:
        return cls
    if cls in NON_CHEMICAL_STATES and cls not in NON_POLYMER_HINT:
        return 'abstain'
    h = NON_POLYMER_HINT.get(cls)
    return f'non_polymer ({h}?)' if h else 'non_polymer (?)'


def to_tier(cls):
    """Top tier of the three: polymer / non_polymer / abstain."""
    if cls in POLYMER_CLASSES:
        return 'polymer'
    if cls in NON_CHEMICAL_STATES and cls not in NON_POLYMER_HINT:
        return 'abstain'
    return 'non_polymer'


# Scoring: two non-polymers agreeing must not be counted as a correct identification.
# 95 of the 230 spectra (41%) are non_polymer, so scoring non_polymer against non_polymer
# as a hit would inflate agreement with a quantity unrelated to the question. The metric
# is therefore split in two.
SCORING_PRIMARY   = 'polymer_only'   # primary: score only particles both streams call polymer
SCORING_SECONDARY = 'tier_recall'    # secondary: are non-polymers recognised as non-polymer

print("reporting labels ready")
print(f"  polymer     : {len(POLYMER_CLASSES)} classes (kept distinct)")
print(f"  non_polymer : {sorted(set(NON_POLYMER_HINT.values()))} (hint in parentheses)")
print(f"  abstain     : QC and unresolved")
print(f"  scoring     : primary={SCORING_PRIMARY} · secondary={SCORING_SECONDARY}")

reporting labels ready
  polymer     : 11 classes (kept distinct)
  non_polymer : ['carbonate', 'silicate/organic'] (hint in parentheses)
  abstain     : QC and unresolved
  scoring     : primary=polymer_only · secondary=tier_recall


In [5]:
# === Phase gating helpers ==============================================
# _md() and date_key() are GLOBAL helpers. Re-binding either name as a variable in a later
# cell destroys the function, and the next call fails with "'int' object is not callable".
# Prefix temporaries in later cells with _t (_t_max, and so on).
def date_key(date):
    """'05.17' -> (2026,5,17)  |  '2025.10.14' -> (2025,10,14)"""
    p = str(date).split('.')
    if len(p) == 3: return (int(p[0]), int(p[1]), int(p[2]))
    if len(p) == 2: return (2026, int(p[0]), int(p[1]))
    return (9999, 99, 99)

def _md(mmdd):                       # "05.17" -> (5,17)
    y, m, d = date_key(mmdd); return (m, d)

def assign_phase(date, station=None):
    y, m, d = date_key(date)
    if y != 2026:
        return 'historical'
    k = (m, d)
    if k <= _md(PHASE1_END):   return 'phase1'
    if k <= _md(PHASE2_END):   return 'phase2'
    if k >= _md(HELDOUT_FROM): return 'heldout_july'
    return 'gap'

PHASE_ORDER = ['phase1', 'phase2', 'heldout_july', 'historical', 'gap']
print("phase gating:", f"phase1 <= {PHASE1_END} | phase2 <= {PHASE2_END} | heldout >= {HELDOUT_FROM}",
      "| PHASE2_S1_ONLY =", PHASE2_S1_ONLY)

phase gating: phase1 <= 06.01 | phase2 <= 06.28 | heldout >= 07.01 | PHASE2_S1_ONLY = True


### 0.1 Diagnostic band provenance ledger

Records where each band window in `features()` and each rule in `classify()` comes from (author-year keys). Read-only; takes no part in the assignment. Writes `SI_band_provenance.csv` and `SI_rule_provenance.csv`.

In [6]:
# === Provenance ledger for diagnostic bands and rules (author-year keys) =====
# Links every band window in features() and every rule in classify() to its source.
# Read-only: nothing here feeds the decision engine.
#   Requires OUT_DIR from the CONFIG cell.

import os
import pandas as pd

LIT_SOURCES = {
    # -- preprocessing --
    'Eilers2005':    dict(ref="P. H. C. Eilers and H. F. M. Boelens, Baseline Correction with "
                              "Asymmetric Least Squares Smoothing, Leiden Univ. Medical Centre, 2005.",
                          type='technical report (unpublished)',
                          note="ALS lambda=1e5, p=0.01 lie inside the ranges recommended there "
                               "(lambda 1e2-1e9, p 0.001-0.1). Cite as an unpublished technical "
                               "report and say so in a footnote."),
    'Barnes1989':    dict(ref="R. J. Barnes, M. S. Dhanoa and S. J. Lister, Appl. Spectrosc., "
                              "1989, 43, 772–777. doi:10.1366/0003702894202",
                          type='primary', note="Origin of SNV; snv() implements exactly this form."),
    'Savitzky1964':  dict(ref="A. Savitzky and M. J. E. Golay, Anal. Chem., 1964, 36, 1627–1639.",
                          type='primary',
                          note="SG second derivative. Window 15 and order 3 are a user choice, "
                               "not a literature value."),
    # -- polymer bands: measurement anchor and assignment sources --
    'Jung2018':      dict(ref="M. R. Jung et al., Mar. Pollut. Bull., 2018, 127, 704–716. "
                              "doi:10.1016/j.marpolbul.2017.12.061",
                          type='primary',
                          note="ATR-FTIR of reference materials, +/-4 cm-1 (Tables 1-2). Primary "
                               "anchor for all polymer bands and the basis of the +/-5 to 10 "
                               "matching windows."),
    'Asensio2009':   dict(ref="R. C. Asensio, M. S. A. Moya, J. M. de la Roja and M. Gomez, "
                              "Anal. Bioanal. Chem., 2009, 395, 2081–2096.", type='primary',
                          note="Source of the PE, PP and PS vibrational assignments."),
    'Noda2007':      dict(ref="I. Noda, A. E. Dowrey, J. L. Haynes and C. Marcott, in Physical "
                              "Properties of Polymers Handbook, Springer, 2007, pp. 395–406.",
                          type='handbook', note="PE, PP, PS assignments."),
    'Nishikida2003': dict(ref="K. Nishikida and J. Coates, in Handbook of Plastics Analysis, "
                              "Marcel Dekker, 2003, pp. 186–316.", type='handbook',
                          note="PE, PP, PS assignments."),
    'Beltran1997':   dict(ref="M. Beltran and A. Marcilla, Eur. Polym. J., 1997, 33, 1135–1142.",
                          type='primary', note="PVC assignments: 616 C-Cl, 1427, 1331, 1255."),
    'Rotter1992':    dict(ref="G. Rotter and H. Ishida, J. Polym. Sci. B, 1992, 30, 489–495.",
                          type='primary', note="Nylon amide I 1634, amide II 1538, N-H 3298."),
    'Coates2000':    dict(ref="J. Coates, in Encyclopedia of Analytical Chemistry, Wiley, 2000, "
                              "pp. 10815–10837.", type='handbook', note="Nitrile C-N triple bond 2237."),
    'Verleye2001':   dict(ref="G. A. Verleye, N. P. G. Roeges and M. O. De Moor, Easy Identification "
                              "of Plastics and Rubbers, Rapra, 2001.", type='handbook',
                          note="Aromatic and general fingerprint regions."),
    # -- PVDF, minerals, cellulose --
    'Cai2017':       dict(ref="X. Cai, T. Lei, D. Sun and L. Lin, RSC Adv., 2017, 7, 15382–15389. "
                              "doi:10.1039/c7ra01267e", type='primary',
                          note="FTIR of the PVDF alpha/beta/gamma phases: CF1180 (beta CH2 wag), "
                               "CF1402 (CH2 bend), CF876 (CF2). Cross-checked by XRD."),
    'Vagenas2003':   dict(ref="N. V. Vagenas, A. Gatsouli and C. G. Kontoyannis, Talanta, 2003, "
                              "59, 831–836.", type='primary',
                          note="Quantitative separation of calcite, aragonite and vaterite: "
                               "carb1420 (nu3), carb875 (nu2, calcite 874), carb712 (nu4), "
                               "arag855 (aragonite nu2, 855-858). Direct basis of the polymorph rule."),
    'Ellerbrock2024':dict(ref="R. H. Ellerbrock, M. Stein and J. Schaller, Front. Environ. Chem., "
                              "2024, 5, 1462678.", type='primary',
                          note="Si-O-Si asymmetric stretch, 1000-1090. Farmer 1974 may be cited "
                               "instead as the standard reference."),
    'Oh2005':        dict(ref="S. Y. Oh, D. I. Yoo, Y. Shin and G. Seo, Carbohydr. Res., 2005, "
                              "340, 417–428. doi:10.1016/j.carres.2004.11.027", type='primary',
                          note="Cellulose C-O and C-O-C stretch, 1020-1060. Ilharco 2000 covers "
                               "cellulose acetate and is therefore only indirect."),
    # -- rules --
    'Anshari2025':   dict(ref="R. Anshari, M. Tsuboi, H. Sato, K. Tashiro and Y. Ozaki, Sci. Rep., "
                              "2025, 15, 2518. doi:10.1038/s41598-025-85837-y", type='primary',
                          note="Naturally weathered PP loses the crystalline bands 998, 973, 842 "
                               "and 808 first. Direct basis of the polyolefin_unresolved rule."),
    'Almond2020':    dict(ref="J. Almond, P. Sugumaar, M. N. Wenzel, G. Hill and C. Wallis, "
                              "e-Polymers, 2020, 20, 369–381. doi:10.1515/epoly-2020-0041",
                          type='primary',
                          note="Original carbonyl index, 1850-1650 over 1500-1420. The windows used "
                               "here (1690-1760 over 1420-1490) are narrower, so report the quantity "
                               "as a MODIFIED carbonyl index and note the debate over its "
                               "reliability for PP."),
}

# -- band -> literature, one to one with the windows in features() ------------
#    'general' marks a band at the level of common IR knowledge: non-diagnostic,
#    never used on its own to decide a class.
BAND_PROVENANCE = {
    'CH2920': ['Jung2018','Asensio2009'],  'CH2850': ['Jung2018','Asensio2009'],
    'CH2950': ['Jung2018'],
    'b1472': ['Jung2018'], 'b1462': ['Jung2018'], 'b730': ['Jung2018'], 'b717': ['Jung2018'],
    'b1375': ['Jung2018'], 'b1450': ['Jung2018'], 'b998': ['Jung2018','Anshari2025'],
    'b973': ['Jung2018','Anshari2025'], 'b840': ['Jung2018','Anshari2025'],
    'b3025': ['Jung2018','Verleye2001'], 'b757': ['Jung2018'], 'b698': ['Jung2018'],
    'b1715': ['Jung2018','Almond2020'], 'b1240': ['Jung2018'], 'b1090': ['Jung2018'],
    'NH3300': ['Jung2018','Rotter1992'], 'amideI': ['Rotter1992'], 'amideII': ['Rotter1992'],
    'CN2237': ['Jung2018','Coates2000'], 'CCl': ['Jung2018','Beltran1997'],
    'b1250': ['Beltran1997'], 'CHdef1426': ['Beltran1997'], 'CHwag1330': ['Beltran1997'],
    'CF1180': ['Cai2017'], 'CF1402': ['Cai2017'], 'CF876': ['Cai2017'],
    'carb1420': ['Vagenas2003'], 'carb875': ['Vagenas2003'], 'carb712': ['Vagenas2003'],
    'arag855': ['Vagenas2003'],
    'SiO': ['Ellerbrock2024'], 'cellCO': ['Oh2005'],
    'OH3400': ['general'], 'water1640': ['general'], 'broadOH': ['general'],
}

RULE_PROVENANCE = {
    'ALS baseline (lambda=1e5, p=0.01)':                        ['Eilers2005'],
    'SNV':                                                      ['Barnes1989'],
    'SG 2nd derivative (window 15, order 3 = user choice)':      ['Savitzky1964'],
    'PP crystalline bands (998/973/841) lost first '
    '-> polyolefin_unresolved':                                  ['Anshari2025'],
    'carbonyl index (modified windows 1690-1760 / 1420-1490)':   ['Almond2020'],
    '1715 read as PE oxidation, not PET: carbonyl on a PE skeleton':
                                                                 ['Jung2018','Almond2020'],
    'SAN = nitrile 2237 + aromatic bands':                       ['Coates2000','Jung2018'],
    'PVC multi-band combination (CCl + 1250 + 1426/1330)':       ['Beltran1997','Jung2018'],
    'calcite / aragonite polymorph (855)':                       ['Vagenas2003'],
    'thresholds REL_PRESENT / REL_STRONG / SIO_DOMINATES / CI_WEATHERED':
                                                                 ['(data-tuned values, not from literature)'],
}


def provenance_report():
    """Write the two SI provenance tables and flag any band whose source key is unknown."""
    rows = [dict(band=b, sources=' ; '.join(ks),
                 diagnostic=(ks != ['general']),
                 unresolved_source=any(k not in LIT_SOURCES and not k.startswith('(')
                                       for k in ks if k != 'general'))
            for b, ks in BAND_PROVENANCE.items()]
    df = pd.DataFrame(rows)
    _bad = df[df.unresolved_source]
    print(f"bands {len(df)} | diagnostic {int(df.diagnostic.sum())} | "
          f"general (non-diagnostic) {int((~df.diagnostic).sum())}")
    if len(_bad):
        print("source key missing from LIT_SOURCES:", _bad.band.tolist())
    else:
        print("every diagnostic band is linked to the ledger")
    df.drop(columns='unresolved_source').to_csv(
        os.path.join(OUT_DIR, 'SI_band_provenance.csv'), index=False, encoding='utf-8-sig')
    pd.DataFrame([dict(rule=r, sources=' ; '.join(ks)) for r, ks in RULE_PROVENANCE.items()]
                ).to_csv(os.path.join(OUT_DIR, 'SI_rule_provenance.csv'), index=False, encoding='utf-8-sig')
    print(f"[saved] SI_band_provenance.csv, SI_rule_provenance.csv -> {OUT_DIR}/")
    return df


_prov = provenance_report()


bands 38 | diagnostic 35 | general (non-diagnostic) 3
every diagnostic band is linked to the ledger
[saved] SI_band_provenance.csv, SI_rule_provenance.csv -> /content/out/


### 0.2 Spectrum loader, preprocessing, band features, decision engine

In [7]:
# === File-name parser and CSV loader ========================================
# parse_id is the shared parser: both the spectrum file names and the OMNIC log use it.
def parse_id(token):
    """'S1_9-2' -> ('S1','9',2);  's1-3' -> ('S1','3',1);  'S3' -> ('S3','',1)."""
    t=token.strip()
    m=re.match(r'^[Ss](\d+)[_-](\d+)(?:-(\d+))?$', t)
    if m:
        st='S'+m.group(1); part=m.group(2); rep=int(m.group(3)) if m.group(3) else 1
        return st, part, rep
    m=re.match(r'^[Ss](\d+)$', t)              # bare station e.g. S3
    if m: return 'S'+m.group(1), '', 1
    return None, None, 1

def parse_filename(path):
    base = os.path.splitext(os.path.basename(path))[0]
    m = re.match(r'^(\d{2})(\d{2})(\d{2})(.*)$', base)      # YYMMDD + remainder (2025 onward)
    if m:
        date = f"20{m.group(1)}.{m.group(2)}.{m.group(3)}"  # 2025.08.28
        rest = m.group(4)
    else:
        m = re.match(r'^(\d{2})(\d{2})(.*)$', base)          # MMDD + remainder (short form)
        if not m: return None
        date = f"{m.group(1)}.{m.group(2)}"                  # 08.28, year unknown
        rest = m.group(3)
    st, part, rep = parse_id(rest)
    if st is None: return None        # unreadable station -> None, so a warning is raised
                                      # rather than the file being silently mis-ingested
    return dict(date=date, station=st, particle=part, replicate=rep, file=os.path.basename(path))

def load_csv(path):
    # utf-8-sig prevents load failures on CSVs that carry a BOM (e.g. 0601S1-8)
    try:
        d = np.loadtxt(path, delimiter=',', encoding='utf-8-sig')
    except TypeError:                      # numpy<1.14 fallback
        d = np.loadtxt(path, delimiter=',')
    x, y = d[:,0], d[:,1]; o = np.argsort(x); x, y = x[o], y[o]
    return GRID.copy(), np.interp(GRID, x, y)

In [8]:
# === Preprocessing: ALS baseline, SNV, SG derivative, masking, fringe detection ===
# Method provenance:
#   ALS baseline (als_baseline, lambda=1e5, p=0.01) -> Eilers & Boelens 2005
#       unpublished technical report; lambda and p lie inside the recommended ranges
#       (lambda 1e2-1e9, p 1e-3-1e-1)
#   SNV (snv)                                       -> Barnes, Dhanoa & Lister 1989
#   SG 2nd derivative (savgol_filter, win=15, ord=3) -> Savitzky & Golay 1964
#       window 15 and order 3 are a user choice, not a literature value
#   CO2 masking (MASK_REGIONS 2280-2400)            -> atmospheric CO2 near 2349 (standard)
#   detect_fringe(score>8) and ref = 99th percentile are heuristics, not from the literature.
def als_baseline(y, lam=ALS_LAM, p=ALS_P, niter=10):
    L=len(y); D=sparse.diags([1.,-2.,1.],[0,-1,-2],shape=(L,L-2)).tocsc(); D=lam*D.dot(D.T)
    w=np.ones(L)
    for _ in range(niter):
        W=sparse.spdiags(w,0,L,L); z=spsolve((W+D).tocsc(), w*y); w=p*(y>z)+(1-p)*(y<z)
    return z

def snv(y): return (y-y.mean())/(y.std()+1e-9)

def detect_fringe(x, corr):
    """Thin-film interference -> quasi-periodic ripple in a normally flat window (2000-2700)."""
    m=(x>=2000)&(x<=2700); seg=corr[m]
    if len(seg)<64: return False, 0.0
    seg=seg-savgol_filter(seg, 51 if len(seg)>51 else len(seg)//2*2+1, 2)
    f=np.abs(np.fft.rfft(seg-seg.mean()))
    if f[1:].max()<=0: return False, 0.0
    score=float(f[2:].max()/(f[1:].mean()+1e-9))   # dominant ripple vs background
    return score>8.0, score

def preprocess(x, y):
    base=als_baseline(y); corr=np.clip(y-base, 0, None)
    for lo,hi in MASK_REGIONS:                       # atmospheric (CO2) masking
        corr[(x>=lo)&(x<=hi)]=0.0
    fr, fscore=detect_fringe(x, corr)
    # snv and d2 are diagnostic outputs only: features() never consumes them, reading just
    # corr and ref. SG_WIN and SG_ORD therefore cannot affect any assignment (proved in 0.10).
    return dict(corr=corr, base=base, snv=snv(corr),
                d2=savgol_filter(corr, SG_WIN, SG_ORD, deriv=2),
                ref=np.percentile(corr,99)+1e-9, fringe=fr, fringe_score=fscore)

In [9]:
# === Band features: each window measured relative to the strongest signal (ref) ====
# Band provenance is held in one place, the ledger cell (BAND_PROVENANCE); this is only a
# summary. Polymers: Jung2018, with assignments from Asensio2009, Noda2007, Nishikida2003,
# Beltran1997 (PVC), Rotter1992 (nylon) and Coates2000 (nitrile). PVDF: Cai2017.
# Carbonate and aragonite: Vagenas2003. Silicate: Ellerbrock2024. Cellulose: Oh2005.
# OH3400, water1640 and broadOH are general O-H / water bands and are non-diagnostic.
#
# The windows are DATA, not code: exposing them as BAND_WINDOWS lets section 0.5 check them
# against pure standards. If a standard shows a window to be wrong, correct it here only,
# raise PARAM_VERSION and re-run end to end.

BAND_WINDOWS = {
  'CH2920': (2915, 2925),
  # The CH2 symmetric stretch is near 2848 in PE but near 2837 in PP, shifted down by the
  # methyl substitution. The former window 2845-2855 was PE-only and missed PP: measured
  # 0.280 (below rel) against a true peak of 0.717, which left ch_back False and skipped the
  # whole polyolefin branch (Sigma-Aldrich 427888, section 0.5). The upper limit of 2856
  # keeps the PP CH3 symmetric stretch (~2867) outside the window.
  'CH2850': (2832, 2856),
  'CH2950': (2945, 2960),
  'b1472': (1468, 1476),
  'b1462': (1458, 1466),
  'b730': (726, 734),
  'b717': (713, 721),
  'b1375': (1370, 1380),
  'b1450': (1445, 1458),
  'b998': (993, 1003),
  'b973': (968, 978),
  'b840': (835, 845),
  'b3025': (3018, 3032),
  'b757': (752, 762),
  'b698': (693, 703),
  'b1715': (1690, 1725),
  'b1240': (1230, 1250),
  'b1090': (1080, 1100),
  'NH3300': (3270, 3330),
  'amideI': (1630, 1660),
  'amideII': (1530, 1560),
  'CN2237': (2230, 2245),
  'CCl': (600, 700),
  'b1250': (1240, 1260),
  # --- for the PVC combination ---
  'CHdef1426': (1420, 1432),
  'CHwag1330': (1325, 1340),
  'CF1180': (1170, 1195),
  'CF1402': (1395, 1410),
  'CF876': (870, 884),
  # --- minerals ---
  'carb1420': (1400, 1450),
  'carb875': (870, 880),
  'carb712': (708, 716),
  'arag855': (850, 860),
  'SiO': (1000, 1090),
  'OH3400': (3350, 3450),
  'water1640': (1632, 1648),
  'broadOH': (3200, 3500),
  # --- cellulose C-O ---
  'cellCO': (1020, 1060),
}

# _h and _a are GLOBAL helpers used by features(). Do not re-bind these names later.
def _h(x, c, lo, hi):
    m = (x >= lo) & (x <= hi); return float(c[m].max()) if m.any() else 0.0
def _a(x, c, lo, hi):
    m = (x >= lo) & (x <= hi); return float(_TRAPZ(c[m], x[m])) if m.sum() > 1 else 0.0

def _noise(c):
    """Robust standard deviation of the first difference. Peaks are sparse, so the
    background dominates."""
    d = np.diff(c)
    return 1.4826 * float(np.median(np.abs(d - np.median(d)))) / np.sqrt(2) + 1e-12


def features(x, pp):
    c, ref = pp['corr'], pp['ref']
    f = {k: _h(x, c, lo, hi) / ref for k, (lo, hi) in BAND_WINDOWS.items()}
    # Per-band S/N, the second criterion for presence: rel measures height, S/N measures
    # detectability.
    _nz = _noise(c)
    f['_noise'] = _nz
    for k, (lo, hi) in BAND_WINDOWS.items():
        f['snr_' + k] = _h(x, c, lo, hi) / _nz
    f['totalA'] = _a(x, c, 400, 4000)
    f['carbonyl_num'] = _a(x, c, 1690, 1760); f['carbonyl_den'] = _a(x, c, 1420, 1490)
    # Band sharpness: a thin film or poor ATR contact gives a weak, featureless mid-IR.
    f['feature_density'] = float(np.mean(np.abs(np.diff(c)) > 0.01 * ref))
    return f
print(f"features() ready | {len(BAND_WINDOWS)} windows")

features() ready | 38 windows


In [10]:
# === Decision engine: skeleton-first, hierarchical rules =====================
# Rule provenance is held in one place, RULE_PROVENANCE in section 0.1. In summary:
#   weathering rule (PP crystalline 998/973/841 lost first -> polyolefin_unresolved)
#     = Anshari2025; carbonyl index = Almond2020 (the windows here are narrower, a MODIFIED
#     CI); band assignments = BAND_PROVENANCE. Thresholds are data-tuned, not literature.
# Rules are read top down and THE FIRST ONE THAT MATCHES IS THE ANSWER. If none does, the
# engine abstains.

def band_present(f, k):
    """Band presence.
       rel >= REL_PRESENT,
       or, for a whitelisted band, rel >= REL_PRESENT_WEAK together with snr >= SNR_PRESENT.
       Relaxing every band instead would produce a burst of false positives in the classes
       whose evidence is a single band (SAN, PVDF) - see the CONFIG comments."""
    v = f[k]
    if v >= REL_PRESENT:
        return True
    if not USE_SNR_PRESENCE or k not in SNR_RELAXABLE:
        return False
    return v >= REL_PRESENT_WEAK and f.get('snr_' + k, 0.0) >= SNR_PRESENT


def classify(f):
    P=lambda k: band_present(f, k)
    S=lambda k: f[k]>=REL_STRONG          # 'strong', for the CONFIDENCE tier (no S/N)
    D=lambda k: f[k]>=REL_DOMINANT        # 'dominant', for CLASS decisions and vetoes

    # ---- QC gate: is this spectrum fit for a chemical assignment at all? ----
    if f['totalA'] < QC_MIN_TOTAL_A:
        return 'QC_low_absorbance','unresolved',['low total absorbance']
    diag=any(P(b) for b in ['CH2920','b1715','CCl','NH3300','carb875','SiO','b698','CF1180'])
    if P('OH3400') and P('water1640') and not diag:
        return 'QC_water_dominated','unresolved',['3400 broad + 1640, no diagnostics']

    ch_back=P('CH2920') and P('CH2850')
    back_lvl=max(f['CH2920'],f['CH2850'])
    sio=f['SiO']; carb=f['carb1420'] if P('carb875') else 0.0
    min_lvl=max(sio,carb)

    # ---- guard against over-calling minerals: a strong mineral band must not swallow
    #      an organic one. Mineral dominance requires all three: the band is dominant
    #      (REL_DOMINANT), it exceeds the CH skeleton by SIO_DOMINATES, and no polymer
    #      signature is present.
    polymer_sig = P('b1715') or P('b698') or P('CN2237') or P('NH3300') or ch_back
    mineral_dom = (min_lvl>=REL_DOMINANT) and (min_lvl > back_lvl*SIO_DOMINATES) and not polymer_sig

    if mineral_dom and ch_back:
        cls='silicate' if sio>=carb else 'carbonate_calcite'
        return cls,'tentative',[f'mineral dominant ({min_lvl:.2f}) > CH ({back_lvl:.2f})x{SIO_DOMINATES}; mixed overlay']

    # ---- 1. polyolefin skeleton first. Under the weathering rule, 1715 on a PE skeleton
    #         is oxidation, not PET.
    if ch_back:
        strong = S('CH2920') or S('CH2850')
        pe_doublet = P('b1472') and P('b730') and P('b717')
        pp_cryst   = P('b1375') and (P('b998') or P('b973') or P('b840'))
        ci = (f['carbonyl_num']/f['carbonyl_den']) if f['carbonyl_den']>0 else 0.0
        weathered = ci > CI_WEATHERED
        # The PP crystalline bands (998/973/841) are the first to disappear on weathering,
        # leaving only the 1472 + 720 doublet, which looks like PE. Hence: (a) PP only when
        # the crystalline evidence is explicit; (b) a clean PE doublet is called PE but is
        # never 'confident' from FT-IR alone and needs Raman confirmation; (c) weathering
        # plus a doublet alone cannot separate PE from PP -> polyolefin_unresolved.
        if pp_cryst:
            conf = 'confident' if (strong and not weathered) else 'tentative'
            why = ['CH3 backbone','1375','998/973/840'] + (['weathered->cap'] if weathered else [])
            return 'polyolefin_PP', conf, why
        # ---- how far CI may intervene is set by CI_GATE (see the CONFIG comments) ----
        #   'incomplete_only' : a COMPLETE doublet (1472+730+717) means a complete PE
        #                       fingerprint, so CI does not withdraw it. CI withholds only
        #                       on a partial fingerprint.
        #   'always'          : withholds even when the doublet is complete.
        if pe_doublet and (CI_GATE == 'incomplete_only' or not weathered):
            why = ['CH2 backbone','1472','730/717 doublet','FT-IR cannot resolve PE/PP -> Raman']
            if P('b1715'): why.append('1715=OXIDATION on PE (NOT PET)')
            if weathered:
                why.append(f'oxidised (CI {ci:.2f}) - doublet complete, not withheld')
            return 'polyolefin_PE', 'tentative', why
        if pe_doublet and weathered:      # reached only when CI_GATE == 'always'
            return 'polyolefin_unresolved', 'tentative', \
                   ['CH2 doublet + carbonyl: weathered, PE and PP not separable','Raman-required']
        # Partial doublet: this is the only place where CI decides to withhold.
        # Plant and cellulose material also gives ch_back True with a high CI (lignin and
        # adsorbed water). Measured on 06.14 S1-9: broad 3300 O-H plus a strong 1030 C-O, a
        # plant fragment, was captured here as polyolefin_unresolved (OMNIC also returned
        # Peppermint Leaf, 86.9). A plant signature therefore skips this branch and falls
        # through to the natural_organic test below.
        # P('broadOH') cannot serve as that test: the cellulose O-H band is 300-400 cm-1 wide
        # and ALS (lam=1e5) absorbs it into the baseline, leaving rel below 0.01 (measured).
        # The baseline-insensitive criterion is whether C-O (1030) exceeds the CH stretch:
        # plants give cellCO/CH2920 around 2.7, pure PE around 0.00, which separates cleanly.
        plant_sig = P('cellCO') and f['cellCO'] > f['CH2920']
        if CI_GATE == 'incomplete_only' and weathered and not plant_sig and \
           (P('b1472') or P('b730') or P('b717')):
            return 'polyolefin_unresolved', 'tentative', \
                   [f'partial CH2 fingerprint + carbonyl (CI {ci:.2f}): PE/PP not separable','Raman-required']
        # fall through: aliphatic but no clean doublet -> keep checking natural/pigment below

    # ---- 2. industrial polymers settled by a combination of bands ----
    if P('CF1180') and P('CF1402') and P('CF876'):
        return 'PVDF','confident',['C-F 1402/1180/876']
    if P('b3025') and P('b698') and P('b757'):
        # Aromatic bands alone are PS; SAN requires the nitrile as well.
        if P('CN2237'):
            return 'SAN','confident',['aromatic 3025/757/698 + nitrile 2237']
        return 'polystyrene','confident',['aromatic 3025/757/698']

    # ---- SAN = nitrile + aromatic. A CH skeleton only raises confidence; it is not required.
    if P('CN2237') and P('b698'):
        conf='confident' if ch_back else 'tentative'
        return 'SAN',conf,['nitrile 2237 + aromatic'+(' + CH backbone' if ch_back else '')]

    # ---- PVC requires a COMBINATION of bands, and a dominant Si-O holds a veto ----
    pvc_combo = P('CCl') and P('b1250') and (P('CHdef1426') or P('CHwag1330'))
    if pvc_combo and not (sio>=REL_DOMINANT and sio>f['CCl']*SIO_DOMINATES):
        return 'PVC',('confident' if S('CCl') else 'tentative'),['C-Cl 600-700 + 1250 + CH(1426/1330)']

    if P('NH3300') and P('amideI') and P('amideII'):
        return 'nylon_PA','confident',['N-H 3300 + amide I/II']
    if P('b1715') and P('b1240') and P('b1090') and not ch_back:
        return 'polyester_PET','confident',['1715 + 1240/1090, no CH backbone']
    if P('carb875') and P('carb1420'):
        return ('carbonate_aragonite' if P('arag855') else 'carbonate_calcite'),'tentative',['carbonate 1420/875; polymorph provisionally split by 855, not verified by Raman']

    # ---- weak, featureless mid-IR is a thin film or poor ATR contact, not a pigment ----
    #      Calling it a pigment would overreach, so it is named honestly as a QC state.
    #      Identifying real pigments is left to Raman / SLoPP-E.
    if (not ch_back) and (not P('SiO')) and (not P('carb875')) \
       and f['feature_density']<0.10 and f['totalA']>=QC_MIN_TOTAL_A:
        return 'QC_low_contrast','unresolved',['weak/featureless mid-IR: thin film or poor ATR contact (not a pigment)']

    # ---- natural organic matter is tested before the silicate fallback ----
    # Plant cuticle and cellulose give a broad O-H with C-O/C-H and often a shoulder in the
    # Si-O region. The class is passed to silicate only when Si-O strongly overwhelms the
    # organic signal.
    organic_sig = max(f['broadOH'], f['OH3400'], f['CH2920'])
    if P('broadOH') and (P('cellCO') or P('b1090') or P('CH2920')) \
       and not (f['SiO']>=REL_DOMINANT and f['SiO'] > organic_sig*SIO_DOMINATES):
        return 'natural_organic','tentative',['broad O-H + C-O/C-H (cellulose-like); Si-O not strongly dominant']

    # ---- residual rules: elimination, not positive identification. Describe them that way
    #      in the manuscript.
    if ch_back:
        return 'polyolefin_unresolved','tentative',['aliphatic CH, no clean PE/PP doublet; possible Si-O shoulder']
    if D('SiO'):
        return 'silicate','tentative',['Si-O strong, no polymer backbone (sand/glass)']
    if P('SiO'):
        return 'mineral_or_unresolved','unresolved',['weak Si-O only; ambiguous C-O region']
    return 'unresolved','unresolved',['no diagnostic band pattern']

def mixed_flag(f):
    P=lambda k: band_present(f, k)
    mineral=P('SiO') or (P('carb875') and P('carb1420'))
    polymer=(P('CH2920') and P('CH2850')) or P('b1715') or (P('CCl') and P('b1250'))
    return bool(mineral and polymer)

def carbonyl_index(f, cls):
    if CI_POLYOLEFIN_ONLY and not cls.startswith('polyolefin'): return np.nan
    return f['carbonyl_num']/f['carbonyl_den'] if f['carbonyl_den']>0 else np.nan
print("classify() ready")

classify() ready


### 0.3 Load all spectra (parse name, preprocess, cache band features); shared utilities

In [ ]:
# === Load all spectra: parse name -> load -> preprocess -> cache band features ====
# Every later section reads only this cache (_feat_cache). Nothing is classified here.
import glob
paths = sorted(glob.glob(os.path.join(SPECTRA_DIR, '*.CSV')) +
               glob.glob(os.path.join(SPECTRA_DIR, '*.csv')))
if not paths:
    # Do not mistake Colab's default sample_data (california_housing, mnist) for spectra,
    # and note that the Code 1 hand-off artefacts are CSVs too.
    _EXCLUDE_DIRS = ('sample_data', '.ipynb_checkpoints',
                     'raman_manual_out',          # Code 1 hand-off folder
                     os.path.basename(OUT_DIR))   # this notebook's own output folder
    _EXCLUDE_FILES = ('ground_truth_for_code2.csv', 'worksheet_template.csv',
                      'method_comparison.csv', 'sens_', 'null_A1_', 'band_')
    paths = sorted(p for p in glob.glob('/content/**/*.CSV', recursive=True) +
                                glob.glob('/content/**/*.csv', recursive=True)
                   if not any(d in p.split(os.sep) for d in _EXCLUDE_DIRS)
                   and not os.path.basename(p).startswith(_EXCLUDE_FILES))
    if paths:
        print(f"(SPECTRA_DIR empty - falling back to {len(paths)} CSV(s) found elsewhere)")

if not paths:
    raise FileNotFoundError(
        f"no spectrum CSV found.\n  SPECTRA_DIR = {SPECTRA_DIR}\n"
        f"  Upload the FT-IR spectra to that folder, or correct SPECTRA_DIR.\n"
        f"  Name format: MMDD + station/particle (e.g. 0601S1-8.csv) or YYMMDD "
        f"(e.g. 250828S1-3.csv)")
print(f"spectra files: {len(paths)}")

# ── guard: were the preceding cells run? ───────────────────────────────────
#   This cell depends on the functions defined in 0.2. If one is missing, the loop below
#   swallows the same NameError on every file and reports the WRONG diagnosis ("check the
#   CSV format"), so the condition is caught here instead.
_REQUIRED = {
    'assign_phase'  : "0.0 phase gating helpers",
    'parse_filename': "0.2 file-name parser / CSV loader",
    'load_csv'      : "0.2 file-name parser / CSV loader",
    'preprocess'    : "0.2 preprocessing  (prints nothing)",
    'features'      : "0.2 band features  (prints 'features() ready')",
    'classify'      : "0.2 decision engine  (prints 'classify() ready')",
}
_missing = [f"{k}()   <- {v}" for k, v in _REQUIRED.items() if not callable(globals().get(k))]
if _missing:
    raise NameError(
        "a preceding cell has not been run - these functions are undefined:\n  "
        + "\n  ".join(_missing)
        + "\n\n  Use Runtime > Run all, or run the cells above in order and then re-run this"
          "\n  one. The CSV files are fine; this is an execution-order problem.")

_feat_cache = {}; _meta = {}; _load_fail = []
for p in paths:
    meta = parse_filename(p)
    if meta is None:
        print('  ! unparsed filename:', os.path.basename(p))
        _load_fail.append((os.path.basename(p), 'unparsed')); continue
    try:
        x, y = load_csv(p); pp = preprocess(x, y); f = features(x, pp)
        _feat_cache[p] = (x, pp, f); _meta[p] = meta
    except (NameError, AttributeError, ImportError) as e:
        # An environment error, not a problem with this file. It will repeat on every file,
        # so stop immediately rather than filling the log with identical failures.
        raise type(e)(
            f"{type(e).__name__} while processing {os.path.basename(p)}: {e}\n"
            "  This is an execution-order or environment problem, not a file problem "
            "(the same error would repeat on every file).\n"
            "  Use Runtime > Run all to execute section 0 from the start.") from e
    except Exception as e:
        print('  ! feature error', os.path.basename(p), e)
        _load_fail.append((os.path.basename(p), str(e)))

# Guard: without this, inv['date'] below fails with a KeyError that hides the real cause.
if not _feat_cache:
    _n_unparsed = sum(1 for _, r in _load_fail if r == 'unparsed')
    raise RuntimeError(
        f"the feature cache is empty - all {len(paths)} file(s) failed "
        f"(name parsing {_n_unparsed}, load or feature errors {len(_load_fail)-_n_unparsed}).\n"
        f"  files found: {[os.path.basename(p) for p in paths[:8]]}\n"
        f"  If parsing failed, these CSVs are probably not spectra - check SPECTRA_DIR.\n"
        f"  If loading failed, check the CSV format (two columns: wavenumber, absorbance).")

# inventory by phase and station
inv = pd.DataFrame([dict(path=p, **_meta[p]) for p in _feat_cache])
inv['phase'] = [assign_phase(d, s) for d, s in zip(inv['date'], inv['station'])]
print(f"\nfeature-cached: {len(inv)} / {len(paths)}  |  failed: {len(_load_fail)}")
display(pd.crosstab(inv['phase'], inv['station'], margins=True).reindex(
        [p for p in PHASE_ORDER if p in inv['phase'].unique()] + ['All']))
print("\nphase x date:")
display(pd.crosstab([inv['phase'], inv['date']], inv['station'], margins=True))

spectra files: 230


In [ ]:
# === Shared utilities ======================================================
#   run_pipeline()        applies classify(); OMNIC never intervenes
#   attach_omnic()        record-only merge; never alters an assignment (4.3 and 5)
#   class_accumulation()  accumulation curve over chemical classes (Part 1)
#   wilson_ci()           small-sample confidence interval for a proportion
def run_pipeline(paths_subset, tag=""):
    """Apply classify() to a subset. Stream A only - OMNIC never intervenes."""
    recs=[]; seen={}
    for p in paths_subset:
        meta=dict(_meta[p])
        key=(meta['date'],meta['station'],meta['particle'],meta['replicate'])
        meta['duplicate_key']= key in seen; seen[key]=seen.get(key,0)+1
        try:
            x,pp,f=_feat_cache[p]
            cls,conf,why=classify(f)
            ci=carbonyl_index(f,cls); mx=mixed_flag(f)
            meta.update(dict(spec_class=cls, confidence=conf, carbonyl_index=ci, mixed=mx,
                             fringe=pp['fringe'], totalA=f['totalA'],
                             feature_density=f['feature_density'], reasons='; '.join(why)))
        except Exception as e:
            meta.update(dict(spec_class='ERROR', confidence='unresolved', reasons=str(e)))
        recs.append(meta)
    d=pd.DataFrame(recs)
    d['phase']=[assign_phase(dt,st) for dt,st in zip(d['date'],d['station'])]
    d['is_chemical']=~d['spec_class'].isin(NON_CHEMICAL_STATES)
    print(f"[{tag}] classified: {len(d)} rows | thresholds P={REL_PRESENT:.2f} S={REL_STRONG:.2f} (fixed) "
          f"| chemical-class rows {int(d['is_chemical'].sum())} | state rows {int((~d['is_chemical']).sum())} "
          f"| dup-key {int(d['duplicate_key'].sum())}")
    return d

def attach_omnic(d, omnic_df):
    """Record-only merge for comparison; spec_class and confidence are never changed."""
    return d.merge(omnic_df[['date','station','particle','replicate','omnic_name','hqi']],
                   on=['date','station','particle','replicate'], how='left')

def class_accumulation(labels, title, ax=None, color=None, save=None, chemical_only=True):
    """Shuffle the particle order repeatedly and accumulate the number of chemical classes
       seen so far. chemical_only=True (default) drops the abstention states
       (NON_CHEMICAL_STATES) and ERROR. Nothing is classified here: the function returns
       the curve and descriptive statistics only."""
    drop = NON_CHEMICAL_STATES if chemical_only else {'ERROR'}
    kept = [l for l in labels if l not in drop]
    n_drop = len(labels) - len(kept)
    if len(kept) < 2:
        print(f"[{title}] fewer than 2 chemical classes - skipped"); return None
    rng = np.random.default_rng(RAREF_SEED); arr = np.array(kept); n = len(arr)
    curve = np.zeros((RAREF_REPS, n))
    for r in range(RAREF_REPS):
        seen=set()
        for i,l in enumerate(rng.permutation(arr)):
            seen.add(l); curve[r,i]=len(seen)
    mean, sd = curve.mean(0), curve.std(0)
    xs = np.arange(1, n+1)
    tail = max(1, int(n*TAIL_FRAC)); gain = float(mean[-1]-mean[-tail])
    own = ax is None
    if own: fig, ax = plt.subplots(figsize=(7,4.5))
    ax.plot(xs, mean, lw=2, label=f"{title} (n={len(kept)})", color=color)
    ax.fill_between(xs, mean-sd, mean+sd, alpha=.2, color=color)
    ax.set_xlabel('particles analyzed'); ax.set_ylabel('distinct chemical spec_class')
    ax.grid(alpha=.3)
    if own:
        ax.set_title(f'Class accumulation: {title}'); ax.legend(); plt.tight_layout()
        if save: plt.savefig(os.path.join(OUT_DIR,save),dpi=130)
        plt.show()
    res = dict(title=title, n=len(kept), n_state_dropped=n_drop,
               classes=int(mean[-1]), tail_n=tail, tail_new=round(gain,2))
    print(f"[{title}] chemical n={res['n']} (states dropped {n_drop}) | {res['classes']} classes | "
          f"{gain:.2f} new classes in the last {tail} particles")
    return res

def wilson_ci(k, n, z=1.96):
    if n==0: return (np.nan, np.nan, np.nan)
    p=k/n; d=1+z*z/n
    c=(p+z*z/(2*n))/d
    h=(z/d)*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return p, max(0,c-h), min(1,c+h)

print("utils ready: run_pipeline / attach_omnic / class_accumulation / wilson_ci")

### 0.4 OMNIC library match table: the verbatim record stream

A transcription of the laboratory log. It takes no part in the assignment and is not translated into chemical classes: the name and HQI are attached to each particle exactly as recorded and used only in 4.3 and 5. A mapping such as `'duct tape' -> polyolefin_PE` would be an assumption by the authors, so 4.3 scores by cluster purity, which needs no shared vocabulary.

In [ ]:
# === OMNIC library match table (name + HQI), transcribed from the lab log =========
# Format per line:  "Sxx_yy | Match name, HQI".  Blank match -> no hit.
OMNIC_RAW = """
# 03.15
03.15 S1_1  | Triethylene glycol monoethyl ether, 65.8
03.15 S1_2  | Mugo Pine, 81.50
03.15 S1_3  | Dandelion, 84.77
03.15 S1_4  | Carrot Seeds, 65.95
03.15 S1_5  | Mugo Pine, 77.83
03.15 S1_6  | Mugo Pine, 81.50
03.15 S1_7  |
03.15 S1_8  | Paromomycin sulfate, 64.21
03.15 S1_9  | Analcime #1, 0.086 wt %, 79.00
03.15 S1_10 | Analcime #1, 0.086 wt %, 77.68
03.15 S2_1  | Poly(bisphenol a-co-epichlorohydrin), glycidyl end-capped, a, 85.51
03.15 S2_2  | Newsprint (black ink), 83.80
03.15 S2_3  | Red Pepper Seed, 65.11
03.15 S3_1  | Saponite, 92.25
# 03.16
03.16 S1_1  | Hoover 512-BU, 84.11
03.16 S1_2  | Hoover 512-BU, 78.75
03.16 S1_3  | Licorice Root, 68.06
03.16 S1_4  | Felly's Professional Mix Topsoil, 75.47
03.16 S1_5  | Felly's Professional Mix Topsoil, 76.28
03.16 S1_6  | Newsprint (color ink), 79.62
03.16 S1_7  | Mugo Pine, 77.06
03.16 S1_8  | Mugo Pine, 80.10
03.16 S1_9  | Felly's Professional Mix Topsoil, 81.95
03.16 S1_10 | Formaldehyde, 37 wt.% solution in water, 77.61
03.16 S2_1  | Felly's Professional Mix Topsoil, 76.49
03.16 S2_2  | Red Pepper Seed, 80.76
03.16 S2_3  | Felly's Professional Mix Topsoil, 76.28
03.16 S3_1  | Saponite, 93.00
# 03.27
03.27 S1_1  | Cement, cured, 64.93
03.27 S1_2  | Polyblend Ceramic Tile Caulk (cured), 79.06
03.27 S1_3  | Plumber's Putty, 73.03
03.27 S1_4  | Plumber's Putty, 80.94
03.27 S1_5  | Felly's Professional Mix Topsoil, 77.29
03.27 S1_6  | ALUMINA SILICATE #1, 75.89
03.27 S1_7  | Parking Lot Tar, 61.97
03.27 S2_1  | Lead Monosilicate, 59.81
03.27 S2_2  | Lead Monosilicate, 80.00
03.27 S2_3  | Lead Monosilicate, 55.09
03.27 S2_4  | Deep Rose blush, 76.34
03.27 S2_5  | Deep Rose blush, 81.53
03.27 S3_1  | Busan 11-M1, 64.28
03.27 S3_2  | Deep Rose blush, 75.83
03.27 S3_3  | White Play Sand, 90.16
03.27 S3_4  | Deep Rose blush, 60.89
# 03.28
03.28 S1_1  | POLY(VINYLIDENE FLUORIDE), 86.49
03.28 S1_2  | Poly(vinyl chloride), inherent viscosity 0.92, 88.70
03.28 S1_3  | Deep Rose blush, 84.34
03.28 S1_4  | Wheat Gluten fluor, 79.27
03.28 S1_5  | Red Pepper Seed, 80.22
03.28 S1_6  | Spanish Saffron, 68.58
03.28 S1_7  | Styrene/acrylonitrile copolymer 32%, 88.20
03.28 S1_8  | Paraffin wax flakes mp 65 deg C min, 92.10
03.28 S1_9  | POLYEHTYLENE (Mn 1400), 97.39
03.28 S2_1  | Plumber's Putty, 79.47
03.28 S2_2  | Comet Disinfectant Cleaner, 95.34
03.28 S2_3  | Wheat Gluten fluor, 79.27
03.28 S2_4  | AJAX with Bleach Cleanser, 89.16
03.28 S2_5  | Plumber's Putty, 88.29
03.28 S3_1  | SILICA SAND, 83.55
03.28 S3_2  | Cement, cured, 68.60
# 04.21
04.21 S1_1  | Comet Disinfectant Cleaner, 92.72
04.21 S1_2  | Japanese maple, 73.80
04.21 S1_3  | Comet Disinfectant Cleaner, 92.85
04.21 S1_4  | AJAX with Bleach Cleanser, 94.27
04.21 S1_5  | Cement, cured, 90.6
04.21 S2_1  | Polyester tere- and isophthalic acids, 68.37
04.21 S2_2  | Phenoxy resin nom mw 30000, 87.67
04.21 S2_3  | Comet Disinfectant Cleaner, 91.20
04.21 S2_4  | Zein purified, 86.67
04.21 S2_5  | Cement, cured, 90.29
04.21 S3_1  | Diopside, 70.82
04.21 S3_2  | PELLYITE, 61.06
04.21 S3_3  | Diopside, 66.84
# 05.10
05.10 S1_1  | Poly(bisphenol a-co-epichlorohydrin) average MW ca 40000, 92.75
05.10 S1_2  | Deep Rose blush, 69.87
05.10 S1_3  | POLYESTER(TS) #3, 65.07
05.10 S1_4  | Felly's Professional Mix Topsoil, 86.68
05.10 S1_5  | Felly's Professional Mix Topsoil, 78.85
05.10 S1_6  | Mugo Pine, 81.46
05.10 S1_7  | Vinyl Gloves, 85.62
05.10 S1_8  | Nylon 6, 88.89
05.10 S1_9  | Poly(bisphenol a-co-epichlorohydrin) average MW ca 40000, 79.60
05.10 S1_9-2 | Phenoxy resin low m.w., 76.40
05.10 S1_10 | Felly's Professional Mix Topsoil, 77.80
05.10 S1_11 | BAYLITH 1 (POWDER), 77.72
05.10 S1_12 | Hornblende, 62.22
05.10 S1_13 | Face powder, 67.28
05.10 S2_1  | Cement, cured, 60.32
05.10 S2_2  | ALUMINA SILICATE #1, 69.17
05.10 S3_1  | Felly's Professional Mix Topsoil, 79.32
05.10 S3_2  | Painter's Masking Tape (sticky side), 90.77
05.10 S3_3  | Methyl formate, 79.23
05.10 S3_4  | Plumber's Putty, 90.58
# 05.17
05.17 S1_1  | Duct tape (Back side), 94.05
05.17 S1_2  | Duct tape (Back side), 97.85
05.17 S1_3  | Polypropylene+Vistalon 404 1:1, 77.53
05.17 S1_4  | Duct tape (Back side), 98.34
05.17 S1_5  | Duct tape (Back side), 96.84
05.17 S1_6  | Castor Bean, 73.45
05.17 S1_7  | ARC 1001 (cast), 89.79
05.17 S1_8  | Deep Rose blush, 66.47
05.17 S2_1  | NAT PALE CREPE RUB #1, 84.97
05.17 S2_2  | Polypropylene isotactic, 93.77
05.17 S2_3  | Polyester 41% soybean oil modified phthalic alkyd, 79.49
05.17 S2_4  | Myristic acid 99.5+%, 91.69
05.17 S2_5  | Cement, cured, 72.10
05.17 S3_1  | Plumber's Putty, 77.34
05.17 S3_2  | Formaldehyde 37 wt.% solution in water, 65.70
05.17 S3_3  | Plumber's Putty, 84.45
05.17 S3_4  | Visa card (blue plastic), 85.59
05.17 S3_5  | AJAX with Bleach Cleanser, 91.83
05.17 S3_6  | Black enamel paint, 67.92
05.17 S3_7  | Bleach, 80.95
05.17 S3_8  | Urethane Sealant, 87.28
# 06.01
06.01 S1_1  | Crayon (red), 84.36
06.01 S1_2  | Ethyl triacontanoate 98%, 78.36
06.01 S1_3  | HazMat Glove, 90.38
06.01 S1_4  | POLYETHYLENE (Mn 1400), 90.09
06.01 S1_5  | Polyethylene high density, 87.07
06.01 S1_6  | Zinc stearate, 78.72
06.01 S1_7  | POLYEHTYLENE (Mn 1800), 90.63
06.01 S1_8  | Blue Spruce, 80.48
06.01 S1_9  | Duct tape (Back side), 97.52
06.01 S1_10 | 100% Polyester, 85.20
06.01 S1_11 | ARC 1001 (cast), 92.30
06.01 S1_12 | Duct tape (Back side), 98.32
06.01 S2_1  | Lemon Basil, 94.26
06.01 S2_2  | Methyl docosanoate 98+%, 82.70
06.01 S2_3  | Polypropylene isotactic, 85.20
06.01 S2_4  | ARC 1001 (cast), 92.55
06.01 S2_5  | Wheat Gluten fluor, 84.53
06.01 S2_6  | Polypropylene isotactic, 86.48
06.01 S2_7  | Duct tape (Back side), 96.48
06.01 S3_1  | Human skin (live), 74.85
06.01 S3_2  | Plumber's Putty, 73.13
06.01 S3_3  | Red Pepper Seed, 68.40
06.01 S3_4  | 60 grit sand paper (grit side), 60.09
06.01 S3_5  | Cyanoguanidine, 39.05
# 06.07
06.07 S1_1  | Arsine, 37
06.07 S1_2  | Rubbed Sage leaf, 73
06.07 S1_3  | Castor Bean Leaf, 81.20
06.07 S1_4  | Polypropylene isotactic, 94.20
06.07 S1_5  | Polypropylene isotactic, 94.63
06.07 S1_6  | Deep Rose blush, 83.61
06.07 S1_7  | COMPLEX of Cr(III) hexathiocyanatochromite(III) potassium, 59.54
06.07 S1_8  | ALUMINA SILICATE #1, 97.10
06.07 S1_9  | Hoover 520, 85
06.07 S1_10 | Red Pepper Seed, 80.43
06.07 S1_11 | 2-Propyn-1-ol, 44.30
06.07 S1_12 | Duct tape (Back side), 98.97
06.07 S1_13 | Comet Disinfectant Cleaner, 90.87
06.07 S1_14 | ALUMINA SILICATE #1, 88.36
06.07 S1_15 | Cement, cured, 90.86
06.07 S1_16 | Thyme, 83.20
06.07 S1_17 | Deep Rose blush, 84.16
# 06.14
06.14 S1_1  | Polypropylene, isotactic, 93.61
06.14 S1_2  | Reduced Fat Bisquick, 92.86
06.14 S1_3  | Duct tape (Back side), 94.06
06.14 S1_4  | ARC 1001 (cast), 89.26
06.14 S1_5  | 2-(Trifluoromethyl)benzaldehyde chromium tricarbonyl
06.14 S1_6  | Duct tape (Back side), 98.31
06.14 S1_7  | ARC 1001 (cast), 89.88
06.14 S1_8  | Red Pepper Seed, 77.03
06.14 S1_9  | Peppermint Leaf, 86.86
06.14 S1_10 | Bee's wax, 94.87
06.14 S1_11 | Mugo Pine, 86.42
06.14 S1_12 | ARC 1001 (cast), 89.43
06.14 S1_13 | Dry Milk Powder, 65.63
06.14 S1_14 | ARC 1001 (cast), 89.83
06.14 S1_15 | Oven Cleaner, 63.04
06.14 S1_16 | ARC 1001 (cast), 90.24
# 06.21
06.21 S1_1  | Wheat Gluten fluor, 89.28
06.21 S1_3  | Phenyl isothiocyanate, 57.68
06.21 S1_4  | 1-PHENYL-2-(TRIMETHYLSILYL)ACETYLENE, 99%, 82.95
06.21 S1_5  | Fumaronitrile; (E)-1,2-Dicyanoethene, 67.64
06.21 S1_6  | ATTAGEL 50, 73.84
06.21 S1_7  | Antigorite, 83.16
06.21 S1_8  | ALUMINA SILICATE #1, 81.04
06.21 S1_9  | ALUMINA SILICATE #1, 77.38
# 06.26
06.26 S1_1   | Duct tape (Back side), 96.59
06.26 S1_2   | Duct tape (Back side), 96.91
06.26 S1_3   | Polypropylene, isotactic, average MW ca. 250,000, 77.46
06.26 S1_4   | Duct tape (Back side), 96.80
06.26 S1_5   | Duct tape (Back side), 97.18
06.26 S1_6   | CASTROL Syntilo RHS, 65.74
06.26 S1_7   | Duct tape (Back side), 95.81
06.26 S1_8   | Duct tape (Back side), 97.12
06.26 S1_9   | Duct tape (Back side), 96.88
06.26 S1_10  | Duct tape (Back side), 97.74
06.26 S1_11  | Polypropylene, isotactic, 78.46
06.26 S1_12  | Duct tape (Back side), 97.80
# 06.28
06.28 S1_1   | Blue Spruce, 71.04
06.28 S1_2   | Enfamil Lactose-free, 81.30
06.28 S1_3   | Nylon 6, 94.50
06.28 S1_4   | ARC 1001 (cast), 92.41
06.28 S1_5   | Duct tape (Back side), 90.36
06.28 S1_6   | POLYPROPYLENE #2, 80.18
06.28 S1_7   | Modified starch - quaternary ammonium hydroxyethyl starch et, 77.78
06.28 S1_8   | Starch, 79.87

"""

In [ ]:
# === OMNIC transcript parser (parse_id() comes from the loader cell in 0.2) =======


def parse_hqi(match):
    """name, hqi from 'Foo, bar, 88.70' (HQI is the trailing number; % stripped)."""
    if not match or not match.strip(): return None, np.nan
    s=match.strip()
    if ',' in s:
        name, tail = s.rsplit(',',1)
        m=re.search(r'(\d+(?:\.\d+)?)', tail)
        if m: return name.strip(), float(m.group(1))
    return s, np.nan

def parse_omnic_raw(raw):
    rows=[]
    for line in raw.splitlines():
        line=line.strip()
        if not line or line.startswith('#'): continue
        left, _, match = line.partition('|')
        toks=left.split()
        if len(toks)<2:
            print('  ! OMNIC line skipped:', line); continue
        date, sid = toks[0], toks[1]
        st, part, rep = parse_id(sid)
        name, hqi = parse_hqi(match)
        rows.append(dict(date=date, station=st, particle=part, replicate=rep,
                         omnic_name=name, hqi=hqi, sample_id=sid))
    return pd.DataFrame(rows)

print("parse_omnic_raw() ready")


OMNIC_DATE_ALIAS = {}

def build_omnic(raw, alias=None):
    """Transcript -> DataFrame. Dates are reconciled and nothing else is interpreted:
       names and HQI stay exactly as recorded."""
    d = parse_omnic_raw(raw)
    alias = OMNIC_DATE_ALIAS if alias is None else alias
    if alias:
        _n = int(d['date'].isin(alias).sum())
        d['date_logged'] = d['date']                     # keep the original for audit
        d['date'] = d['date'].map(lambda x: alias.get(x, x))
        print(f"[OMNIC date reconciliation] {alias} -> {_n} row(s) remapped "
              f"(original kept in date_logged)")
        _fired  = sorted(set(d['date_logged']) & set(alias))
        _unused = sorted(set(alias) - set(d['date_logged']))
        print(f"  · applied to: {_fired if _fired else '(none)'}")
        if _unused:
            print(f"  ! unused alias entries: {_unused} - either OMNIC_RAW has already been "
                  f"edited, or the entry is a typo.")
            print("    If the transcript itself was edited, record that edit in the Methods too,")
            print("    so that only one side of the reconciliation is not left on the record.")
    else:
        d['date_logged'] = d['date']
    return d

df_omnic = build_omnic(OMNIC_RAW)
print(f"OMNIC rows parsed: {len(df_omnic)}")

In [ ]:
# === Reconcile the spectrum inventory against the OMNIC log =================
# The merge key is (date, station, particle, replicate), so a mismatch in date vocabulary
# silently empties omnic_name and hqi for that whole date - no exception, no warning.
# It is therefore counted here in advance. July arrives in 3.1, so being empty at this
# point is expected for the held-out dates.
_KEY = ['date','station','particle','replicate']
_inv = inv.copy(); _omn = df_omnic.copy()
for _c in _KEY:
    _inv[_c] = _inv[_c].astype(str); _omn[_c] = _omn[_c].astype(str)

_heldout = {d for d in set(inv['date']) if assign_phase(d)=='heldout_july'}
_only_sp = sorted(set(inv['date']) - set(df_omnic['date']) - _heldout)
_only_om = sorted(set(df_omnic['date']) - set(inv['date']))
print("(1) date vocabulary")
print(f"   dates only in the spectra (July excluded): {_only_sp or 'none'}")
print(f"   dates only in OMNIC                      : {_only_om or 'none'}")
if _only_sp or _only_om:
    print("   these dates will fail to merge -> check OMNIC_DATE_ALIAS in 0.4 and the file names")

# Dry-run merge: how many rows would actually lose their OMNIC record
_probe = _inv.merge(_omn[_KEY+['omnic_name']], on=_KEY, how='left', indicator=True)
print("\n(2) dry-run merge: missing OMNIC by phase")
for _ph in [p for p in PHASE_ORDER if p in set(_probe['phase'])]:
    _s = _probe[_probe['phase']==_ph]; _n = int((_s['_merge']=='left_only').sum())
    _tag = "  <- filled in 3.1 (expected)" if _ph=='heldout_july' else ("  ok" if _n==0 else "  <-- check")
    print(f"   {_ph:14s} spectra {len(_s):3d} | OMNIC missing {_n:3d}{_tag}")
_lost = _probe[(_probe['_merge']=='left_only') & (_probe['phase']!='heldout_july')]
if len(_lost):
    display(_lost[['date','station','particle','replicate','phase','file']])
_orphan = _omn.merge(_inv[_KEY], on=_KEY, how='left', indicator=True)
_orphan = _orphan[_orphan['_merge']=='left_only']
if len(_orphan):
    print(f"\n   OMNIC log entries with no spectrum: {len(_orphan)} (dropped by the merge):")
    display(_orphan[['date','station','particle','omnic_name','hqi']])

### 0.5 Pure-standard verification (PE, PP): diagnostic bands, decision chain, CI zero point

In [ ]:
# === Section 0.5 - verification against pure standards (PE / PP FT-IR) ==========
# Counterpart of section 14 in Code 1. Verifies the DECISION ENGINE, not the wavenumber axis:
#   (1) diagnostic bands rise above REL_PRESENT / REL_STRONG
#   (2) features() -> classify() gives the right answer on a sample whose answer is known
#   (3) carbonyl-index zero point: CI below CI_WEATHERED for virgin material
# Keep these files OUT of SPECTRA_DIR (section 0.3 would ingest them as field particles).
# Samples: PE Sigma-Aldrich 428043 (LDPE, MI 25, CAS 9002-88-4); PP Sigma-Aldrich 427888
#   (isotactic, Mw ~250k, CAS 9003-07-0). Virgin resin: positions and heights only.
STD_FTIR_DIR   = "/content/standards_ftir"
STD_FTIR_FILES = {'polyolefin_PE': 'PE_428043.CSV', 'polyolefin_PP': 'PP_427888.CSV'}
STD_FTIR_LOT   = {'polyolefin_PE': 'Sigma-Aldrich 428043 (LDPE)',
                  'polyolefin_PP': 'Sigma-Aldrich 427888 (isotactic PP)'}

# Expected verdicts. PE can never be 'confident' by design (the Raman-required rule in
# classify()), so the expectation is stated here and a tentative PE is not read as a failure.
STD_FTIR_EXPECT = {
    'polyolefin_PE': dict(cls='polyolefin_PE', conf='tentative',
                          note='confident is disallowed by design - FT-IR cannot resolve PE from PP'),
    'polyolefin_PP': dict(cls='polyolefin_PP', conf='confident',
                          note='confident when the crystalline 1375 + 998/973/840 are clear'),
}
# Diagnostic bands reported by the check (only those classify() actually consults)
STD_FTIR_BANDS = {
    'polyolefin_PE': [('CH2920', 'CH2 asym str'), ('CH2850', 'CH2 sym str'),
                      ('b1472', 'CH2 bend'), ('b730', 'rocking doublet'),
                      ('b717', 'rocking doublet'), ('b1715', 'oxidation C=O (should be absent)')],
    'polyolefin_PP': [('CH2920', 'CH str'), ('CH2850', 'CH2 sym str'),
                      ('b1375', 'CH3 sym bend'), ('b998', 'helix'),
                      ('b973', 'helix'), ('b840', 'helix'),
                      ('b1715', 'oxidation C=O (should be absent)')],
}


def validate_ftir_standard(cls, path):
    x, y = load_csv(path)
    pp = preprocess(x, y)
    f = features(x, pp)
    got_cls, got_conf, why = classify(f)
    ci = carbonyl_index(f, got_cls)
    exp = STD_FTIR_EXPECT[cls]

    rows = []
    for key, lab in STD_FTIR_BANDS[cls]:
        v = f[key]; sn = f.get('snr_' + key, 0.0)
        rows.append(dict(band=key, label=lab, rel=round(v, 3), snr=round(sn, 0),
                         by_rel=v >= REL_PRESENT,
                         by_snr=(v >= REL_PRESENT_WEAK and sn >= SNR_PRESENT),
                         present=band_present(f, key), strong=v >= REL_STRONG))
    df_ = pd.DataFrame(rows)

    print("=" * 72)
    print(f"[{cls}]  {os.path.basename(path)}   {STD_FTIR_LOT.get(cls, '')}")
    print(df_.to_string(index=False))
    print(f"\n  -- (1) diagnostic bands (rel>={REL_PRESENT} or S/N>={SNR_PRESENT:.0f}) --")
    _t_resc = df_[(~df_.by_rel) & df_.by_snr & df_.band.isin([k for k, _ in STD_FTIR_BANDS[cls]])]
    if len(_t_resc):
        print(f"     admitted by S/N rather than rel: {list(_t_resc.band)} "
              f"(rel {list(_t_resc.rel)}, S/N {[int(v) for v in _t_resc.snr]})")
    _need = [k for k, _ in STD_FTIR_BANDS[cls] if k != 'b1715']
    _hit = int(df_[df_.band.isin(_need)].present.sum())
    print(f"     passed {_hit}/{len(_need)}"
          + ("" if _hit == len(_need) else
             f"   not passed: {list(df_[(df_.band.isin(_need)) & (~df_.present)].band)}"))
    if bool(df_.loc[df_.band == 'b1715', 'present'].any()):
        print( "     1715 detected - virgin resin should show no oxidation band. Suspect surface")
        print( "        contamination or an additive; re-measure a freshly cut face.")

    print(f"  -- (2) decision chain, end to end --")
    _hit_cls  = (got_cls == exp['cls'])
    _hit_conf = (got_conf == exp['conf'])
    print(f"     classify() -> {got_cls} / {got_conf}")
    print(f"     expected   -> {exp['cls']} / {exp['conf']}   ({exp['note']})")
    print(f"     grounds: {'; '.join(why)}")
    print(f"     {'class matches' if _hit_cls else 'CLASS MISMATCH - re-examine the rules and thresholds'}"
          f" · {'confidence matches' if _hit_conf else 'confidence differs'}")

    print(f"  -- (3) carbonyl index zero point --")
    if pd.isna(ci):
        print(f"     CI not computed (CI_POLYOLEFIN_ONLY={CI_POLYOLEFIN_ONLY}, cls={got_cls})")
    else:
        print(f"     CI = {ci:.3f}  (weathering cut CI_WEATHERED = {CI_WEATHERED})"
              + ("   virgin baseline established" if ci < CI_WEATHERED else
                 "   VIRGIN RESIN ABOVE THE WEATHERING CUT - re-examine the cut or the sample"))
    return dict(cls=cls, got_cls=got_cls, got_conf=got_conf, ci=(None if pd.isna(ci) else round(ci, 3)),
                bands_ok=_hit, bands_need=len(_need), hit=_hit_cls and _hit_conf)


_have = {c: os.path.join(STD_FTIR_DIR, fn) for c, fn in STD_FTIR_FILES.items()
         if os.path.exists(os.path.join(STD_FTIR_DIR, fn))}
if not _have:
    print(f"[standards] no files - upload them to {STD_FTIR_DIR} or edit STD_FTIR_FILES")
    print(f"   expected: {list(STD_FTIR_FILES.values())}")
    print( "   Do NOT place them in SPECTRA_DIR - section 0.3 would ingest them as field particles")
    STD_FTIR_RESULT = None
    STD_VERIFIED = False
    print("\n" + "!" * 72)
    print("Standard verification was skipped. SNR_PRESENT, REL_PRESENT_WEAK and CI_WEATHERED")
    print("are derived from these standards, so running section 2 onward without the check")
    print("leaves those values ungrounded. Upload the standards and re-run from this cell.")
    print("!" * 72)
else:
    STD_VERIFIED = True
    _res = [validate_ftir_standard(c, p) for c, p in _have.items()]
    STD_FTIR_RESULT = pd.DataFrame(_res)
    print("\n" + "=" * 72)
    print("Summary")
    print(STD_FTIR_RESULT.to_string(index=False))
    _todo = [c for c in STD_FTIR_FILES if c not in _have]
    if bool(STD_FTIR_RESULT.hit.all()):
        print(f"\n  bands and decision chain verified on the {len(_have)} standard(s) measured")
        print( "    Methods: 'the rule engine was verified against pure polymer standards")
        print( "       measured on the same ATR-FTIR under identical settings'")
    else:
        print("\n  MISMATCH - check the thresholds (REL_PRESENT/REL_STRONG) and the ATR contact first")
    if _todo:
        print(f"  not measured: {_todo}")
    # -- re-derive the SNR_PRESENT window: the standards bracket it from both sides --
    #   bands that SHOULD be there (each standard's own diagnostics), minimum S/N -> upper bound
    #   bands that should NOT be there (1715 in virgin resin), maximum S/N -> lower bound
    _t_need_snr, _t_absent_snr = [], []
    for _c2, _p2 in _have.items():
        _x2, _y2 = load_csv(_p2); _pz = preprocess(_x2, _y2); _f2 = features(_x2, _pz)
        for _k, _ in STD_FTIR_BANDS[_c2]:
            (_t_absent_snr if _k == 'b1715' else _t_need_snr).append(_f2.get('snr_' + _k, 0.0))
    if _t_need_snr and _t_absent_snr:
        _t_hi, _t_lo = min(_t_need_snr), max(_t_absent_snr)
        print("\n" + "=" * 72)
        print("SNR_PRESENT window set by the standards")
        print(f"  lower bound (false positives) : max S/N of a band that should be absent = {_t_lo:.0f}")
        print(f"  upper bound (false negatives) : min S/N of a band that should be present = {_t_hi:.0f}")
        print(f"  -> admissible {_t_lo:.0f} < SNR_PRESENT <= {_t_hi:.0f}   |   current {SNR_PRESENT:.0f}")
        if _t_lo < SNR_PRESENT <= _t_hi:
            print(f"  inside the window. Margin x{SNR_PRESENT/max(_t_lo,1e-9):.1f} against false "
                  f"positives, x{_t_hi/SNR_PRESENT:.1f} against false negatives")
        else:
            print("  OUTSIDE the window - adjust SNR_PRESENT and raise PARAM_VERSION")
        print("  The IUPAC LOQ (10) grounds the lower bound; the value itself is chosen inside "
              "this window.")

    # -- (4) band-window audit: does each window actually contain its peak? --
    #   Catches a window whose true peak lies just outside it (e.g. PE 2848 vs PP 2837).
    _t_WINS = BAND_WINDOWS
    print("\n" + "=" * 72)
    print("(4) band-window audit - maximum inside the window vs the true peak just outside it")
    _t_flag = []
    for _c2, _p2 in _have.items():
        _x2, _y2 = load_csv(_p2); _pz = preprocess(_x2, _y2)
        _cc, _rf = _pz['corr'], _pz['ref']
        for _k, _win in [(k, w) for k, w in _t_WINS.items()
                        if k in [b for b, _ in STD_FTIR_BANDS[_c2]]]:
            lo, hi = _win
            _m = (_x2 >= lo) & (_x2 <= hi)
            if not _m.any():
                continue
            _v = float(_cc[_m].max()) / _rf
            _ms = (_x2 >= lo - 15) & (_x2 <= hi + 15)
            _i2 = int(np.argmax(_cc[_ms])); _pk = float(_x2[_ms][_i2]); _pv = float(_cc[_ms][_i2]) / _rf
            if _v < REL_PRESENT and _pv >= REL_PRESENT and not (lo <= _pk <= hi):
                _t_flag.append((_c2, _k, lo, hi, _pk, _v, _pv))
    if _t_flag:
        print("  a window is missing its peak - check the literature position and correct it:")
        for _c2, _k, lo, hi, _pk, _v, _pv in _t_flag:
            print(f"     [{_c2}] {_k}: window {lo}-{hi} vs actual {_pk:.0f} cm-1 "
                  f"(delta {_pk-(lo+hi)/2:+.0f}), rel {_v:.3f} -> {_pv:.3f}")
        print("     This can also be a false positive from a neighbouring band. Confirm the "
              "literature assignment before changing anything.")
    else:
        print("  every diagnostic window contains its peak")

    print("\n  This verifies virgin resin only. It is not a verification of the weathering rule "
          "(Anshari2025) or of CI - state that in the limitations.")

### 0.6 Effect of admitting bands by S/N, over all spectra

In [ ]:
# === Section 0.6 - presence-rule sweep, including the whitelist effect ==========
# Against the baseline of rel alone, two routes are compared:
#   'all bands' - relax every band. PP is rescued, but SAN and PVDF rise with it.
#   'whitelist' - relax only SNR_RELAXABLE. In force.
# The criterion is a combination that rescues PP WITHOUT adding implausible classes.

IMPLAUSIBLE = ['SAN', 'PVDF', 'nylon_PA']   # industrial polymers rare in surface seawater

if '_feat_cache' not in dir() or not _feat_cache:
    print("run section 0.3 first (_feat_cache is empty)")
else:
    from collections import Counter
    _t_sv = (USE_SNR_PRESENCE, REL_PRESENT_WEAK, SNR_PRESENT, set(SNR_RELAXABLE))
    _t_ALL = set(BAND_WINDOWS)

    def _t_run(weak, snr, wl, use=True):
        globals().update(USE_SNR_PRESENCE=use, REL_PRESENT_WEAK=weak,
                         SNR_PRESENT=snr, SNR_RELAXABLE=wl)
        return [classify(f)[0] for _, _, f in _feat_cache.values()]

    _t_base = _t_run(0, 0, set(), use=False)
    _t_bc = Counter(_t_base); _t_imp0 = sum(_t_bc[c] for c in IMPLAUSIBLE)
    print(f"baseline (rel alone, n={len(_t_base)}):")
    print("   " + " · ".join(f"{k} {v}" for k, v in _t_bc.most_common(6)))
    print(f"   PP={_t_bc['polyolefin_PP']} · implausible {IMPLAUSIBLE} total={_t_imp0}")

    _t_rows = []
    for _t_lab, _t_wl in [('all bands', _t_ALL), ('whitelist', _t_sv[3])]:
        for _t_w in [0.15, 0.20, 0.25]:
            for _t_s in [20, 30, 50, 74]:
                _t_lbl = _t_run(_t_w, _t_s, _t_wl)
                _c = Counter(_t_lbl)
                _t_rows.append(dict(route=_t_lab, WEAK=_t_w, SNR=_t_s,
                                  changed=sum(a != b for a, b in zip(_t_base, _t_lbl)),
                                  PP=_c['polyolefin_PP'], PP_delta=_c['polyolefin_PP'] - _t_bc['polyolefin_PP'],
                                  PE=_c['polyolefin_PE'], silicate=_c['silicate'],
                                  implausible=sum(_c[c] for c in IMPLAUSIBLE),
                                  implausible_delta=sum(_c[c] for c in IMPLAUSIBLE) - _t_imp0))
    globals().update(USE_SNR_PRESENCE=_t_sv[0], REL_PRESENT_WEAK=_t_sv[1],
                     SNR_PRESENT=_t_sv[2], SNR_RELAXABLE=_t_sv[3])
    SWEEP = pd.DataFrame(_t_rows)
    print("\ngrid sweep (against the baseline):")
    for _t_lab in ['all bands', 'whitelist']:
        print(f"\n  [{_t_lab}]")
        print(SWEEP[SWEEP.route == _t_lab].drop(columns='route').to_string(index=False))

    print("\n" + "=" * 72)
    print("Reading: does it rescue PP without adding implausible classes?")
    _t_ok = SWEEP[(SWEEP.implausible_delta <= 0) & (SWEEP.PP_delta > 0)]
    if len(_t_ok):
        print(_t_ok.sort_values('PP_delta', ascending=False).to_string(index=False))
        _t_b = _t_ok.sort_values('PP_delta', ascending=False).iloc[0]
        print(f"\n  best: {_t_b.route} · WEAK={_t_b.WEAK} · SNR={_t_b.SNR:.0f}"
              f"  -> PP +{int(_t_b.PP_delta)} · implausible {int(_t_b.implausible_delta):+d}"
              f" · {int(_t_b.changed)} assignments changed in total")
    else:
        print("  no combination rescues PP without also adding implausible classes.")
        print("     Narrow SNR_RELAXABLE further (drop 973), or re-examine the S/N route itself.")

    _t_cur = SWEEP[(SWEEP.route == 'whitelist') & (SWEEP.WEAK == REL_PRESENT_WEAK)
                 & (SWEEP.SNR == SNR_PRESENT)]
    if len(_t_cur):
        _t_r = _t_cur.iloc[0]
        print(f"\n  current setting (whitelist {sorted(SNR_RELAXABLE)} · WEAK={REL_PRESENT_WEAK} · "
              f"SNR={SNR_PRESENT:.0f}): PP {int(_t_r.PP_delta):+d} · "
              f"implausible {int(_t_r.implausible_delta):+d}")
    if not globals().get('STD_VERIFIED', False):
        print("  section 0.5 has not verified the standards - upload them before fixing any value")
    SWEEP.to_csv(os.path.join(OUT_DIR, f'S0_6_presence_sweep_{PARAM_VERSION}.csv'), index=False)
    print(f"\n  [saved] {OUT_DIR}/S0_6_presence_sweep_{PARAM_VERSION}.csv")

### 0.7 Sensitivity of the grade-C thresholds: REL_STRONG, QC_MIN_TOTAL_A, SIO_DOMINATES

In [ ]:
# === Section 0.7 - sensitivity of the grade-C thresholds ======================
# Measures how far the values graded C in THRESHOLD_PROVENANCE (no external grounding) move
# the conclusion. Where grounding cannot be supplied, the least that can be shown is that
# the conclusion is INSENSITIVE to the value.
#
#   REL_STRONG    confidence tier only -> 0 class changes proves it is a reporting threshold
#   REL_DOMINANT  takes part in class decisions (mineral dominance, silicate, veto) -> the
#                 one that actually matters
#   QC_MIN_TOTAL_A / SIO_DOMINATES
#   SG_WIN and SG_ORD act in preprocessing and are handled separately (note at the end)

if 'to_reporting' not in dir():
    print("run the reporting-label cell first")
elif '_feat_cache' not in dir() or not _feat_cache:
    print("run section 0.3 first")
else:
    from collections import Counter
    _t_F = [f for _, _, f in _feat_cache.values()]
    _b_cls = [classify(f)[0] for f in _t_F]
    _b_cnf = [classify(f)[1] for f in _t_F]

    _b_rep = [to_reporting(c) for c in _b_cls]     # reporting label
    _b_tier = [to_tier(c) for c in _b_cls]         # top tier

    def _sweep(name, values):
        """Count changes at three levels: detailed class, reporting label, tier.
           If the detailed class moves but the reporting label does not, that threshold
           does not propagate to the conclusion."""
        _sv = globals()[name]; out = []
        for v in values:
            globals()[name] = v
            cl = [classify(f)[0] for f in _t_F]; cf = [classify(f)[1] for f in _t_F]
            rp = [to_reporting(c) for c in cl];  tr = [to_tier(c) for c in cl]
            out.append({name: v,
                        'detail_changed': sum(a != b for a, b in zip(_b_cls, cl)),
                        'reporting_changed': sum(a != b for a, b in zip(_b_rep, rp)),
                        'tier_changed': sum(a != b for a, b in zip(_b_tier, tr)),
                        'conf_changed': sum(a != b for a, b in zip(_b_cnf, cf)),
                        'polymer': Counter(tr)['polymer'],
                        'PP': Counter(cl)['polyolefin_PP']})
        globals()[name] = _sv
        return pd.DataFrame(out)

    _res = {}
    for _nm, _vals, _note in [
        ('REL_STRONG',     [0.40, 0.50, 0.60, 0.70, 0.80], 'confidence tier only'),
        ('REL_DOMINANT',   [0.40, 0.50, 0.60, 0.70, 0.80], 'takes part in class decisions'),
        ('QC_MIN_TOTAL_A', [0.20, 0.35, 0.50, 0.75, 1.00], 'threshold at which assignment begins'),
        ('SIO_DOMINATES',  [1.00, 1.15, 1.30, 1.50, 2.00], 'mineral dominance ratio'),
        ('CI_WEATHERED',   [0.15, 0.20, 0.30, 0.40, 0.50],
         'separates PE from polyolefin_unresolved; the distribution is examined in 0.8'),
    ]:
        _d = _sweep(_nm, _vals); _res[_nm] = _d
        print("=" * 72)
        print(f"{_nm} - {_note}  (current {globals()[_nm]})")
        print(_d.to_string(index=False))
        _t_detail = int(_d['detail_changed'].max()); _t_rep = int(_d['reporting_changed'].max())
        _t_tier = int(_d['tier_changed'].max()); _t_n = len(_t_F)
        if _t_rep == 0 and _t_tier == 0:
            if _t_detail == 0:
                print("   detailed class unchanged too - the threshold never fires on this data")
            else:
                print(f"   detailed class moves by up to {_t_detail}, but the reporting label by 0")
                print("     movement is confined to non-polymers; shown not to propagate")
        elif _t_rep <= 2:
            print(f"   reporting label moves by at most {_t_rep} ({_t_rep/_t_n*100:.1f}%) - "
                  f"effectively insensitive. Give the figure in the limitations.")
        else:
            print(f"   reporting label {_t_rep} ({_t_rep/_t_n*100:.1f}%), tier {_t_tier} changed")
            print("     either supply grounding or state this as a limitation")
        print()

    for _nm, _d in _res.items():
        _d.to_csv(os.path.join(OUT_DIR, f'S0_7_sens_{_nm}_{PARAM_VERSION}.csv'), index=False)
    print(f"[saved] {OUT_DIR}/S0_7_sens_*.csv")

    # -- update the impact column of the provenance ledger with the measured figures --
    #    The wording must stay inside the vocabulary defined in CONFIG (IMPACT_UNMEASURED,
    #    IMPACT_INERT_RE); threshold_report() classifies by matching it.
    for _nm, _d in _res.items():
        if _nm not in THRESHOLD_PROVENANCE:
            continue
        _r_max = int(_d['reporting_changed'].max()); _c_max = int(_d['detail_changed'].max())
        _rng = f"{_d[_nm].min():g}-{_d[_nm].max():g}"
        THRESHOLD_PROVENANCE[_nm]['impact'] = (
            f"0.7 sweep over {_rng}: reporting label changed {_r_max}/{_t_n}"
            + (f" (detailed {_c_max})" if _c_max != _r_max else ""))
    threshold_report()

    print("\n" + "=" * 72)
    print("SG_WIN / SG_ORD act in preprocessing, so _feat_cache must be rebuilt to test them")
    print("  Put the following before section 0.3, run it twice with different values, and")
    print("  compare the class distributions:")
    print("    SG_WIN, SG_ORD = 11, 3     # or 15,3 (current) / 21,3 / 15,2")
    print("  Derivative smoothing affects band width, so after changing either value confirm")
    print("  that the standards in 0.5 still pass.")

### 0.8 Carbonyl index distribution: is CI_WEATHERED grounded?

In [ ]:
# === Section 0.8 - carbonyl index distribution (grounding for CI_WEATHERED) ======
# CI = area(1690-1760) / area(1420-1490)
#        oxidation C=O      CH2/CH3 bending, unrelated to oxidation and proportional to the
#                           amount of skeleton
#   Dividing by the skeleton makes the ratio insensitive to ATR contact and thickness, so it
#   reads as "oxidation relative to skeleton". The windows are narrower than the original
#   definition of Almond2020 (1850-1650 / 1500-1420), so this is a MODIFIED CI and the
#   published values do not carry over.
#
# CI_WEATHERED decides a class (PE vs polyolefin_unresolved, in classify()), yet the cut of
# 0.30 itself is ungrounded: the standards gave only the ZERO POINT. This cell asks whether
# grounding can be constructed at all:
#   (1) where the standard zero point falls within the field distribution
#   (2) if the distribution is bimodal, the valley is a principled cut
#   (3) if it is not, the honest course is to drop the binary cut and report CI as continuous
#   (4) how many particles sit within +/-20% of the current cut = how many the cut can flip

if '_feat_cache' not in dir() or not _feat_cache:
    print("run section 0.3 first (_feat_cache is empty)")
else:
    import numpy as _np2

    _t_rows = []
    for _t_p, (_t_x, _t_pz, _t_f) in _feat_cache.items():
        _t_cls = classify(_t_f)[0]
        _t_ci = carbonyl_index(_t_f, _t_cls)
        _t_rows.append(dict(**_meta[_t_p], cls=_t_cls, ci=_t_ci))
    CI_DF = pd.DataFrame(_t_rows)
    CI_DF['phase'] = [assign_phase(d, s) for d, s in zip(CI_DF['date'], CI_DF['station'])]
    _t_po = CI_DF[CI_DF.ci.notna()].copy()          # CI_POLYOLEFIN_ONLY: polyolefins only

    print("=" * 72)
    print(f"CI computed for {len(_t_po)}/{len(CI_DF)} particles "
          f"(polyolefins only - CI_POLYOLEFIN_ONLY={CI_POLYOLEFIN_ONLY})")
    if len(_t_po) < 10:
        print("  too few particles to read the distribution")
    else:
        _t_q = _t_po.ci.quantile([0, .05, .25, .5, .75, .95, 1]).round(3)
        print(f"  quantiles:  min {_t_q[0]} · P5 {_t_q[.05]} · P25 {_t_q[.25]} · "
              f"median {_t_q[.5]} · P75 {_t_q[.75]} · P95 {_t_q[.95]} · max {_t_q[1]}")

        # -- (1) position relative to the standard zero point --
        STD_CI_ZERO = {'PE 428043': 0.043, 'PP 427888': 0.076}
        _t_z = max(STD_CI_ZERO.values())
        print(f"\n(1) standard zero point (virgin resin): {STD_CI_ZERO}")
        print(f"    field particles at or below the highest zero point {_t_z:.3f}: "
              f"{int((_t_po.ci <= _t_z).sum())}/{len(_t_po)} "
              f"({(_t_po.ci <= _t_z).mean()*100:.0f}%) - effectively unweathered")
        print(f"    the current cut {CI_WEATHERED} is {CI_WEATHERED/_t_z:.1f}x the zero point")

        # -- (2) bimodality: look for a valley in the histogram --
        _t_h, _t_e = _np2.histogram(_t_po.ci, bins=20)
        _t_ctr = (_t_e[:-1] + _t_e[1:]) / 2
        print("\n(2) distribution (20 bins)")
        _t_mx = max(_t_h.max(), 1)
        for _t_i in range(len(_t_h)):
            _t_mk = ' <- current cut' if _t_e[_t_i] <= CI_WEATHERED < _t_e[_t_i+1] else ''
            print(f"   {_t_ctr[_t_i]:6.3f} | {'#'*int(40*_t_h[_t_i]/_t_mx)}{_t_h[_t_i]:>4}{_t_mk}")
        # first local minimum after the tallest peak
        _t_pk = int(_np2.argmax(_t_h))
        _t_valley = None
        for _t_i in range(_t_pk + 1, len(_t_h) - 1):
            if _t_h[_t_i] <= _t_h[_t_i-1] and _t_h[_t_i] <= _t_h[_t_i+1] and _t_h[_t_i] < _t_h.max() * 0.25:
                if _t_h[_t_i+1:].sum() >= max(3, 0.05 * len(_t_po)):   # a real peak must follow
                    _t_valley = _t_ctr[_t_i]; break
        if _t_valley is not None:
            print(f"\n   bimodal - valley at about {_t_valley:.3f}")
            print(f"     a principled cut would sit here; it differs from the current "
                  f"{CI_WEATHERED} by {abs(_t_valley-CI_WEATHERED):.3f}")
            print(f"     right-hand peak (weathered group): {int((_t_po.ci > _t_valley).sum())}")
        else:
            print("\n   unimodal - there is no valley.")
            print("     No principled basis for a binary cut can be constructed. Either")
            print("        (a) report CI as a CONTINUOUS variable and drop the binary decision, or")
            print("        (b) keep the cut and state in the limitations that it is arbitrary and")
            print("            that the conclusion is insensitive to it (section 0.7).")

        # -- (3) particles the cut can flip --
        _t_lo, _t_hi = CI_WEATHERED * 0.8, CI_WEATHERED * 1.2
        _t_edge = _t_po[(_t_po.ci >= _t_lo) & (_t_po.ci <= _t_hi)]
        print(f"\n(3) particles within +/-20% of the cut ({_t_lo:.2f}-{_t_hi:.2f}): "
              f"{len(_t_edge)} / {len(_t_po)}")
        if len(_t_edge):
            print("    a small move in the cut flips these between PE and polyolefin_unresolved:")
            display(_t_edge[['date', 'station', 'particle', 'cls', 'ci', 'phase']]
                    .sort_values('ci').round(3))

        # -- (4) CI by class --
        print("\n(4) CI by class")
        display(_t_po.groupby('cls')['ci'].agg(['count', 'median', 'mean', 'min', 'max']).round(3))

        # -- (5) update the provenance ledger --
        _t_note = (f"0.8 distribution: zero point {_t_z:.3f}, median {_t_q[.5]:.3f}, "
                   f"{len(_t_edge)} particle(s) within +/-20% of the cut"
                   + (f", bimodal valley at {_t_valley:.3f}" if _t_valley is not None
                      else ", unimodal (no principled cut)"))
        if 'CI_WEATHERED' in THRESHOLD_PROVENANCE:
            THRESHOLD_PROVENANCE['CI_WEATHERED']['src'] += f" | {_t_note}"
            if _t_valley is not None and abs(_t_valley - CI_WEATHERED) < 0.05:
                THRESHOLD_PROVENANCE['CI_WEATHERED']['grade'] = 'B'
                print(f"\n    the valley agrees with the current cut to within "
                      f"{abs(_t_valley-CI_WEATHERED):.3f} - CI_WEATHERED promoted to "
                      f"grade B (derived from the distribution)")
        CI_DF.to_csv(os.path.join(OUT_DIR, f'S0_8_carbonyl_index_{PARAM_VERSION}.csv'), index=False)
        print(f"\n    [saved] {OUT_DIR}/S0_8_carbonyl_index_{PARAM_VERSION}.csv")

### 0.9 CI_GATE, two modes: how far the carbonyl index may decide a class

In [ ]:
# === Section 0.9 - CI_GATE: comparing the two modes ==========================
# Question: if a complete diagnostic fingerprint is not withdrawn on the strength of the
# carbonyl index, how does the outcome change?
#   'always'          a high CI gives polyolefin_unresolved even when pe_doublet is complete
#   'incomplete_only' in force: a complete doublet settles PE; CI withholds only on a
#                     partial fingerprint
#
# Two things are measured:
#   (1) how many assignments differ between the modes, and in which direction
#   (2) how sensitive each mode is to CI_WEATHERED
# The second is the point. CI_WEATHERED is grade C and section 0.8 found no valley in the
# distribution, so no principled cut exists. If narrowing CI's authority makes the reporting
# labels insensitive to the cut, an ungrounded value has been shown not to propagate to the
# conclusion - the same argument that makes REL_STRONG a reporting threshold.

if '_feat_cache' not in dir() or not _feat_cache:
    print("run section 0.3 first (_feat_cache is empty)")
elif 'CI_GATE' not in dir():
    print("run the CONFIG cell first (CI_GATE)")
else:
    from collections import Counter
    _t_F9 = [f for _, _, f in _feat_cache.values()]
    _t_keys = list(_feat_cache.keys())
    _t_save_gate, _t_save_ci = CI_GATE, CI_WEATHERED

    def _t_run9(gate, ci_cut):
        globals().update(CI_GATE=gate, CI_WEATHERED=ci_cut)
        return [classify(f)[0] for f in _t_F9]

    # -- (1) the two modes at the current cut --------------------------------
    _t_old = _t_run9('always', _t_save_ci)
    _t_new = _t_run9('incomplete_only', _t_save_ci)
    _t_chg = [(i, a, b) for i, (a, b) in enumerate(zip(_t_old, _t_new)) if a != b]

    print("=" * 72)
    print(f"(1) mode comparison (CI_WEATHERED = {_t_save_ci})   n={len(_t_F9)}")
    print(f"    assignments that differ: {len(_t_chg)}")
    if _t_chg:
        print("\n    direction of change:")
        for (a, b), n in Counter((a, b) for _, a, b in _t_chg).most_common():
            print(f"      {a}  ->  {b}   {n}")
        _t_tbl = pd.DataFrame([dict(**_meta[_t_keys[i]], always=a, incomplete_only=b)
                               for i, a, b in _t_chg])
        _t_tbl['phase'] = [assign_phase(d, s) for d, s in zip(_t_tbl['date'], _t_tbl['station'])]
        display(_t_tbl[['date', 'station', 'particle', 'always', 'incomplete_only', 'phase']])

    # -- (2) sensitivity to CI_WEATHERED, per mode ---------------------------
    print("\n" + "=" * 72)
    print("(2) sensitivity to CI_WEATHERED, by mode")
    _t_grid = [0.15, 0.20, 0.30, 0.40, 0.50]
    _t_sens = []
    for _t_g, _t_base in [('always', _t_old), ('incomplete_only', _t_new)]:
        _t_brep = [to_reporting(c) for c in _t_base]
        for _t_v in _t_grid:
            _t_r = _t_run9(_t_g, _t_v)
            _t_sens.append(dict(mode=_t_g, CI_WEATHERED=_t_v,
                                detail_changed=sum(a != b for a, b in zip(_t_base, _t_r)),
                                reporting_changed=sum(a != b for a, b in zip(
                                    _t_brep, [to_reporting(c) for c in _t_r])),
                                PE=Counter(_t_r)['polyolefin_PE'],
                                unres=Counter(_t_r)['polyolefin_unresolved']))
    globals().update(CI_GATE=_t_save_gate, CI_WEATHERED=_t_save_ci)
    SENS_CI = pd.DataFrame(_t_sens)
    for _t_g in ['always', 'incomplete_only']:
        _t_sub = SENS_CI[SENS_CI['mode'] == _t_g]
        _t_mx = int(_t_sub['reporting_changed'].max())
        print(f"\n   [{_t_g}]  greatest change in reporting labels {_t_mx}/{len(_t_F9)} "
              f"({_t_mx/len(_t_F9)*100:.1f}%)")
        print(_t_sub.drop(columns='mode').to_string(index=False))

    _t_a = int(SENS_CI[SENS_CI['mode'] == 'always']['reporting_changed'].max())
    _t_b = int(SENS_CI[SENS_CI['mode'] == 'incomplete_only']['reporting_changed'].max())
    print("\n" + "=" * 72)
    print(f"Reading:  always {_t_a}  ->  incomplete_only {_t_b}")
    if _t_b == 0:
        print("   CI_WEATHERED has withdrawn from the conclusion entirely: whatever the cut,")
        print("   the reporting labels do not move. An ungrounded threshold has been shown not")
        print("   to propagate, and CI can be reported as a descriptive quantity only.")
    elif _t_b < _t_a:
        print(f"   sensitivity reduced from {_t_a} to {_t_b} "
              f"({(1-_t_b/max(_t_a,1))*100:.0f}% lower).")
        print("   What remains are particles with a PARTIAL doublet, which is exactly the case")
        print("   in which CI should decide to withhold.")
    else:
        print("   sensitivity did not fall - CI_GATE had no effect here. Re-examine why.")
    if 'CI_WEATHERED' in THRESHOLD_PROVENANCE:
        # Wording must stay inside the CONFIG vocabulary (IMPACT_UNMEASURED / IMPACT_INERT_RE);
        # writing "0/230" rather than "0 / 230" is what lets threshold_report() see a null result.
        THRESHOLD_PROVENANCE['CI_WEATHERED']['impact'] = (
            f"0.9 sweep over {_t_grid[0]}-{_t_grid[-1]}: reporting label changed "
            f"{_t_b}/{len(_t_F9)} under incomplete_only "
            f"({_t_a}/{len(_t_F9)} under always)")
        threshold_report()
    SENS_CI.to_csv(os.path.join(OUT_DIR, f'S0_9_ci_gate_{PARAM_VERSION}.csv'), index=False)
    print(f"\n   [saved] {OUT_DIR}/S0_9_ci_gate_{PARAM_VERSION}.csv")
    print("   Changing CI_GATE means raising PARAM_VERSION and re-running sections 2 to 5.")

### 0.10 Secondary thresholds: SG parameters (structurally inert); non-polymers cannot be separated

In [ ]:
# === Section 0.10 - closing out the secondary thresholds =====================
# The subject of this study is a catalogue of misidentification among weathered field
# particles. The two items below are incidental to that subject, so each is shown to be
# STRUCTURALLY unable to affect the conclusion and is then carried into the limitations.
#   (1) SG_WIN / SG_ORD  - proved not to be consumed by the assignment; no sweep needed
#   (2) natural_organic  - the Si-O window overlaps cellulose C-O, so the two cannot be
#                          separated. The reporting tier already groups them as non_polymer,
#                          so the conclusion is unchanged; only the carbonate hint is checked.

print("=" * 72)
print("(1) SG_WIN / SG_ORD - direct test that they cannot affect an assignment")
#   Reading the source with getsource() is fragile across environments, so instead the values
#   are actually changed and the features re-measured. That is the stronger evidence: the
#   result is measured rather than the code inspected.
_t_save_sg = (SG_WIN, SG_ORD)
if '_feat_cache' in dir() and _feat_cache:
    _t_paths = list(_feat_cache.keys())[:min(20, len(_feat_cache))]
else:
    _t_paths = []
if not _t_paths:
    print("   run section 0.3 first (_feat_cache is empty)")
else:
    def _t_feat_all(win, order):
        globals().update(SG_WIN=win, SG_ORD=order)
        out = []
        for _t_q in _t_paths:
            _t_xx, _t_yy = load_csv(_t_q)
            out.append(features(_t_xx, preprocess(_t_xx, _t_yy)))
        return out
    _t_ref_f = _t_feat_all(*_t_save_sg)
    _t_diff_total, _t_cls_diff = 0, 0
    _t_tab = []
    for _t_w, _t_o in [(9, 3), (11, 3), (21, 3), (15, 2), (25, 4)]:
        _t_alt = _t_feat_all(_t_w, _t_o)
        _t_nd = sum(1 for a, b in zip(_t_ref_f, _t_alt)
                    for k in BAND_WINDOWS if abs(a[k] - b[k]) > 1e-12)
        _t_cd = sum(1 for a, b in zip(_t_ref_f, _t_alt) if classify(a)[0] != classify(b)[0])
        _t_tab.append(dict(SG_WIN=_t_w, SG_ORD=_t_o, band_values_differing=_t_nd,
                           class_changed=_t_cd))
        _t_diff_total += _t_nd; _t_cls_diff += _t_cd
    globals().update(SG_WIN=_t_save_sg[0], SG_ORD=_t_save_sg[1])
    print(f"   {len(_t_paths)} spectra x {len(BAND_WINDOWS)} bands, against the current {_t_save_sg}")
    display(pd.DataFrame(_t_tab))
    if _t_diff_total == 0 and _t_cls_diff == 0:
        print("""
   Not one band value changes.
     SG_WIN and SG_ORD only produce pp['d2'] inside preprocess(), and features() consumes
     only pp['corr'] and pp['ref']. The Savitzky-Golay filter is therefore absent from the
     decision path: this is not an insensitive threshold but a structurally inert one.
     (The savgol_filter inside detect_fringe uses a hard-coded 51/2 and is unrelated.)
   [reporting note] SG_WIN / SG_ORD: computed for inspection only; not consumed by
     features(); cannot affect any assignment.""")
        if 'SG_WIN/SG_ORD' in THRESHOLD_PROVENANCE:
            THRESHOLD_PROVENANCE['SG_WIN/SG_ORD']['impact'] = \
                'structurally 0 - no band value differs (measured directly in 0.10)'
    else:
        print(f"   {_t_diff_total} band value(s) differ, {_t_cls_diff} class change(s) "
              f"-> a sensitivity report is required")

print("\n" + "=" * 72)
print("(2) natural_organic and the carbonate hint - fixing the limit of separation")
print("""   The windows overlap, for physical reasons:
     SiO      1000-1090  vs  cellulose C-O        1030-1050   -> silicate and organic
                                                                 cannot be separated
     carb1420 1400-1450  vs  cellulose CH2 bend   1420-1430
     carb875   870-880   vs  cellulose beta-glycosidic  897
   The bands that would separate them (450-480 Si-O-Si, 890-905, lignin 1590-1620) are not
   in this feature set. The cellulose O-H band is also 300-400 cm-1 wide and is absorbed
   into the baseline by ALS (lam=1e5), so P('broadOH') in the natural_organic rule is not
   satisfied and the rule is effectively unreachable.
   The reporting tier nevertheless groups these as non_polymer, so the primary metric
   (polymers only) is unaffected.""")

if '_feat_cache' not in dir() or not _feat_cache:
    print("\n   run section 0.3 first to obtain the counts below")
else:
    from collections import Counter
    _t_rows = []
    for _t_p, (_t_x, _t_z, _t_f) in _feat_cache.items():
        _t_c = classify(_t_f)[0]
        _t_rows.append(dict(cls=_t_c, tier=to_tier(_t_c), rep=to_reporting(_t_c),
                            carb712=_t_f['carb712'], cellCO=_t_f['cellCO'],
                            CH2920=_t_f['CH2920']))
    _T = pd.DataFrame(_t_rows)
    print(f"\n   tiers over all {len(_T)} particles: {Counter(_T.tier)}")
    print(f"   assigned natural_organic: {int((_T.cls=='natural_organic').sum())} "
          f"{'(confirms the rule is unreachable)' if (_T.cls=='natural_organic').sum()==0 else ''}")

    # Carbonate hint: nu4 at 712 is specific to calcite and absent in cellulose
    _T_carb = _T[_T.cls.str.startswith('carbonate')]
    if len(_T_carb):
        _t_unconf = _T_carb[~(_T_carb.carb712 >= REL_PRESENT)]
        print(f"\n   of {len(_T_carb)} carbonate assignments, nu4 (712 cm-1) is unconfirmed in "
              f"{len(_t_unconf)} ({len(_t_unconf)/len(_T_carb)*100:.0f}%)")
        print("     nu4 is specific to calcite and absent in cellulose, so the unconfirmed ones "
              "may be plant material")
        _t_plantish = _T_carb[(_T_carb.cellCO > _T_carb.CH2920) & (_T_carb.carb712 < REL_PRESENT)]
        print(f"     of those, C-O exceeds the CH stretch (the plant pattern) in {len(_t_plantish)}")
        print("     Report these figures as they stand in the limitations; the rule is not "
              "changed (the primary metric is unaffected and this lies outside the subject).")
    else:
        print("\n   no carbonate assignments")

    print("""
   [reporting note] non-polymers are one reported category: the Si-O window (1000-1090)
     overlaps the cellulose C-O stretch (1030-1050) and the ALS baseline removes the broad
     O-H band, so mineral and cellulosic material are not separable with this band set;
     non-polymer assignments are excluded from the primary metric.""")

if 'threshold_report' in dir():
    threshold_report()

---
# Part 1. Sampling design: the case for continuing with S1 alone (<= 06.01, S1+S2+S3)

One question: does S1 already contain the functional-group diversity seen at this site over this period? Two checks (1.2): class lists compared across stations (did S2 or S3 contribute a class absent from S1?) and the S1 class accumulation curve. This concerns the set of classes only; differences in composition between stations are not addressed (limitation).

In [ ]:
# === Section 1.1 - Phase-1 class spec =====================================
p1_paths=[p for p in _feat_cache if assign_phase(_meta[p]['date'])=='phase1']
print(f"Phase-1 (<= {PHASE1_END}, all stations) spectra: {len(p1_paths)}")

df1 = run_pipeline(p1_paths, tag="S1.1 phase1")

print("\n--- Phase-1 class spec (spec_class distribution; chemical classes vs states) ---")
valid1_all = df1[df1.spec_class!='ERROR']
valid1     = valid1_all[valid1_all.is_chemical]          # chemical classes only
rows=[]
for cls,k in valid1['spec_class'].value_counts().items():
    p,lo,hi = wilson_ci(k, len(valid1))
    rows.append(dict(spec_class=cls, n=k, prop=round(p,3),
                     ci_lo=round(lo,3), ci_hi=round(hi,3)))
spec1=pd.DataFrame(rows); display(spec1)
print(f"Phase-1 chemical classes: {valid1['spec_class'].nunique()} "
      f"(proportions are over the {len(valid1)} chemical rows)")
print("measurement / QC states (not classes, reported separately):",
      valid1_all[~valid1_all.is_chemical]['spec_class'].value_counts().to_dict())

print("\n--- date x station x class inventory ---")
display(pd.crosstab([df1['date'],df1['station']], df1['spec_class'], margins=True))
df1.to_csv(os.path.join(OUT_DIR,'S1_phase1_particles.csv'), index=False)

In [ ]:
# === Section 1.2 - does S1 already contain this period's functional groups? =====
valid1 = df1[(df1.spec_class!='ERROR') & (df1.is_chemical)].copy()

print("="*72)
print("(1) class lists compared across stations - do S2/S3 add a group absent from S1?")
set_s1  = set(valid1[valid1.station=='S1']['spec_class'])
set_oth = set(valid1[valid1.station!='S1']['spec_class'])
only_oth = sorted(map(str, set_oth - set_s1)); only_s1 = sorted(map(str, set_s1 - set_oth))
print(f"   S1 {len(set_s1)} classes | S2+S3 {len(set_oth)} | union {len(set_s1|set_oth)}")
print(f"   found only at S2/S3: {only_oth if only_oth else '(none)'}")
print(f"   found only at S1   : {only_s1 if only_s1 else '(none)'}")

# station x class presence - candidate SI table
pres = (pd.crosstab(valid1['spec_class'], valid1['station']) > 0)
pres['in_S1'] = pres.get('S1', False)
display(pres)

# station of first observation: did S1 see any class only belatedly?
first = (valid1.sort_values(['date','station','particle'])
              .groupby('spec_class').first()[['date','station']]
              .rename(columns={'date':'first_date','station':'first_station'}))
first['n_total'] = valid1['spec_class'].value_counts()
n_first_oth = int((first.first_station!='S1').sum())
print(f"\n   classes first observed at S2/S3: {n_first_oth}")
if n_first_oth:
    display(first[first.first_station!='S1'])

print("\n" + "="*72)
print("(2) class accumulation curves - per station vs pooled (figure)")
fig, ax = plt.subplots(figsize=(7.5,4.8))
res_by_st = {}
for st, c in [('S1','black'), ('S2','tab:blue'), ('S3','tab:orange')]:
    sub = df1[(df1.station==st) & (df1.spec_class!='ERROR')]['spec_class'].tolist()
    if len(sub) >= 2:
        res_by_st[st] = class_accumulation(sub, f"Phase1 {st}", ax=ax, color=c)
res_p1 = class_accumulation(df1[df1.spec_class!='ERROR']['spec_class'].tolist(),
                            "Phase1 ALL (S1+S2+S3)", ax=ax, color='tab:red')
ax.set_title('Phase-1 chemical-class accumulation: per-station vs pooled')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'S1_accumulation_by_station.png'), dpi=130); plt.show()
display(pd.DataFrame([r for r in list(res_by_st.values())+[res_p1] if r]))
print("   tail_new = classes first appearing in the last 25%. Descriptive only; it is not a "
      "pass mark.")
print("   Comparing end points across curves is confounded by the differing n per station - "
      "read the class lists in (1) instead.")

print("\n" + "="*72)
print("(3) reporting note")
print(f"   stations sampled up to {PHASE1_END}; S1 classes = {len(set_s1)}; "
      f"classes seen only at S2/S3 = {sorted(only_oth) if only_oth else 'none'}")
if only_oth:
    print(f"   {len(only_oth)} class(es) rest on S2/S3 only: the S1-only design covers the "
          "abundant class list, not an exhaustive inventory of rare classes.")
print("   Limitation: this argument concerns the SET of classes only, not composition.")
print("   It does not exclude differences in composition between stations, nor the")
print("   non-detection of a rare class where sampling was shallow.")
valid1.to_csv(os.path.join(OUT_DIR,'S1_phase1_chemical.csv'), index=False)

---
# Part 2. Building the in-house library and freezing the vocabulary (<= 06.28, S1)

The `spec_class` library is assembled from the training particles, and the vocabulary, thresholds and preprocessing parameters are written to a JSON snapshot that Part 3 reads back.

In [ ]:
# === Section 2.1 - assembling the training set =============================
train_paths=[p for p in _feat_cache if assign_phase(_meta[p]['date']) in ('phase1','phase2')]

# Check that the data match the narrative: are there non-S1 particles inside phase 2?
p2 = [p for p in train_paths if assign_phase(_meta[p]['date'])=='phase2']
p2_non_s1 = [p for p in p2 if _meta[p]['station']!='S1']
if p2_non_s1:
    tab = pd.Series([f"{_meta[p]['date']} {_meta[p]['station']}" for p in p2_non_s1]).value_counts()
    print("phase 2 (05.18-06.28) contains particles from stations other than S1, which")
    print("contradicts the statement that sampling was S1-only from June:")
    print(tab.to_string())
    if PHASE2_S1_ONLY:
        print(f"  PHASE2_S1_ONLY=True: {len(p2_non_s1)} particle(s) excluded from the training set.")
        print("  [reporting note] the exclusion of these non-S1 phase-2 particles must be stated;")
        print("  alternatively set PHASE2_S1_ONLY=False and describe the design as three stations.")
        train_paths=[p for p in train_paths if not (assign_phase(_meta[p]['date'])=='phase2' and _meta[p]['station']!='S1')]
else:
    print("all phase-2 particles are from S1 - data and narrative agree.")

print(f"\ntraining set (to {PHASE2_END}): {len(train_paths)} spectra")
df_train = run_pipeline(train_paths, tag="S2.1 train")

print("\n--- training-set class spec (chemical classes vs states) ---")
valid_tr_all = df_train[df_train.spec_class!='ERROR']
valid_tr     = valid_tr_all[valid_tr_all.is_chemical]
rows=[]
for cls,k in valid_tr['spec_class'].value_counts().items():
    p,lo,hi = wilson_ci(k, len(valid_tr))
    rows.append(dict(spec_class=cls, n=k, prop=round(p,3),
                     ci_lo=round(lo,3), ci_hi=round(hi,3)))
spec_train=pd.DataFrame(rows); display(spec_train)
print(f"training-set chemical classes: {valid_tr['spec_class'].nunique()} "
      f"(proportions are over the {len(valid_tr)} chemical rows)")
print("measurement / QC states:",
      valid_tr_all[~valid_tr_all.is_chemical]['spec_class'].value_counts().to_dict())

# --- did the June data add any class? ---
print("\n" + "="*72)
fig, ax = plt.subplots(figsize=(7.5,4.8))
class_accumulation(df1[df1.spec_class!='ERROR']['spec_class'].tolist(),
                   f"Phase 1 (<= {PHASE1_END})", ax=ax, color='tab:blue')
res_tr = class_accumulation(df_train[df_train.spec_class!='ERROR']['spec_class'].tolist(),
                            f"Train (<= {PHASE2_END}, S1)", ax=ax, color='black')
ax.set_title('Chemical-class accumulation: Phase 1 vs cumulative training set')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'S2_accumulation_train.png'), dpi=130); plt.show()

set_p1 = set(df1[(df1.spec_class!='ERROR') & (df1.is_chemical)]['spec_class'])
set_tr = set(valid_tr['spec_class'])
print(f"Phase-1 chemical classes {len(set_p1)} -> training set {len(set_tr)}")
print(f"  classes added by June (phase 2): {sorted(set_tr-set_p1) if set_tr-set_p1 else '(none)'}")
print(f"  classes seen only in Phase 1   : {sorted(set_p1-set_tr) if set_p1-set_tr else '(none)'}")
print("  The thresholds are fixed constants, so this difference is purely a difference in data.")
print("  This check is made AFTER the decision to move to S1 - describe it as such.")
df_train.to_csv(os.path.join(OUT_DIR,'S2_train_particles.csv'), index=False)

In [ ]:
# === Section 2.2 - vocabulary and parameter snapshot ========================
# Writes the vocabulary and thresholds of the training window (<= PHASE2_END) to a file so
# that section 3 applies exactly the same rules. If the code is changed, raise PARAM_VERSION
# and re-run from this cell.
import json, datetime
SPEC_CLASS_VOCAB_FROZEN = sorted(set(valid_tr['spec_class']))          # chemical classes only
STATE_VOCAB             = sorted(set(valid_tr_all[~valid_tr_all.is_chemical]['spec_class']))

FROZEN = dict(
    framework_version = FRAMEWORK_VERSION,
    frozen_at_utc     = datetime.datetime.now(datetime.timezone.utc).isoformat(),
    frozen_through    = PHASE2_END,
    param_version     = PARAM_VERSION,
    train_definition  = dict(phase1_end=PHASE1_END, phase2_end=PHASE2_END,
                             phase2_s1_only=PHASE2_S1_ONLY, exclude_historical=EXCLUDE_HISTORICAL),
    n_train_spectra   = int(len(df_train)),
    n_train_chemical  = int(len(valid_tr)),
    thresholds        = dict(REL_PRESENT=REL_PRESENT, REL_STRONG=REL_STRONG,
                             QC_MIN_TOTAL_A=QC_MIN_TOTAL_A, CI_WEATHERED=CI_WEATHERED,
                             SIO_DOMINATES=SIO_DOMINATES,
                             note="declared in advance; data-derived thresholds saturate at "
                                  "the upper bound on every subset"),
    preprocessing     = dict(grid=[400.0,4000.0,1.0], ALS_LAM=ALS_LAM, ALS_P=ALS_P, ALS_NITER=10,
                             SG_WIN=SG_WIN, SG_ORD=SG_ORD, MASK_REGIONS=MASK_REGIONS,
                             fringe_window=[2000,2700], fringe_score_cut=8.0, ref_percentile=99),
    accumulation      = dict(reps=RAREF_REPS, seed=RAREF_SEED, tail_frac=TAIL_FRAC,
                             chemical_only=True, result=res_tr,
                             note="descriptive only; not used as a pass or fail criterion"),
    spec_class_vocab  = SPEC_CLASS_VOCAB_FROZEN,
    state_vocab       = STATE_VOCAB,
    vocab_counts      = {k:int(v) for k,v in valid_tr['spec_class'].value_counts().items()},
    state_counts      = {k:int(v) for k,v in
                         valid_tr_all[~valid_tr_all.is_chemical]['spec_class'].value_counts().items()},
    refit_disclosure  = ("The July sample had been examined before the rule set was finalised, "
                         "so the July results are reported as a stability check under a refit, "
                         "not as held-out generalisation."),
    scope_disclosure  = ("This framework estimates neither species richness nor any mapping from "
                         "OMNIC names to chemical classes: both are deliberately outside its "
                         "scope. classify(), the preprocessing and the literature ledger are the "
                         "whole of the decision path."),
)
with open(FROZEN_PARAMS_PATH,'w',encoding='utf-8') as fh:
    json.dump(FROZEN, fh, ensure_ascii=False, indent=2, default=str)

print(f"=== spec_class vocabulary snapshot - {len(SPEC_CLASS_VOCAB_FROZEN)} chemical classes ===")
for c in SPEC_CLASS_VOCAB_FROZEN:
    print(f"  - {c:24s} n={FROZEN['vocab_counts'][c]}")
print(f"\nmeasurement / QC states (outside the vocabulary, reported separately): {STATE_VOCAB}")
print(f"thresholds (fixed): REL_PRESENT={REL_PRESENT}  REL_STRONG={REL_STRONG}")
print(f"saved: {FROZEN_PARAMS_PATH}")
print("\nreporting status -", FROZEN['refit_disclosure'])
print("Any change to a rule, threshold or vocabulary after this point costs July its "
      "held-out status.")

---
# Part 3. Applying the rules to July

The snapshot rules are applied once to the July sample. The July sample had been examined before the rule set was finalised, so this is a stability check under a refit, not held-out generalisation. Correctness is judged only in Part 4.

In [ ]:
# === Section 3.1 - July OMNIC match table (record only) ====================
# Transcribed from the laboratory log. The library's own spellings are kept verbatim, even
# where misspelled, because they are properties of the reference library.
#   Line format: "MM.DD Sx_y  | Match name, HQI"   (omit the trailing comma and number if the
#   HQI is unknown; leave the text after '|' empty when there was no match)
OMNIC_RAW_JULY = """
# 07.11
07.11 S1_1   | Duct tape (Back side), 97.04
07.11 S1_2   | Duct tape (Back side), 86.88
07.11 S1_3   | Duct tape (Back side), 97.52
# 07.12
07.12 S1_1   | Duct tape (Back side), 97.37
07.12 S1_2   | Polypropylene + 20% talc, 74.4
07.12 S1_3   | Duct tape (Back side), 97.57
07.12 S1_4   | Ethyl ethynyl ether; Ethoxyacetylene, 66.44
07.12 S1_5   | Duct tape (Back side), 95.99
07.12 S1_6   | Duct tape (Back side), 97.85
07.12 S1_7   | Hoover 512-BU, 61.89
07.12 S1_8   | Polypropylene +poly(ethylene:propylene), 78.05
07.12 S1_9   | Duct tape (Back side), 95.44
07.12 S1_10  | Duct tape (Back side), 94.28
07.12 S1_11  | Duct tape (Back side), 88.24
07.12 S1_12  | Polypropylene, isotactic, average MW ca. 250,000, 81.03
07.12 S1_13  | Spearmint Altoid, 67.5
# 07.25
07.25 S1_1   | White Play Sand, 83.50
07.25 S1_2   | Duct tape (Back side), 94.54
07.25 S1_3   | Duct tape (Back side), 88.05
07.25 S1_4   | Duct tape (Back side), 97.79
07.25 S1_5   | ALS-AR100, 74.68
07.25 S1_6   | Polyester (41% soybean oil modified phthalic alkyd), 80.23
07.25 S1_7   | Styrofoam, 73.62
07.25 S1_8   | Polypropylene, isotactic, 86.86
07.25 S1_9   | Iron pentacarbonyl, 39.25
07.25 S1_10  | Duct tape (Back side), 93.49
07.25 S1_11  | AMMONIUM-D4 THIOCYANATE, 99 ATOM % D, 65.07
07.25 S1_12  | SODIUM DICYANAMIDE, 96%, 50.47
07.25 S1_13  | Dry Milk Powder, 62.12
07.25 S1_14  | Plotter paper / glossy, 86.45

"""

df_omnic_july = build_omnic(OMNIC_RAW_JULY)
print(f"July OMNIC rows: {len(df_omnic_july)}")
display(df_omnic_july[['date','station','particle','omnic_name','hqi']])

df_omnic_all = pd.concat([df_omnic, df_omnic_july], ignore_index=True)
dups = df_omnic_all.duplicated(subset=['date','station','particle','replicate'], keep=False)
if dups.any():
    print("\nduplicate keys - check these:"); display(df_omnic_all[dups])
print(f"\nOMNIC combined: {len(df_omnic_all)} rows")
print("Rows with no HQI are excluded from the HQI descriptives in section 4.3.")

In [ ]:
# === Section 3.2 - applying the rules to July (snapshot read-only) ==========
import json, datetime
FR = json.load(open(FROZEN_PARAMS_PATH, encoding='utf-8'))
assert (FR['thresholds']['REL_PRESENT'], FR['thresholds']['REL_STRONG']) == (REL_PRESENT, REL_STRONG)
print(f"snapshot loaded: {FR['framework_version']} (training window to {FR['frozen_through']})")

july_paths = [p for p in _feat_cache if assign_phase(_meta[p]['date'])=='heldout_july']
print(f"July spectra (>= {HELDOUT_FROM}): {len(july_paths)}")
if len(july_paths)==0:
    print("  ! no July CSV in SPECTRA_DIR - add the files and re-run from the cache cell in 0.3")

df_july = run_pipeline(july_paths, tag="S3.2 july (refit)")
display(df_july[['date','station','particle','spec_class','confidence','carbonyl_index','reasons']])

json.dump(dict(scored_at_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),
               framework_version=FR['framework_version'], status="refit (not held-out)",
               thresholds=FR['thresholds'], n_scored=int(len(df_july)),
               files=[os.path.basename(p) for p in july_paths]),
          open(SCORING_LOG_PATH,'w',encoding='utf-8'), ensure_ascii=False, indent=2)
df_july.to_csv(os.path.join(OUT_DIR,'S3_july_particles_v01.csv'), index=False)
print(f"\nscoring log: {SCORING_LOG_PATH}  (overwritten on every run - keep the history "
      f"separately in the SI)")

### 3.3 Stability battery

FT-IR alone cannot say whether an answer was right; it can say whether the rules built in training operate the same way on a new batch.

In [ ]:
# === Section 3.3 - stability battery ======================================
vj_all = df_july[df_july.spec_class!='ERROR']; vj = vj_all[vj_all.is_chemical]
vt_all, vt = valid_tr_all, valid_tr

print("="*70)
print("(1) out-of-vocabulary classes - is July covered by the snapshot vocabulary?")
oov = sorted(set(vj['spec_class']) - set(FR['spec_class_vocab']))
print(f"   July chemical classes : {sorted(set(vj['spec_class']))}")
print(f"   July abstentions / QC : {sorted(set(vj_all[~vj_all.is_chemical]['spec_class']))}")
print(f"   classes outside the snapshot vocabulary: {oov if oov else '(none)'}")
_unused = sorted(set(FR['spec_class_vocab']) - set(vj['spec_class']))
print(f"   training vocabulary unused in July     : {_unused}")
print(f"   -> this batch actually exercised {len(set(vj['spec_class']))} of "
      f"{len(FR['spec_class_vocab'])} classes.")

print("\n" + "="*70)
print("(2) shift in class composition (training vs July)")
rows=[]
for cls in sorted(set(vt['spec_class']) | set(vj['spec_class'])):
    kt=int((vt.spec_class==cls).sum()); kj=int((vj.spec_class==cls).sum())
    pt,lot,hit = wilson_ci(kt,len(vt)); pj,loj,hij = wilson_ci(kj,len(vj))
    rows.append(dict(spec_class=cls, train_n=kt, train_prop=round(pt,3),
                     train_ci=f"[{lot:.2f},{hit:.2f}]", july_n=kj, july_prop=round(pj,3),
                     july_ci=f"[{loj:.2f},{hij:.2f}]", ci_overlap=not (hit<loj or hij<lot)))
display(pd.DataFrame(rows))
print("   The July batch was collected under stratification for microplastics, so the shift")
print("   towards polyolefins and away from minerals follows from the sampling design and is")
print("   not evidence that the rules are unstable.")
print("   July n is small and its intervals are wide - do not read ci_overlap=True as")
print("   'no difference'.")

print("\n" + "="*70)
print("(3) shift in confidence and abstention - does the rule withhold more often now?")
display(pd.DataFrame({'train': vt['confidence'].value_counts(normalize=True).round(3),
                      'july' : vj['confidence'].value_counts(normalize=True).round(3)}).fillna(0))
_ab_t = 1-len(vt)/len(vt_all); _ab_j = 1-len(vj)/len(vj_all)
print(f"   abstention rate: train {len(vt_all)-len(vt)}/{len(vt_all)} = {_ab_t:.3f} | "
      f"july {len(vj_all)-len(vj)}/{len(vj_all)} = {_ab_j:.3f}")

print("\n" + "="*70)
print("(4) decision-path coverage - were the rule paths July took already seen in training?")
tr_reasons = set(vt_all['reasons'])          # abstention paths included, to match vj_all
seen = [r in tr_reasons for r in vj_all['reasons']]
if len(seen):
    k=int(sum(seen)); p,lo,hi = wilson_ci(k,len(seen))
    print(f"   paths seen in training: {k}/{len(seen)} = {p:.2f} [{lo:.2f},{hi:.2f}]")
    for r in sorted(set(r for r,s in zip(vj_all['reasons'], seen) if not s)):
        print(f"   ! new path: {r}")

print("\n" + "="*70)
print("Reading: if (1) is empty, coverage in (4) is high and (3) has not moved much, then the")
print("rules operate on the new batch as they did in training. That is STABILITY UNDER A")
print("REFIT, not held-out validation; correctness is judged only in section 4.")

---
# Part 4. Comparison against the Raman Code1 Reference

### 4.1 Reading the Code 1 hand-off

`ground_truth_for_code2.csv` (section 16 of Code 1) is taken as given: this cell reads it, attaches the strata and aligns the date vocabulary; it recomputes nothing and returns nothing to Code 1.

The label is the Raman Code1 Reference: rule C over the Code 1 band library, filtered by the R3 evidence floor. No human reading, micrograph or FT-IR evidence enters it. It is deterministic but not a ground truth, so 4.2-4.4 report agreement between two automated frameworks, never accuracy.

| | |
|---|---|
| input | `ground_truth_for_code2.csv` (47 rows; join key `file`) |
| label | `gt_label`, with `gt_source` = rule / withheld |
| tier | `A_tier` = confident / tentative / unresolved |
| support | `support_level` = A+ / A / B / C / NT / WH, used as a stratum only |
| second axis | `B_label`, `B_tier`, `B_r` (SLoPP correlation); `D_label`, `D_r` (SG-derivative check) |

**Never filter by `support_level` or QC flag.** Keeping only well-corroborated particles would understate the divergence being measured. All rows are analysed and reported by stratum (`GT_QC_POLICY = 'stratify'`; `'exclude'` is a sensitivity analysis only).

Limitations carried into the manuscript: (1) the reference is a second algorithm designed by the same authors, so correlated error cannot be excluded; (2) WH is not a failure but "the rule did not decide"; exclude from the denominator, always report the count; (3) labelled particles skew towards polyolefins; (4) all 47 particles come from S1; (5) Code 1 applies no wavenumber correction beyond the instrument's Si shift; (6) the calcite / aragonite split is not trusted (all aragonite key bands collide).

In [ ]:
# === Section 4.1 - Raman Code1 Reference (reading the Code 1 hand-off) =========
# Label source: gt_label in ground_truth_for_code2.csv, written by section 16 of Code 1.
#   Code 2 is the consumer. This cell reads the file, attaches the strata and aligns the date
#   vocabulary with the FT-IR side; it does not recompute any assignment.
#   One way only: nothing produced in Code 2 returns to Code 1.

import os, json
import pandas as pd

GT_SOURCE_FILE = 'ground_truth_for_code2.csv'
GT_SOURCE      = f'Raman Code1 Reference (handoff:{GT_SOURCE_FILE}:gt_label)'
REFERENCE_KIND = ("Raman Code1 Reference - the rule output of Code 1 (diagnostic bands + rule C "
                  "+ the R3 evidence floor). NOT a ground truth: agreement with it is "
                  "inter-method agreement, not accuracy.")

# Code 1 writes to a different OUT_DIR than Code 2, so the candidates are tried in order.
HANDOFF_CSV_CANDIDATES = [
    os.path.join(OUT_DIR, GT_SOURCE_FILE),
    '/content/raman_manual_out/' + GT_SOURCE_FILE,      # Code 1 default OUT_DIR
    '/content/' + GT_SOURCE_FILE,
    '/content/drive/MyDrive/' + GT_SOURCE_FILE,
]
HANDOFF_CSV = next((p for p in HANDOFF_CSV_CANDIDATES if os.path.exists(p)), None)
if HANDOFF_CSV is None:
    raise FileNotFoundError(
        "the Code 1 hand-off CSV was not found. It is the only contract between Code 1 and\n"
        "Code 2, so no substitute source is provided.\n"
        f"  searched: {HANDOFF_CSV_CANDIDATES}\n"
        "  Run section 16 of Code 1 and place ground_truth_for_code2.csv in one of these paths.")

# ── integrity expectations ──────────────────────────────────────────────────
#   Edit these by hand when Code 1 is re-run with a different sample, and then re-read the
#   figures in 4.2. The guard exists to stop the hand-off changing silently.
HANDOFF_EXPECT = dict(
    n_particles    = 47,
    stations       = {'S1'},
    support_levels = {'A+', 'A', 'B', 'C', 'NT', 'WH'},   # WH = withheld by the R3 floor
    required_cols  = ['file', 'gt_label', 'gt_source', 'support_level', 'support_note',
                      'A_label', 'A_tier', 'A_k1', 'A_margin',
                      'B_label', 'B_tier', 'B_r', 'D_label', 'D_r',
                      'gt_class_in_library', 'date', 'station', 'particle', 'qc'],
)

# Identity stamp for Code 1. The manifest wins when present; these are the declared fallbacks.
CODE1_STAMP = dict(pipeline="Code 1 v1.0", cell8="v1.0", cell22="v1.0",
                   match_tol=8.0, K_MIN=3, GAP_MIN=4,
                   method_B="correlation matching vs SLoPP / SLoPP-E (NOT Open Specy software)",
                   window=(140.0, 3600.0),
                   wn_shift="instrument-internal, per-file")
_manifest_path = os.path.join(os.path.dirname(HANDOFF_CSV), 'handoff_manifest.json')
MANIFEST = None
if os.path.exists(_manifest_path):
    try:
        MANIFEST = json.load(open(_manifest_path, encoding='utf-8'))
        CODE1_STAMP['pipeline'] = MANIFEST.get('produced_by', CODE1_STAMP['pipeline'])
        CODE1_STAMP['cell8']    = MANIFEST.get('band_library', CODE1_STAMP['cell8'])
        for _k, _v in (MANIFEST.get('rule_C') or {}).items():
            CODE1_STAMP[_k] = _v
    except Exception as _e:
        print(f"! manifest unreadable ({_e}) - using the declared values")

# Wavenumber-axis validation from section 13 of Code 1 (Si 520.7 + ASTM E1840 polystyrene)
CODE1_AXIS = dict(
    si_after_cal   = -0.04,      # silicon residual after correction: accurate on the anchor line
    ps_slope       = -0.00204,   # residual dispersion error; two independent sessions gave
                                 # -0.00203 and -0.00205
    ps_repeat_sd   = 0.21,       # between-session reproducibility SD, at the sub-pixel limit
    zone           = {'fingerprint_600_1700': -2.64, 'CH_stretch_2800_3100': -6.79},
    applied        = False,
    reason         = ("A one-point correction on Si is exact only at 520.7 and cannot capture "
                      "dispersion. The error is not monotonic either - 2852 (-7.5) < 3054 (-6.1) - "
                      "so a single polynomial does not fit. Recalibrating against the library "
                      "would be circular and is not done; the error is reported per zone in the "
                      "limitations. Scope of the effect, measured on 47 files in sens_C_delta: at "
                      "fingerprint magnitude (delta about -2.6) no confident label changes, but at "
                      "CH-stretch magnitude (delta about -6.8) three to six do. Do not write that "
                      "no correction model changes any label."),
)

_sens_path = os.path.join(os.path.dirname(HANDOFF_CSV), 'sens_C_delta.csv')

RAMAN_DATE_ALIAS = {}                       # Raman file-name date -> FT-IR date
RAMAN_DATES_PENDING_FTIR_DECLARED = {'07.25'}   # for reference; decided from the inventory below

# Sample policy - see the markdown above for the bias argument
GT_QC_POLICY  = 'stratify'                  # 'stratify' (recommended) | 'exclude'
GT_QC_EXCLUDE = ('saturated', 'fluorescence')

# Free-text notes for the narrative. Display only; they never change a label.
GT_NOTES = {
 ('07.12','S1','2'): 'FT-IR returned PP + 20% talc: discordant',
 ('07.12','S1','6'): 'suspected FT-IR-induced artefact',
}

SUPPORT_ORDER   = ['A+', 'A', 'B', 'C', 'NT', 'WH']
SUPPORT_PRIMARY = {'A+', 'A'}               # recommended primary stratum; a stratum, not a filter
SUPPORT_SCORED  = ['A+', 'A', 'B', 'C', 'NT']   # WH carries no label and cannot be scored


# ── load and check integrity ────────────────────────────────────────────────
_raw = pd.read_csv(HANDOFF_CSV, encoding='utf-8-sig',
                   dtype={'station': str, 'particle': str, 'date': str})
print("=" * 72)
print(f"[hand-off] {HANDOFF_CSV}")
print(f"           {len(_raw)} rows x {len(_raw.columns)} columns"
      + (f"  |  manifest {os.path.basename(_manifest_path)} ok" if MANIFEST else
         "  |  no manifest (optional)"))

_missing = [c for c in HANDOFF_EXPECT['required_cols'] if c not in _raw.columns]
if _missing:
    raise KeyError(f"required columns absent from the hand-off CSV: {_missing}\n"
                   f"  present: {list(_raw.columns)}\n"
                   "  Re-run section 16 of Code 1 from the current version.")

_guard = []
if len(_raw) != HANDOFF_EXPECT['n_particles']:
    _guard.append(f"{len(_raw)} rows, expected {HANDOFF_EXPECT['n_particles']}")
if set(_raw['station']) - HANDOFF_EXPECT['stations']:
    _guard.append(f"unknown station {sorted(set(_raw['station']) - HANDOFF_EXPECT['stations'])}")
if set(_raw['support_level']) - HANDOFF_EXPECT['support_levels']:
    _guard.append(f"unknown support_level {sorted(set(_raw['support_level']) - HANDOFF_EXPECT['support_levels'])}")
if _raw['file'].duplicated().any():
    _guard.append(f"{int(_raw['file'].duplicated().sum())} duplicate file key(s)")
if MANIFEST and MANIFEST.get('n_particles') not in (None, len(_raw)):
    _guard.append(f"manifest n_particles {MANIFEST['n_particles']} != CSV {len(_raw)}")
if _guard:
    print("[guard] the hand-off differs from what is expected. If Code 1 was re-run, update")
    print("        HANDOFF_EXPECT and re-read every figure from 4.2 onward:")
    for _g in _guard:
        print(f"          - {_g}")
else:
    print("[guard] rows, columns, station and support_level all as expected")

# Optional columns of the hand-off (present when Code 1 wrote them).
_OPT = {c: (c in _raw.columns) for c in
        ['ab_agree', 'gt_a_agree', 'gt_b_agree', 'D_tier', 'xmethod_agree', 'xmethod_metric_stable']}
if not _OPT['ab_agree']:
    print("  (older CSV: ab_agree/gt_a_agree/gt_b_agree absent - computing A_label == B_label)")


# ── build the reference frame ───────────────────────────────────────────────
def _pick(name, default=None):
    return _raw[name] if name in _raw.columns else default

gt = pd.DataFrame(index=_raw.index)
# The join key is taken from the FILE NAME, avoiding the float round trip of the date column
# ('5.17' -> '05.17').
gt['date']     = _raw['file'].str[:2] + '.' + _raw['file'].str[2:4]
gt['station']  = _raw['station'].astype(str)
gt['particle'] = _raw['particle'].astype(str)
_date_col = _raw['date'].map(lambda x: f"{float(x):05.2f}")
if not (gt['date'] == _date_col).all():
    print("[guard] the date from the file name differs from the date column - check the naming")

gt['raman_file']    = _raw['file']
gt['qc']            = _raw['qc'].astype(str)
gt['raman_gt']      = _raw['gt_label']
gt['gt_source']     = _raw['gt_source']
gt['raman_conf']    = _raw['A_tier']                  # read directly; no string parsing
gt['A_label']       = _raw['A_label']
gt['key_hit']       = pd.to_numeric(_raw['A_k1'], errors='coerce').astype('Int64')
gt['gap']           = pd.to_numeric(_raw['A_margin'], errors='coerce').astype('Int64')
gt['tie_break']     = (gt['gap'].fillna(-1) == 0)
gt['support_level'] = _raw['support_level']
gt['support_note']  = _raw['support_note']
gt['B_label']       = _raw['B_label']
gt['B_tier']        = _raw['B_tier']
gt['B_r']           = pd.to_numeric(_raw['B_r'], errors='coerce')
gt['D_label']       = _raw['D_label']
gt['D_r']           = pd.to_numeric(_raw['D_r'], errors='coerce')
gt['gt_class_in_library'] = _raw['gt_class_in_library'].astype(bool)
gt['ab_agree']      = (_pick('ab_agree') if _OPT['ab_agree']
                       else (_raw['A_label'] == _raw['B_label']))
gt['metric_stable'] = (_raw['xmethod_metric_stable'].astype(bool)
                       if _OPT['xmethod_metric_stable'] else pd.NA)
gt['wn_delta_applied'] = pd.to_numeric(_pick('wn_delta_applied', 0.0), errors='coerce')
gt['delta_version']    = _pick('delta_version', 'unknown')
gt['top_peaks_cm1']    = _pick('top_peaks_cm1', '')     # 4.4 rebuilds the helix table from this
DELTA_MEANING = ("instrument-internal Si shift applied per file at save time; "
                 "no post-hoc correction by Code1 (avoids circularity)")

# key_hit is the integer A_k1. The denominator (key bands per class) is deliberately NOT
# transferred: copying it would give Code 2 a duplicate of the Code 1 band list, and the two
# notebooks would drift apart silently.

# ── scoreability: Code 1's judgement is taken as given ──────────────────────
#   The R3 evidence floor in section 16 of Code 1 already marks the particles the rule could
#   not decide with gt_source='withheld' and sends an empty gt_label. Code 2 does not re-derive
#   that decision: implementing the same rule twice guarantees they eventually diverge.
gt['gt_label_raw']        = gt['raman_gt']
gt['gt_withheld_reason']  = _pick('gt_withheld_reason', '')
if (gt['gt_source'] == 'withheld').any():
    _wh = gt['gt_source'] == 'withheld'
    gt.loc[_wh, 'raman_gt'] = 'unresolved'          # reuses the existing exclusion mechanism
else:
    # Older CSV (gt_source in {auto, manual}, no floor): apply the first condition here instead.
    _auto_unres = (gt['gt_source'] == 'auto') & (gt['raman_conf'] == 'unresolved')
    gt.loc[_auto_unres, 'raman_gt'] = 'unresolved'
    gt.loc[_auto_unres, 'gt_withheld_reason'] = 'rule_C_unresolved'
    print("  (older CSV, no R3 floor: only rule_C_unresolved is applied here; tie-break and")
    print("   fluorescence cases are NOT filtered)")

# ── sample policy ───────────────────────────────────────────────────────────
gt['qc_flagged'] = gt['qc'].str.contains('|'.join(GT_QC_EXCLUDE), case=False, na=False)
_n_all, _n_qc = len(gt), int(gt['qc_flagged'].sum())
if GT_QC_POLICY == 'exclude':
    gt = gt[~gt['qc_flagged']].copy()
    print(f"[sample] GT_QC_POLICY='exclude' -> {_n_qc} QC-flagged particle(s) dropped, "
          f"{len(gt)} remain")
    print("         QC flags concentrate in the lower support levels, so excluding them cuts")
    print("         the harder strata. 'stratify' is recommended for the primary analysis.")
else:
    print(f"[sample] GT_QC_POLICY='stratify' -> all {len(gt)} kept "
          f"({_n_qc} QC-flagged, stratified by the qc_flagged column)")
    print(f"         (for reference, 'exclude' would have left {_n_all - _n_qc})")

# ── align the date vocabulary; derived columns ──────────────────────────────
gt['date'] = gt['date'].map(lambda d: RAMAN_DATE_ALIAS.get(d, d))
gt['phase'] = [assign_phase(d, s) for d, s in zip(gt['date'], gt['station'])]
gt['flag']  = gt.apply(lambda r: GT_NOTES.get((r['date'], r['station'], r['particle']), ''), axis=1)
gt['raman_note'] = gt.apply(
    lambda r: (f"A={r['A_label']}/{r['raman_conf']} k1={r['key_hit']} gap={r['gap']}"
               f" | B={r['B_label']}/{r['B_tier']} r={r['B_r']:.3f}"
               f" | {r['support_level']} | qc={r['qc']}"
               + (" | tie-break" if r['tie_break'] else "")
               + (f" | {r['flag']}" if r['flag'] else "")), axis=1)

# ── check against the FT-IR inventory: which dates can be cross-compared ────
_ram_dates_all = set(gt['date'])
_ftir_frames = [f for f in (globals().get('df_train'), globals().get('df_july'))
                if f is not None and len(f)]
if _ftir_frames:
    _ftir_dates_inv = set(pd.concat(_ftir_frames, ignore_index=True)['date'].astype(str))
    RAMAN_DATES_PENDING_FTIR = _ram_dates_all - _ftir_dates_inv
    print(f"[FT-IR check] FT-IR dates  {sorted(_ftir_dates_inv)}")
    print(f"              Raman dates  {sorted(_ram_dates_all)}")
    _resolved = RAMAN_DATES_PENDING_FTIR_DECLARED - RAMAN_DATES_PENDING_FTIR
    _newpend  = RAMAN_DATES_PENDING_FTIR - RAMAN_DATES_PENDING_FTIR_DECLARED
    if _resolved:
        print(f"              FT-IR now available, entering the comparison: {sorted(_resolved)}")
    if _newpend:
        print(f"              undeclared dates without FT-IR: {sorted(_newpend)} - check first "
              f"whether a RAMAN_DATE_ALIAS entry is missing")
    if not RAMAN_DATES_PENDING_FTIR:
        print("              every date has FT-IR - nothing excluded from the comparison")
else:
    RAMAN_DATES_PENDING_FTIR = set(RAMAN_DATES_PENDING_FTIR_DECLARED)
    print(f"[FT-IR check] df_train / df_july undefined - using the declared value: "
          f"{sorted(RAMAN_DATES_PENDING_FTIR)}")

gt['ftir_pending'] = gt['date'].isin(RAMAN_DATES_PENDING_FTIR)

GT_COLS = ['date', 'station', 'particle', 'raman_file', 'qc', 'qc_flagged',
           'raman_gt', 'gt_label_raw', 'gt_source', 'gt_withheld_reason',
           'raman_conf', 'A_label',
           'key_hit', 'gap', 'tie_break',
           'support_level', 'support_note', 'B_label', 'B_tier', 'B_r',
           'D_label', 'D_r', 'ab_agree', 'metric_stable', 'gt_class_in_library',
           'flag', 'phase', 'raman_note', 'top_peaks_cm1',
           'wn_delta_applied', 'delta_version', 'ftir_pending']
gt = gt[GT_COLS].reset_index(drop=True)

# labels the rule withheld are excluded from the accuracy denominator
GT_EXCLUDE_FROM_ACCURACY = {'unresolved', 'coating_alkyd'}

# subset used for the cross-comparison in 4.2 and 4.3: dates with FT-IR only
gt_x = gt[~gt['ftir_pending']].reset_index(drop=True)

_n_wh = int((gt['gt_source'] == 'withheld').sum())
_pipe = str(CODE1_STAMP['pipeline'])
_pipe = _pipe if _pipe.lower().startswith('code') else f"Code1 {_pipe}"
GT_VERSION = (f"Raman Code1 Reference ({_pipe}, "
              f"emitted {len(gt) - _n_wh}/{len(gt)}, withheld {_n_wh}"
              + (f" [cross-compared {len(gt_x)} + {int(gt['ftir_pending'].sum())} without FT-IR]"
                 if len(gt) != len(gt_x) else " [all cross-compared]") + ")")
gt['gt_ver'] = GT_VERSION
gt_x['gt_ver'] = GT_VERSION


# ── report ──────────────────────────────────────────────────────────────────
_scored = gt_x[~gt_x.raman_gt.isin(GT_EXCLUDE_FROM_ACCURACY)]
print("\n" + "=" * 72)
print(f"REFERENCE = {GT_VERSION}")
print(f"  source        : {GT_SOURCE}")
print(f"  Code 1        : {CODE1_STAMP['pipeline']} · bands {CODE1_STAMP['cell8']} · "
      f"K_MIN={CODE1_STAMP.get('K_MIN')} GAP_MIN={CODE1_STAMP.get('GAP_MIN')}")
print(f"  method B      : {CODE1_STAMP['method_B']}")
print(f"  hand-off      : {len(_raw)} rows -> {len(gt)} in the sample -> {len(gt_x)} "
      f"cross-compared -> {len(_scored)} SCORED "
      f"({len(gt_x) - len(_scored)} withheld by the R3 floor)")
print(f"  delta         : {gt['delta_version'].iloc[0]}")
print("  raman_conf    :", gt['raman_conf'].value_counts().to_dict())
print("  phase         :", gt['phase'].value_counts().to_dict())
print("  class         :", gt['raman_gt'].value_counts().to_dict())

print("\n" + "=" * 72)
print("What the reference is -", REFERENCE_KIND)
print("  Sections 4.2 to 4.4 measure AGREEMENT between two automated frameworks, not accuracy.")
print("  [reporting note] term: 'agreement with the Raman Code1 Reference' (not 'accuracy',")
print("                   not 'ground truth')")
if _n_wh:
    print(f"  {_n_wh} particle(s) withheld - the R3 evidence floor judged that the rule did not")
    print("  decide:")
    for _r, _c in gt.loc[gt.gt_source == 'withheld', 'gt_withheld_reason'].value_counts().items():
        print(f"      {_r:<24} {_c}")
    print("  Neither a poor particle nor a failure of Code 2. Exclude them from the denominator,")
    print("  but report the count.")

# ── support levels ──────────────────────────────────────────────────────────
print("\n[support_level] cross-validation support (WH = withheld by the R3 floor)")
_lv = gt['support_level'].value_counts().reindex(SUPPORT_ORDER).fillna(0).astype(int)
for _l in SUPPORT_ORDER:
    _sub = gt[gt.support_level == _l]
    if _l == 'WH':
        _rs = dict(_sub.gt_withheld_reason.value_counts()) if len(_sub) else {}
        print(f"   {_l:>2}  n={_lv[_l]:>2}  (no label) reason={_rs}")
    else:
        print(f"   {_l:>2}  n={_lv[_l]:>2}  "
              f"class={dict(_sub.raman_gt.value_counts()) if len(_sub) else {}}")
print(f"   recommended primary stratum A+/A = {int(_lv['A+'] + _lv['A'])} "
      f"(report every stratum, do not narrow to one)")
print("   Do not filter particles by support_level: defining the reference as 'the particles")
print("   the library matched well' keeps only what libraries handle well and understates the")
print("   divergence by construction. Analyse all rows and report by stratum.")
if int(_lv['NT']):
    print(f"   NT {int(_lv['NT'])}: method B had no reference to compare against, which is a")
    print("   property of the library rather than of the particle. Report apart from C.")
    display(gt[gt.support_level == 'NT'][['date', 'station', 'particle', 'raman_gt', 'support_note']])

# QC x support level - the evidence behind the sample policy
print("\n[QC x support_level] which strata an 'exclude' policy would cut")
display(pd.crosstab(gt['support_level'], gt['qc_flagged'], margins=True)
          .reindex(SUPPORT_ORDER + ['All']))

# ── wavenumber axis ─────────────────────────────────────────────────────────
print("\n[delta] the instrument-internal Si shift is already applied per file at save time.")
print(f"    accuracy of the correction : Si 520.7 residual {CODE1_AXIS['si_after_cal']:+.2f} cm-1")
print(f"    residual dispersion error  : PS slope {CODE1_AXIS['ps_slope']:+.5f} "
      f"(two-session SD {CODE1_AXIS['ps_repeat_sd']:.2f}), by zone {CODE1_AXIS['zone']}")
print("    applied                    : NO - " + CODE1_AXIS['reason'])
if os.path.exists(_sens_path):
    try:
        _sens_live = pd.read_csv(_sens_path)
        print(f"    delta sensitivity from the current Code 1 run "
              f"({os.path.basename(_sens_path)}, {len(_sens_live)} rows)")
        display(_sens_live)
        _cc = 'confident_label_changed'
        if _cc in _sens_live.columns:
            _z = _sens_live.loc[_sens_live[_cc] > 0, 'delta_cm1']
            print(f"      confident labels start changing at delta {_z.abs().min():+.0f} cm-1 "
                  f"(measured zone errors: fingerprint -2.6, CH stretch -6.8)")
    except Exception as _e:
        print(f"    ! sens_C_delta.csv unreadable ({_e})")
else:
    print(f"    sens_C_delta.csv not found in {os.path.dirname(HANDOFF_CSV)}.")
    print("    Copy it from the Code 1 output before quoting any delta sensitivity figure;")
    print("    no substitute table is shipped here, so that stale numbers cannot be quoted.")

_tie = gt[gt.tie_break]
if len(_tie):
    _tie_wh = int((_tie.gt_source == 'withheld').sum())
    print(f"\n[tie-break] {len(_tie)} particle(s) with a margin of 0, where the sort order "
          f"decided the top candidate. The R3 floor withheld {_tie_wh}"
          + ("" if _tie_wh == len(_tie) else
             f" - {len(_tie)-_tie_wh} still carry a label; re-check the floor rule"))
    display(_tie[['date', 'station', 'particle', 'raman_gt', 'key_hit',
                  'support_level', 'raman_note']])

_pend = gt[gt.ftir_pending]
if len(_pend):
    print(f"\n[no FT-IR] {len(_pend)} particle(s) with a Raman Code1 Reference only - excluded "
          f"from scoring, recorded here:")
    display(_pend[['date', 'station', 'particle', 'raman_gt', 'raman_conf',
                   'support_level', 'phase']])
    print("   They enter the comparison automatically once the FT-IR spectra are added.")
    print("   07.25 was collected after the 07.11-12 results were seen, so report the two")
    print("   held-out batches separately.")

# ── provenance ──────────────────────────────────────────────────────────────
#   Code 1 is the single source for the Raman band ledger and decision rules; Code 2 keeps no
#   copy. If Code 1 changes, the CSV changes, and the HANDOFF_EXPECT guard above catches it.
print("\n[provenance] Raman bands and decision rules: Code 1 is the single source")
print(f"       {CODE1_STAMP['pipeline']} · bands {CODE1_STAMP['cell8']}"
      + (f" · manifest {MANIFEST.get('produced_at')}" if MANIFEST else ""))
print("       Do not call method B 'Open Specy'. The software was not run; correlation")
print("       matching was performed against the same SLoPP / SLoPP-E libraries.")

### 4.2 spec_class vs the Raman Code1 Reference (primary result)

Decision tiers are defined in advance. An abstention by `spec_class` is a design feature, removed from the denominator and reported separately.

In [ ]:
# === Section 4.2 - spec_class vs Raman Code1 Reference (primary result) =======
# Decision tiers, defined in advance:
#   exact        spec_class == raman_gt after the vocabulary bridge (SAN <-> acrylonitrile)
#   compatible   polyolefin_unresolved <-> {PE, PP}: family agrees, FT-IR declines to split
#   abstain      FT-IR returned a state (NON_CHEMICAL_STATES); not a wrong answer
#   mismatch     anything else
# primary metric: exact/compatible over POLYMERS ONLY (two non-polymers agreeing is not a
#   hit for the question asked). secondary: recognition of non-polymers at tier level.
# 07.11-12 and 07.25 were scored under the same frozen rules; reported separately only
#   because each batch is small.
HELDOUT_BATCH = {'07.11': 'heldout_1', '07.12': 'heldout_1', '07.25': 'heldout_2'}

VOCAB_BRIDGE = {'SAN': 'acrylonitrile_copolymer'}       # FT-IR term -> Raman term
COMPATIBLE_PAIRS = {
    ('polyolefin_unresolved', 'polyolefin_PE'),
    ('polyolefin_unresolved', 'polyolefin_PP'),
}

def spec_vs_gt_tier(spec, gt_lbl):
    s = VOCAB_BRIDGE.get(spec, spec)
    if s in NON_CHEMICAL_STATES:            return 'abstain'
    if s == gt_lbl:                         return 'exact'
    if (s, gt_lbl) in COMPATIBLE_PAIRS:     return 'compatible'
    return 'mismatch'

# ── merge: the reference (4.1) with the raw spec_class ──────────────────────
df_all_raw = pd.concat([df_train, df_july], ignore_index=True)
for _c in ['particle', 'date']:
    df_all_raw[_c] = df_all_raw[_c].astype(str)
    gt[_c]         = gt[_c].astype(str)
    gt_x[_c]       = gt_x[_c].astype(str)

# ── orphan-date guard ───────────────────────────────────────────────────────
_ram_dates, _ftir_dates = set(gt['date']), set(df_all_raw['date'])
_orphan_all     = _ram_dates - _ftir_dates
_orphan_pending = sorted(_orphan_all & RAMAN_DATES_PENDING_FTIR)
_orphan_unexp   = sorted(_orphan_all - RAMAN_DATES_PENDING_FTIR)
if _orphan_unexp:
    raise AssertionError(
        f"reference dates absent from the FT-IR set, which would make the whole date NaN: "
        f"{_orphan_unexp}\n"
        f"  FT-IR dates: {sorted(_ftir_dates)}\n"
        f"  Check RAMAN_DATE_ALIAS in 4.1; if the date genuinely has no FT-IR, declare it in "
        f"RAMAN_DATES_PENDING_FTIR.")
if _orphan_pending:
    print(f"[excluded from the comparison] no FT-IR for {_orphan_pending}: "
          f"{int(gt['ftir_pending'].sum())} of {len(gt)} dropped, {len(gt_x)} remain")

MS = gt_x.merge(df_all_raw[['date', 'station', 'particle',
                            'spec_class', 'confidence', 'carbonyl_index', 'reasons']],
                on=['date', 'station', 'particle'], how='left')
_miss = MS[MS['spec_class'].isna()]
if len(_miss):
    print(f"{len(_miss)} particle(s) have a reference but no FT-IR spectrum - check the file "
          f"names and keys:")
    display(_miss[['date', 'station', 'particle', 'raman_gt']])

# ── attach tier and batch labels ────────────────────────────────────────────
A_s = MS[~MS['raman_gt'].isin(GT_EXCLUDE_FROM_ACCURACY)].copy()   # a Raman label exists
A_s['tier']       = [spec_vs_gt_tier(s, g) for s, g in zip(A_s['spec_class'], A_s['raman_gt'])]
A_s['gt_tier']    = A_s['raman_gt'].map(to_tier)                  # tier on the Raman side
A_s['spec_tier']  = A_s['spec_class'].map(lambda c: to_tier(c) if isinstance(c, str) else 'abstain')
A_s['batch']      = A_s['date'].map(lambda d: HELDOUT_BATCH.get(d, 'train'))

# ── partition every labelled particle into three, with nothing lost ─────────
#   ABSTAIN    spec returned a measurement state: an abstention, not a wrong answer
#   CROSS      Raman says polymer, FT-IR says non-polymer: a real failure, but outside the
#              primary metric
#   POLY_PAIR  both call it a polymer: the primary scoring set
ABSTAIN   = A_s[A_s.tier == 'abstain']
CROSS     = A_s[(A_s.tier != 'abstain') &
                ((A_s.gt_tier != 'polymer') | (A_s.spec_tier != 'polymer'))]
POLY_PAIR = A_s[(A_s.tier != 'abstain') &
                (A_s.gt_tier == 'polymer') & (A_s.spec_tier == 'polymer')]
scored    = POLY_PAIR
assert len(ABSTAIN) + len(CROSS) + len(scored) == len(A_s), "partition lost rows"

print("=" * 72)
print("A. PRIMARY METRIC - naming agreement, polymers only")
print(f"   particles with a Raman label {len(A_s)}")
print(f"     FT-IR abstained    {len(ABSTAIN):>3}  (block E; not a wrong answer)")
print(f"     tier mismatch      {len(CROSS):>3}  (a failure, but outside the primary metric - A-2)")
print(f"     SCORED PAIRS       {len(scored):>3}")
k_e = int((scored.tier == 'exact').sum())
k_c = int((scored.tier == 'compatible').sum())
for lab, k in [('exact', k_e), ('exact+compatible', k_e + k_c)]:
    p_, lo, hi = wilson_ci(k, len(scored))
    print(f"     {lab:18s}: {k}/{len(scored)} = {p_:.2f}  Wilson 95% CI [{lo:.2f},{hi:.2f}]")
display(pd.crosstab(scored['raman_gt'], scored['tier'], margins=True))

print("\n" + "-" * 72)
print("A-2. TIER MISMATCH - excluded from the primary metric, but report it as a failure")
print("     Particles the reference calls a polymer and FT-IR calls a non-polymer.")
_deno = len(scored) + len(CROSS)
_hit  = len(scored)
if _deno:
    p_, lo, hi = wilson_ci(_hit, _deno)
    print(f"   polymer recall: {_hit}/{_deno} = {p_:.2f} [{lo:.2f},{hi:.2f}]")
    print("     = of the reference-polymers on which FT-IR did not abstain, the fraction it")
    print("       also called a polymer")
    print("     [reporting note] report alongside exact.")
if len(CROSS):
    display(CROSS[['date', 'station', 'particle', 'raman_gt', 'raman_conf',
                   'spec_class', 'confidence', 'batch']])
else:
    print("   (none)")

print("\n" + "=" * 72)
print("B. SECONDARY METRIC - recognition of non-polymers (never mixed into the primary)")
print("   Question: does FT-IR also call a non-polymer what the reference calls a non-polymer?")
_NP = A_s[A_s.gt_tier == 'non_polymer']
if len(_NP):
    _hit = int((_NP.spec_tier == 'non_polymer').sum())
    p, lo, hi = wilson_ci(_hit, len(_NP))
    print(f"   tier agreement: {_hit}/{len(_NP)} = {p:.2f}  [{lo:.2f},{hi:.2f}]")
    _NP = _NP.copy()
    _NP['gt_rep']   = _NP['raman_gt'].map(to_reporting)
    _NP['spec_rep'] = _NP['spec_class'].map(lambda c: to_reporting(c) if isinstance(c, str) else 'abstain')
    display(_NP[['date', 'station', 'particle', 'gt_rep', 'spec_rep', 'confidence']])
else:
    print("   The Raman Code1 Reference contains no non-polymers, so this metric cannot be")
    print("   computed. The key bands of rule C are concentrated in the polymer classes, so")
    print("   non-polymer candidates mostly reach only k1<=1 and are withheld by the R3 floor:")
    print("   a property of the rule rather than of the sample.")
    print("   FT-IR assigns many particles to non-polymer classes, and those assignments are")
    print("   therefore UNVERIFIED. The metric requires reading non-polymer particles by Raman")
    print("   as well - state this in the limitations.")
_np_ftir = df_all_raw['spec_class'].map(lambda c: to_tier(c)).value_counts()
print(f"\n   for reference, tiers over all {len(df_all_raw)} FT-IR particles: {_np_ftir.to_dict()}")

print("\n" + "=" * 72)
print("C. every mismatch (source table for the figure)")
# Worst-case bound: counting abstentions and tier mismatches as errors. Excluding the
# abstentions from the denominator is the design, but a reader who disagrees with that
# treatment should see the lower bound; the sample supports the range between the two.
_n_lab   = int(len(A_s))
_n_exact = int((scored.tier == 'exact').sum())
if _n_lab:
    print(f"\n   reporting range (exact)")
    print(f"      abstentions and tier mismatches excluded : {_n_exact}/{len(scored)} = "
          f"{_n_exact/len(scored):.2f}   <- primary metric")
    print(f"      all counted as errors (lower bound)      : {_n_exact}/{_n_lab} = "
          f"{_n_exact/_n_lab:.2f}")
    print(f"      [reporting note] range: "
          f"'{_n_exact/_n_lab:.2f}-{_n_exact/len(scored):.2f}'")

Dm = scored[scored.tier == 'mismatch'][['date', 'station', 'particle', 'raman_gt',
                                        'support_level', 'B_label', 'raman_conf',
                                        'spec_class', 'confidence', 'tie_break', 'batch', 'reasons']]
display(Dm)
if len(Dm):
    _tb = Dm[Dm['tie_break'].fillna(False)]
    if len(_tb):
        print(f"   {len(_tb)} of these rest on a tie-break in the reference itself (4.1), so the")
        print("   disagreement may not originate with FT-IR.")

print("\n" + "=" * 72)
print("D. every compatible case - describe as a difference in specificity, not as an error")
display(scored[scored.tier == 'compatible'][['date', 'station', 'particle', 'raman_gt',
                                             'spec_class', 'confidence', 'batch']])

print("\n" + "=" * 72)
print("E. every FT-IR abstention - the reference gave a label, FT-IR returned a state")
if len(ABSTAIN):
    display(ABSTAIN[['date', 'station', 'particle', 'raman_gt', 'spec_class',
                     'confidence', 'batch', 'reasons']])
    print(f"   {len(ABSTAIN)} particle(s). An abstention is not a wrong answer, but the rate of")
    print(f"   abstention is itself a performance figure: "
          f"{len(ABSTAIN)}/{len(A_s)} = {len(ABSTAIN)/len(A_s):.2f}")
else:
    print("   (none)")

print("\n" + "=" * 72)
print("E-2. the reverse direction - what FT-IR said where the Raman rule withheld a label")
U_s = MS[MS['raman_gt'].isin(GT_EXCLUDE_FROM_ACCURACY)]
if len(U_s):
    display(U_s[['date', 'station', 'particle', 'raman_note', 'spec_class', 'confidence']])
else:
    print("   (nothing withheld)")

print("\n" + "=" * 72)
print("F. strata - the two held-out batches are reported separately (exact; n is small, so")
print("   speak only through the intervals)")
def _rate(sub, label):
    if not len(sub):
        print(f"   {label:26s} n=0"); return
    k = int((sub.tier == 'exact').sum()); p, lo, hi = wilson_ci(k, len(sub))
    print(f"   {label:26s} exact {k}/{len(sub)} = {p:.2f} [{lo:.2f},{hi:.2f}]")

# support_level strata - report every one; do not narrow to a single stratum
print("   -- support_level (cross-validation support from Code 1) --")
for v in SUPPORT_SCORED:
    _rate(scored[scored.support_level == v], f"support_level={v}")
_n_wh_all = int((gt_x.support_level == 'WH').sum()) if 'support_level' in gt_x.columns else 0
if _n_wh_all:
    print(f"   support_level=WH        n=0 (cannot be scored) - {_n_wh_all} withheld by the "
          f"R3 floor, no label")
_rate(scored[scored.support_level.isin(SUPPORT_PRIMARY)], "support_level=A+/A (primary)")
_rate(scored, "all strata")

# Reported one by one the intervals all overlap at this n; a single ordinal trend test says
# more. Scores A+=0 < A=1 < B=2 < C=3. NT is 'not testable' and does not lie on that axis,
# so it is excluded and reported separately.
def _cochran_armitage(sub, score_map):
    import math
    g = [(score_map[l], int((d.tier == 'exact').sum()), len(d))
         for l, d in sub.groupby('support_level') if l in score_map and len(d)]
    if len(g) < 3:
        return None
    N = sum(n for _, _, n in g); X = sum(x for _, x, _ in g)
    if N == 0 or X in (0, N):
        return None
    p = X / N
    T  = sum(t * (x - n * p) for t, x, n in g)
    V  = p * (1 - p) * (sum(n * t * t for t, _, n in g) - sum(n * t for t, _, n in g) ** 2 / N)
    if V <= 0:
        return None
    z = T / math.sqrt(V)
    return z, 2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2)))), N

_ca = _cochran_armitage(scored[scored.support_level != 'NT'],
                        {'A+': 0, 'A': 1, 'B': 2, 'C': 3})
if _ca:
    _z, _p, _n = _ca
    print(f"   -- trend test (Cochran-Armitage, A+<A<B<C, n={_n}): z={_z:+.2f}, p={_p:.3f}")
    print("      " + ("exact agreement falls significantly as the support level drops."
                      if _p < 0.05 else
                      "no significant monotonic trend across strata - write 'no detectable trend'."))
else:
    print("   -- trend test: fewer than three strata, or zero variance; not computable")
_nt = scored[scored.support_level == 'NT']
if len(_nt):
    print(f"   -- NT ({len(_nt)}) is excluded from the trend test as a not-testable stratum; "
          f"report it separately")
print()

for v in ['confident', 'tentative']:
    _rate(scored[scored.raman_conf == v], f"raman_conf={v}")
for v in ['confident', 'tentative']:
    _rate(scored[scored.confidence == v], f"ftir_confidence={v}")
print()
for v in ['phase1', 'phase2']:
    _rate(scored[scored.phase == v], f"phase={v}")
_b1 = scored[scored.batch == 'heldout_1']
_b2 = scored[scored.batch == 'heldout_2']
_rate(_b1, "held-out 1 (07.11-12)")
_rate(_b2, "held-out 2 (07.25)")
_rate(scored[scored.batch.isin(['heldout_1', 'heldout_2'])], "held-out pooled")
if len(_b1) and len(_b2):
    from scipy import stats as _st42
    _e1 = int((_b1.tier == 'exact').sum()); _e2 = int((_b2.tier == 'exact').sum())
    _, _pb = _st42.fisher_exact([[_e1, len(_b1) - _e1], [_e2, len(_b2) - _e2]])
    print(f"\n   Sampling continued through July; both batches were scored under the same frozen")
    print(f"   rules. They are shown separately because each is small ({len(_b1)} and {len(_b2)}),")
    print(f"   not because they differ in status - Fisher exact p = {_pb:.2f} between them.")
print("   Where the intervals overlap, write 'cannot reliably distinguish' and make no ranking")
print("   claim.")

print("\n" + "=" * 72)
print("G. no FT-IR yet - a Raman Code1 Reference only (excluded from scoring, recorded here)")
_pend = gt[gt['ftir_pending']]
if len(_pend):
    display(_pend[['date', 'station', 'particle', 'raman_gt', 'raman_conf', 'key_hit', 'phase']])
    print(f"   {len(_pend)} particle(s). Once measured, remove the date from "
          f"RAMAN_DATES_PENDING_FTIR in 4.1 and re-run.")
else:
    print("   (none)")

print("\n" + "=" * 72)
print("H. effect of the rule change on the metric - the primary figures recomputed under both")
print("   CI_GATE modes. Publishing this is what distinguishes a principled change from tuning.")
if '_feat_cache' not in dir() or not _feat_cache or 'CI_GATE' not in dir():
    print("   _feat_cache or CI_GATE unavailable - not computed")
else:
    _t_lut = {}
    for _t_p, (_t_x, _t_pz, _t_f) in _feat_cache.items():
        _t_m = _meta[_t_p]
        _t_lut.setdefault((str(_t_m['date']), str(_t_m['station']), str(_t_m['particle'])), _t_f)
    _t_save = CI_GATE
    _t_out = []
    for _t_g in ['always', 'incomplete_only']:
        globals()['CI_GATE'] = _t_g
        _t_rows = []
        for _, _t_r in A_s.iterrows():
            _t_f2 = _t_lut.get((str(_t_r['date']), str(_t_r['station']), str(_t_r['particle'])))
            if _t_f2 is None:
                continue
            _t_sc = classify(_t_f2)[0]
            _t_rows.append((_t_r['raman_gt'], _t_sc, spec_vs_gt_tier(_t_sc, _t_r['raman_gt']),
                            to_tier(_t_sc)))
        _t_ab = sum(1 for _, _, t, _x in _t_rows if t == 'abstain')
        _t_sc2 = [(g, s2, t) for g, s2, t, st in _t_rows if t != 'abstain' and st == 'polymer'
                  and to_tier(g) == 'polymer']
        _t_cr = len(_t_rows) - _t_ab - len(_t_sc2)
        _t_e = sum(1 for _, _, t in _t_sc2 if t == 'exact')
        _t_c = sum(1 for _, _, t in _t_sc2 if t == 'compatible')
        _t_n = len(_t_sc2)
        _t_out.append(dict(CI_GATE=_t_g, scored_pairs=_t_n, abstained=_t_ab, tier_mismatch=_t_cr,
                           exact=f"{_t_e}/{_t_n}",
                           exact_p=round(_t_e/_t_n, 3) if _t_n else float('nan'),
                           ex_comp=f"{_t_e+_t_c}/{_t_n}",
                           ex_comp_p=round((_t_e+_t_c)/_t_n, 3) if _t_n else float('nan'),
                           polymer_recall=round(_t_n/max(_t_n+_t_cr, 1), 3)))
    globals()['CI_GATE'] = _t_save
    GATE_IMPACT = pd.DataFrame(_t_out)
    display(GATE_IMPACT)
    if len(GATE_IMPACT) == 2:
        _t_d = GATE_IMPACT.iloc[1]['exact_p'] - GATE_IMPACT.iloc[0]['exact_p']
        _t_d2 = GATE_IMPACT.iloc[1]['ex_comp_p'] - GATE_IMPACT.iloc[0]['ex_comp_p']
        print(f"\n   always -> incomplete_only :  exact {_t_d:+.3f}  ·  "
              f"exact+compatible {_t_d2:+.3f}")
        print(f"   in force = {CI_GATE!r}")
        print(f"   [reporting note] exact agreement, always -> incomplete_only: "
              f"{GATE_IMPACT.iloc[0]['exact_p']:.2f} -> {GATE_IMPACT.iloc[1]['exact_p']:.2f} "
              f"(n={GATE_IMPACT.iloc[1]['scored_pairs']})")
    GATE_IMPACT.to_csv(os.path.join(OUT_DIR, f'S4_gate_impact_{PARAM_VERSION}.csv'), index=False)

A_s.to_csv(os.path.join(OUT_DIR, f'S4_spec_vs_gt_{PARAM_VERSION}.csv'), index=False)
print(f"\nsaved: {os.path.join(OUT_DIR, f'S4_spec_vs_gt_{PARAM_VERSION}.csv')}")

### 4.3 OMNIC names vs the Raman Code1 Reference, scored without vocabulary matching

Each OMNIC name is kept as a verbatim string label. The question, which needs no chemical interpretation: are particles that received the same name the same material under the reference?

| metric | definition |
|---|---|
| purity | weighted mean over label groups of the share held by the most frequent reference class |
| n_groups | number of distinct labels |
| heterogeneous group | a label with n >= 2 spanning two or more reference classes |

The same calculation is applied to `spec_class`, so the two streams are compared on equal terms. A singleton group has purity 1.0 by construction; `purity_ge2` (groups with n >= 2) is the figure that matters.

In [ ]:
# === Section 4.3 - OMNIC names vs the Raman Code1 Reference (no vocabulary map) ===

from collections import Counter

def normalize_name(x):
    """Absorbs only case and whitespace variants (Duct tape / Duct Tape). Nothing is
       interpreted."""
    if not isinstance(x, str) or not x.strip():
        return '(no hit)'
    return ' '.join(x.lower().split())

def label_purity(labels, refs, exclude_refs=GT_EXCLUDE_FROM_ACCURACY):
    """Score a stream's labels against the Raman Code1 Reference using GROUP STRUCTURE ONLY,
       with no vocabulary matching.
       purity = sum over groups of the most frequent reference class / total scorable particles.
       Returns (summary dict, per-group DataFrame)."""
    df = pd.DataFrame(dict(label=list(labels), ref=list(refs)))
    df = df[~df['ref'].isin(exclude_refs) & df['ref'].notna() & df['label'].notna()]
    rows = []
    for lab, g in df.groupby('label'):
        c = Counter(g['ref'])
        top, ktop = c.most_common(1)[0]
        rows.append(dict(label=lab, n=len(g), n_gt_classes=len(c),
                         top_gt=top, n_top=ktop, purity=round(ktop/len(g), 3),
                         gt_breakdown=' · '.join(f"{k}×{v}" for k, v in c.most_common())))
    G = pd.DataFrame(rows).sort_values(['n', 'purity'], ascending=[False, True])
    tot = int(G['n'].sum()); hit = int(G['n_top'].sum())
    M2 = G[G['n'] >= 2]
    summ = dict(n_scored=tot, n_groups=len(G),
                mean_group_size=round(tot/len(G), 2) if len(G) else np.nan,
                n_singleton_groups=int((G['n'] == 1).sum()),
                purity_all=round(hit/tot, 3) if tot else np.nan,
                n_groups_ge2=len(M2), n_scored_ge2=int(M2['n'].sum()),
                purity_ge2=round(int(M2['n_top'].sum())/int(M2['n'].sum()), 3) if len(M2) else np.nan,
                n_heterogeneous_ge2=int((M2['n_gt_classes'] > 1).sum()))
    return summ, G

# ── merge: reference (4.1) with the OMNIC record and spec_class ─────────────
#   gt_x is used, not gt: dates without FT-IR have no OMNIC record either, and including them
#   would turn every such row into '(no hit)' and distort the metric.
df_all_om = attach_omnic(df_all_raw, df_omnic_all)
M = gt_x.merge(df_all_om[['date', 'station', 'particle',
                          'omnic_name', 'hqi', 'spec_class', 'confidence']]
               .drop_duplicates(['date', 'station', 'particle']),
               on=['date', 'station', 'particle'], how='left')
M['omnic_label'] = M['omnic_name'].map(normalize_name)
print(f"[merge] {len(M)} particle(s) in the comparison "
      f"({int(gt['ftir_pending'].sum())} of {len(gt)} excluded for having no FT-IR); "
      f"omnic_name present for {int(M['omnic_name'].notna().sum())}/{len(M)}")

print("\n" + "=" * 72)
print("A. both streams on the same footing - group structure against the reference, both ways")
# purity alone is not enough: a stream that gives every particle the same single label scores
# a high purity whenever the reference is itself skewed. Always read it with its inverse.
#     purity       (label -> reference) : are particles sharing a name the same material?
#     completeness (reference -> label) : does the same material receive the same name?
_Ms = M[~M['raman_gt'].isin(GT_EXCLUDE_FROM_ACCURACY) & M['raman_gt'].notna()]
sum_om, G_om = label_purity(M['omnic_label'], M['raman_gt'])
sum_sc, G_sc = label_purity(M['spec_class'],  M['raman_gt'])
cmp_om, R_om = label_purity(_Ms['raman_gt'], _Ms['omnic_label'], exclude_refs=set())
cmp_sc, R_sc = label_purity(_Ms['raman_gt'], _Ms['spec_class'],  exclude_refs=set())
comp = pd.DataFrame([
    dict(stream='OMNIC hit name',     **sum_om, completeness=cmp_om['purity_all']),
    dict(stream='spec_class (Code2)', **sum_sc, completeness=cmp_sc['purity_all']),
]).set_index('stream')
display(comp.T)
print("   purity       (label -> reference) = particles sharing a label are the same material")
print("   completeness (reference -> label) = the same material receives the same label")
print("   Read them together. Purity reaches 1.0 automatically if labels are split finely")
print("   enough (one particle per label), and completeness reaches 1.0 if everything is")
print("   given a single label; quoting either alone lets any stream win.")
print("   Only groups of n>=2 support a testable claim - read purity_ge2.")
print("   Because only group structure is used, a stream that shares no vocabulary with the")
print("   reference is not penalised. That is what makes the comparison possible without a")
print("   chemical translation table.")

print("\n" + "=" * 72)
print("B. OMNIC names - every group with n>=2 (source table)")
_om2 = G_om[G_om['n'] >= 2]
display(_om2[['label', 'n', 'n_gt_classes', 'purity', 'gt_breakdown']])
_het = _om2[_om2['n_gt_classes'] > 1]
if len(_het):
    print(f"   {len(_het)} single name(s) that are CHEMICALLY HETEROGENEOUS: one library entry")
    print("   spanning different materials under the reference:")
    for _, r in _het.iterrows():
        print(f"      \"{r['label']}\"  n={r['n']}  ->  {r['gt_breakdown']}")

print("\n" + "=" * 72)
print("C. spec_class - every group with n>=2 (comparison stream)")
display(G_sc[G_sc['n'] >= 2][['label', 'n', 'n_gt_classes', 'purity', 'gt_breakdown']])

print("\n" + "=" * 72)
print("D. the reverse direction - how many names did OMNIC give one material?")
print("   (the completeness of block A, opened out by reference class. Failures that purity")
print("   cannot see appear here.)")
_rev = (_Ms.groupby('raman_gt')
        .agg(n=('omnic_label', 'size'),
             n_distinct_names=('omnic_label', 'nunique'),
             names=('omnic_label', lambda s: ' · '.join(f"{k}×{v}" for k, v in Counter(s).most_common())))
        .sort_values('n', ascending=False))
_rev['completeness'] = [round(Counter(_Ms[_Ms.raman_gt == c]['omnic_label']).most_common(1)[0][1] / n, 3)
                        for c, n in zip(_rev.index, _rev['n'])]
display(_rev[['n', 'n_distinct_names', 'completeness', 'names']])
print("   A row with many distinct names and low completeness is one material named")
print("   differently each time. Those rows are the subject of this catalogue.")

print("\n" + "=" * 72)
print("E. HQI against group structure (descriptive; no test is performed)")
#   The grouping uses structure alone, with no chemical judgement: is the name repeated, and
#   is it homogeneous?
_t_hqi = M.dropna(subset=['hqi']).copy()
_stat = G_om.set_index('label')[['n', 'n_gt_classes']]
_t_hqi = _t_hqi.join(_stat, on='omnic_label')
def _grp(r):
    if pd.isna(r['n']):            return '4. not scored against the reference'
    if r['n'] < 2:                 return '3. name occurring once (not testable)'
    if r['n_gt_classes'] > 1:      return '2. repeated and CHEMICALLY HETEROGENEOUS'
    return '1. repeated and chemically homogeneous'
_t_hqi['group'] = _t_hqi.apply(_grp, axis=1)
display(_t_hqi.groupby('group')['hqi'].agg(['count', 'mean', 'median', 'min', 'max']).round(1))
print("   If the highest HQI belongs to group 2 - names that recur but span materials - that")
print("   is the first quantitative sign that the hit name, and its match score, are artefacts")
print("   of the reference library.")
print("   No test is run between groups (unequal sizes, multiple comparisons): descriptive only.")

print("\n" + "=" * 72)
print("F. denominators and purity figures (reporting note)")
_n_res = int((~M['raman_gt'].isin(GT_EXCLUDE_FROM_ACCURACY)).sum())
print(f"  Raman reference labels were available for {len(M)} particles (532 nm, QC-passing subset).")
print(f"  {len(M)-_n_res} were left unresolved by the ranking rule and are excluded, leaving n = {_n_res}.")
print(f"  delta = {gt['delta_version'].iloc[0]} - {DELTA_MEANING}")
print("  OMNIC names were compared to the Raman reference WITHOUT mapping them to chemical")
print(f"  classes: purity = {sum_om['purity_ge2']} over {sum_om['n_groups_ge2']} names occurring "
      f"twice or more (n = {sum_om['n_scored_ge2']}),")
print(f"  versus {sum_sc['purity_ge2']} for spec_class over {sum_sc['n_groups_ge2']} classes "
      f"(n = {sum_sc['n_scored_ge2']}).")

G_om.assign(stream='omnic_name').to_csv(
    os.path.join(OUT_DIR, 'S4_omnic_name_vs_gt_v01.csv'), index=False, encoding='utf-8-sig')
print(f"\nsaved: {os.path.join(OUT_DIR, 'S4_omnic_name_vs_gt_v01.csv')}")

### 4.4 Are only weathered PP particles misnamed as PE? Raman 809 against the FT-IR call

In [ ]:
# === Section 4.4 - are only weathered PP particles misnamed as PE? ============
# Hypothesis (Anshari2025): PP that kept its helical order is named correctly by FT-IR; PP
#   that lost it is misnamed as PE.
# Not circular: the weathering axis is the RAMAN 809 cm-1 band (PP-specific, Nava2021); what
#   is judged is the FT-IR rule pp_cryst = 1375 + (998|973|840). Different instruments and
#   bands; co-movement is the evidence.
# No size comparison: particle size was not recorded.

# -- Raman helix bands, from the Code 1 hand-off (top_peaks_cm1, +/-MATCH_TOL) --
RAMAN_HELIX_BANDS = {809: 'C-C/CH2 rock (specific to PP)', 841: 'C-C',
                     973: 'CH3 rock', 998: 'CH3 rock (helical)'}
RAMAN_MATCH_TOL = float(CODE1_STAMP.get('MATCH_TOL', CODE1_STAMP.get('match_tol', 8.0)))

# Transcript of the Code 1 peak output. The CSV is authoritative when present; this is the
# cross-check.
RAMAN_HELIX_SNAPSHOT = {          # (date, particle) -> {band: present}
 ('06.07','5'):  {809:True , 841:True , 973:True , 998:True },
 ('06.14','1'):  {809:True , 841:True , 973:True , 998:True },
 ('06.14','2'):  {809:False, 841:False, 973:False, 998:False},
 ('06.26','11'): {809:True , 841:True , 973:True , 998:False},
 ('06.26','3'):  {809:True , 841:True , 973:True , 998:True },
 ('06.28','4'):  {809:True , 841:True , 973:True , 998:False},
 ('07.12','12'): {809:True , 841:True , 973:True , 998:False},
 ('07.12','13'): {809:True , 841:True , 973:False, 998:False},
 ('07.12','2'):  {809:True , 841:False, 973:False, 998:False},
 ('07.12','3'):  {809:False, 841:True , 973:True , 998:False},
 ('07.25','5'):  {809:False, 841:False, 973:False, 998:False},
 ('07.25','6'):  {809:False, 841:False, 973:False, 998:True },
 ('07.25','10'): {809:False, 841:True , 973:False, 998:False},
}


def _raman_helix_table():
    """(date, particle) -> {band: present}. The hand-off CSV first, the transcript as fallback."""
    if 'top_peaks_cm1' not in gt.columns or not gt['top_peaks_cm1'].astype(str).str.strip().any():
        print(f"   [helix] top_peaks_cm1 absent from the hand-off - using the transcript "
              f"({len(RAMAN_HELIX_SNAPSHOT)} particles)")
        return dict(RAMAN_HELIX_SNAPSHOT), 'snapshot'
    out = {}
    for r in gt.itertuples():
        try:
            pk = [float(v) for v in str(r.top_peaks_cm1).split(',') if v.strip()]
        except ValueError:
            continue
        out[(r.date, str(r.particle))] = {
            b: any(abs(p - b) <= RAMAN_MATCH_TOL for p in pk) for b in RAMAN_HELIX_BANDS}
    # Cross-check against the transcript: a change in Code 1's peak extraction surfaces here.
    _bad = [k for k in RAMAN_HELIX_SNAPSHOT if k in out and out[k] != RAMAN_HELIX_SNAPSHOT[k]]
    print(f"   [helix] hand-off CSV used ({len(out)} particles). Transcript check: "
          f"{len(_bad)} mismatch" + (f" {_bad}" if _bad else ""))
    return out, 'handoff'


if 'A_s' not in dir():
    print("run section 4.2 first")
else:
    HELIX, _src = _raman_helix_table()

    # ── particles the reference calls PP ────────────────────────────────────
    #   The default policy in 4.1 is GT_QC_POLICY='stratify', so QC-flagged particles are NOT
    #   excluded; only what the R3 floor withheld is missing. Item (4) below prints this.
    PPG = A_s[A_s.raman_gt == 'polyolefin_PP'].copy()
    PPG['helix'] = [HELIX.get((d, str(p)), {}) for d, p in zip(PPG.date, PPG.particle)]
    PPG['b809'] = [h.get(809) for h in PPG.helix]
    PPG['n_helix'] = [sum(1 for b in RAMAN_HELIX_BANDS if h.get(b)) for h in PPG.helix]
    PPG['named_pp'] = PPG.tier.isin(['exact', 'compatible'])
    _miss = PPG[PPG.b809.isna()]
    if len(_miss):
        print(f"   {len(_miss)} particle(s) without helix information - excluded: "
              f"{list(zip(_miss.date, _miss.particle))}")
    PPG = PPG[PPG.b809.notna()].copy()

    print("=" * 72)
    print(f"(1) every particle the reference calls PP  (n={len(PPG)})")
    display(PPG[['date', 'station', 'particle', 'b809', 'n_helix', 'key_hit',
                 'raman_conf', 'spec_class', 'tier', 'batch']]
            .sort_values(['b809', 'date'], ascending=[False, True]))

    # ── (2) 2x2 and Fisher's exact test ─────────────────────────────────────
    print("\n" + "=" * 72)
    print("(2) Raman 809 retained or lost, against whether FT-IR called it PP")
    _ct = pd.crosstab(PPG.b809.map({True: '809 retained', False: '809 lost'}),
                      PPG.named_pp.map({True: 'named PP', False: 'not named PP'}))
    for _r in ['809 retained', '809 lost']:
        for _c2 in ['named PP', 'not named PP']:
            if _r not in _ct.index: _ct.loc[_r] = 0
            if _c2 not in _ct.columns: _ct[_c2] = 0
    _ct = _ct.loc[['809 retained', '809 lost'], ['named PP', 'not named PP']].astype(int)
    display(_ct)
    from scipy import stats as _st
    _odds, _p = _st.fisher_exact(_ct.values)
    _n1, _n2 = _ct.loc['809 retained'].sum(), _ct.loc['809 lost'].sum()
    _k1, _k2 = _ct.loc['809 retained', 'named PP'], _ct.loc['809 lost', 'named PP']
    print(f"   809 retained: {_k1}/{_n1} named PP  ·  809 lost: {_k2}/{_n2} named PP")
    print(f"   Fisher exact  p = {_p:.5f}   ({'significant' if _p < 0.05 else 'not significant'})")

    # ── (3) continuous relation: number of helix bands ──────────────────────
    print("\n" + "=" * 72)
    print("(3) continuous relation - FT-IR naming rate by the number of helix bands (0-4)")
    _g = PPG.groupby('n_helix').agg(n=('named_pp', 'size'), named_pp=('named_pp', 'sum'))
    _g['rate'] = (_g.named_pp / _g.n).round(2)
    display(_g)
    if PPG.n_helix.nunique() > 1:
        _r_pb, _p_pb = _st.pointbiserialr(PPG.named_pp.astype(int), PPG.n_helix)
        print(f"   point-biserial r = {_r_pb:+.3f}  (p = {_p_pb:.4f})")

    # ── (4) limitations, generated from the data ────────────────────────────
    print("\n" + "=" * 72)
    print("(4) limitations - report these with the result")
    _c_keep = PPG[PPG.b809 == True]; _c_lost = PPG[PPG.b809 == False]
    _conf_keep = int((_c_keep.raman_conf == 'confident').sum())
    _conf_lost = int((_c_lost.raman_conf == 'confident').sum())
    print(f"   (a) The sample is small - {len(_c_keep)} retained, {len(_c_lost)} lost. "
          f"Speak through the intervals only.")
    print(f"   (b) The two groups differ in how firmly the reference is established: "
          f"{_conf_keep}/{len(_c_keep)} vs {_conf_lost}/{len(_c_lost)} Raman-confident.")
    print( "       Weathering degrades the Raman signal too, which is consistent with the")
    print( "       mechanism, but the confidence of the reference is not equal across groups")
    print( "       and that has to be stated.")
    print( "   (c) No size comparison: particle size was not recorded. Describe 1-5 mm only as")
    print( "       a constraint of the sampling design.")
    print( "   (d) DO NOT USE 809 AS THE WEATHERING AXIS. Section 12-5 of Code 1 measured the")
    print( "       weathering loss of PP 809 across 261 external corpus spectra as 0.095")
    print( "       [-0.101, +0.289] - NOT significant. The PP key bands that are significantly")
    print( "       lost are 998 (+0.45) and 1330 (+0.32). The axis of the table above is")
    print( "       therefore read more faithfully as the STRENGTH OF EVIDENCE for the PP label")
    print( "       than as weathering. To test weathering, either change the helix bands to")
    print( "       998 and 1330, or use the FT-IR carbonyl_index as a continuous axis and")
    print( "       restrict the test to support_level A+/A.")
    print(f"   (e) The QC policy in 4.1 is GT_QC_POLICY = '{GT_QC_POLICY}', so QC-flagged "
          + ("particles were NOT excluded; " if GT_QC_POLICY == 'stratify'
             else "particles were excluded; ")
          + "the only particles missing from this table are those the R3 floor withheld.")
    print(f"   (f) Helix information came from: {_src}.")

    print("\n   [reporting note] figures from the table above:")
    print(f"     PP named PP by FT-IR: 809-retained {_k1}/{_n1} vs 809-lost {_k2}/{_n2}; "
          f"Fisher's exact p = {_p:.3f}")
    print(f"     Raman confident: 809-lost {_conf_lost}/{len(_c_lost)} vs 809-retained "
          f"{_conf_keep}/{len(_c_keep)} (the groups differ in reference firmness as well)")
    print("     809 cm-1 is PP-specific and not read by the FT-IR rule: not an artefact of the logic")
    PPG.drop(columns='helix').to_csv(
        os.path.join(OUT_DIR, f'S4_4_pp_helix_{PARAM_VERSION}.csv'), index=False)
    print(f"\n   [saved] {OUT_DIR}/S4_4_pp_helix_{PARAM_VERSION}.csv")

---
# Part 5. Hand-off to Code 3

In [ ]:
# === Section 5 - hand-off to Code 3: the particle x stream catalogue ==========
# One row per particle, each labelling stream in parallel. Stream numbering (Code 3, Table 2):
#   stream 2  SLoPP       correlation matching vs SLoPP / SLoPP-E, done in Code 1 (B_label,
#                         B_tier, B_r); not the Open Specy software
#   stream 3  spec_class  the in-house rule output, OMNIC never involved
#   stream 4  omnic_name  OMNIC hit name and HQI, verbatim
#   streams 5-7 (OS_cor, OS_ml, siMPLe) are attached by Code 3.
#   reference  Raman Code1 Reference (section 4.1): not a verified identity.
# The verbatim column of each stream must never be overwritten downstream.
cat = attach_omnic(df_all_raw, df_omnic_all)
for _c in ['date', 'station', 'particle']:
    cat[_c] = cat[_c].astype(str)
# The FULL reference table is attached, not gt_x: the catalogue is a record, so particles
# without FT-IR are kept together with the ftir_pending flag, letting Code 3 tell the
# reasons for a gap apart.
cat = cat.merge(gt[['date', 'station', 'particle', 'raman_gt', 'raman_conf', 'gt_source',
                    'support_level', 'support_note', 'B_label', 'B_tier', 'B_r',
                    'tie_break', 'qc', 'qc_flagged', 'delta_version', 'ftir_pending']]
                .rename(columns={'qc': 'raman_qc'}),
                on=['date', 'station', 'particle'], how='left')

# Emission flag for Code 3: True only where the evidence floor emitted a class
# (column name 'validated' kept for backward compatibility with the Code 3 contract).
# Reference-anchored analyses run on True (n = 33); withheld particles (False among the 47)
# enter the reference-free layer only and must never mix into the anchored figures.
cat['validated'] = cat.raman_gt.notna() & (cat.raman_gt != 'unresolved')

COLS = ['date', 'station', 'particle', 'replicate', 'phase', 'file',
        'spec_class', 'confidence', 'is_chemical', 'carbonyl_index', 'mixed', 'reasons',
        'omnic_name', 'hqi',
        'validated', 'raman_gt', 'raman_conf', 'gt_source', 'support_level', 'support_note',
        'B_label', 'B_tier', 'B_r',
        'tie_break', 'raman_qc', 'qc_flagged', 'delta_version', 'ftir_pending']
cat = cat[[c for c in COLS if c in cat.columns]].sort_values(['date', 'station', 'particle'])

_p = os.path.join(OUT_DIR, 'code3_input_catalogue_v01.csv')
cat.to_csv(_p, index=False, encoding='utf-8-sig')
_n_ref  = int(cat.raman_gt.notna().sum())
_n_emit = int(cat.validated.sum())
print(f"catalogue of {len(cat)} rows saved -> {_p}")
print(f"  reference entries {_n_ref} | class emitted {_n_emit} | "
      f"withheld {_n_ref - _n_emit}   <- Code 3 contract checks these")
print(f"  emitted = rows where the evidence floor emitted a class (column 'validated').")
print( "  Reference-anchored figures and tables use emitted rows only; withheld particles")
print( "  stay in the reference-free layer.")
print(f"  spec_class present {int(cat['spec_class'].notna().sum())} | "
      f"omnic_name {int(cat['omnic_name'].notna().sum())} | "
      f"raman_gt {int(cat['raman_gt'].notna().sum())}")
print(f"  parameter snapshot: {FROZEN_PARAMS_PATH} (Code 3 reads the vocabulary and "
      f"thresholds from this JSON)")
_only_raman = gt[gt['ftir_pending']]
if len(_only_raman):
    print(f"  {len(_only_raman)} particle(s) have Raman but no FT-IR and are ABSENT from this")
    print(f"  catalogue, which is built on the FT-IR spectra. Dates: "
          f"{sorted(RAMAN_DATES_PENDING_FTIR)}")
    print( "     Once measured, clear RAMAN_DATES_PENDING_FTIR in 4.1 and re-run sections 4-5.")
display(cat.head(8))

print("\nChecklist for starting Code 3:")
print("  1) Attach the Open Specy and siMPLe results as their own name/score columns")
print("     (key: date, station, particle, replicate). Keep each stream verbatim.")
print("  2) Do NOT translate a stream's names into chemical classes. Reusing label_purity()")
print("     from 4.3 scores every stream on equal terms without any vocabulary matching;")
print("     a translation table would be an assumption of the authors and the result would")
print("     follow the assumption.")
print("  3) The Sankey and the score-vs-agreement analyses are drawn from this one CSV.")
print("     Sankey nodes are the VERBATIM labels of each stream - do not collapse them")
print("     into chemical classes beforehand, and split (or restrict) reference-anchored")
print("     panels on 'validated'.")

## 6. Copy the results back to Drive

`/content/out` disappears with the runtime. Code 3 reads the catalogue and the parameter snapshot from `code2_out/` in the project directory.

In [ ]:
# === 6. Copy the results back to Drive - run this last ======================
import os, shutil

_dst = os.path.join(PROJECT, 'code2_out')      # PROJECT is defined in the setup cell at the top
os.makedirs(_dst, exist_ok=True)
_n = 0
for _f in sorted(os.listdir(OUT_DIR)):
    _p = os.path.join(OUT_DIR, _f)
    if os.path.isfile(_p):
        shutil.copy(_p, _dst); _n += 1
print(f"  {_n} file(s) -> {_dst}")

# What Code 3 requires. The names are built from PARAM_VERSION so that raising it does not
# silently turn these checks into misses.
for _f, _who in [('code3_input_catalogue_v01.csv',       'REQUIRED - particle x stream catalogue'),
                 (os.path.basename(FROZEN_PARAMS_PATH),  'REQUIRED - vocabulary and thresholds'),
                 (f'S4_4_pp_helix_{PARAM_VERSION}.csv',  'SI table')]:
    print(f"    {'ok ' if os.path.exists(os.path.join(_dst, _f)) else '-- '}{_f:<34}{_who}")

print("\n  Do not overwrite Code 1's raman_manual_out/. That folder holds Code 1's output and")
print("  this notebook only reads from it; the transfer is one way.")

In [ ]:
# === Session report (for Methods). Run last; transcribe the output. ===
import sys, platform, datetime
import numpy, pandas, scipy, matplotlib
print("run date  :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Python    :", sys.version.split()[0], "|", platform.platform())
for m in (numpy, pandas, scipy, matplotlib):
    print(f"{m.__name__:<10}: {m.__version__}")